In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Step 1: Install Academic Research Tools
#
# This cell installs every Python package the pipeline needs and patches the
# running event loop so LlamaIndex's async code works inside Jupyter.
# Run this cell once per kernel session (or after a fresh environment create).
#
# Package manifest (what each package does):
#   llama-index                        — Core LlamaIndex framework: nodes, indices, query engines
#   llama-index-llms-replicate         — Replicate.com LLM integration (hosts Granite 3.1)
#   llama-index-embeddings-huggingface — HuggingFace embedding models (BGE, MiniLM, etc.)
#   llama-index-readers-file           — SimpleDirectoryReader + PDF / text loaders
#   llama-index-packs-fusion-retriever — Fusion retriever pack (query rewriting + RRF merging)
#   llama-index-vector-stores-chroma   — ChromaDB vector store adapter for LlamaIndex
#   chromadb                           — Persistent vector database (Memory Palace backend)
#   sentence-transformers              — Required by HuggingFaceEmbedding to load .pt models
#   huggingface_hub[hf_xet]            — Model hub downloads; hf_xet speeds up large model fetches
#   hf_xet                             — High-speed XET protocol transport for HuggingFace
#   certifi                            — Mozilla CA certificate bundle (SSL root trust store)
#   python-certifi-win32               — Merges Windows certificate store into certifi's bundle
#   truststore                         — Uses the OS native TLS trust store as a fallback
#   nest-asyncio                       — Allows asyncio event loops to be nested (required in Jupyter)
#   requests                           — HTTP library used for Google Drive PDF downloads
#   replicate                          — Official Replicate Python client (wraps REST API calls)
#   pytesseract                        — Python wrapper for the Tesseract OCR engine
#   pdf2image                          — Converts PDF pages to PIL images for OCR input
#   Pillow                             — Image processing library; required by pdf2image output
#   PyMuPDF                            — Fast PDF text extraction (fitz module); primary read path
# ─────────────────────────────────────────────────────────────────────────────

# %pip install runs pip inside the current kernel's environment (not a subprocess),
# so packages are immediately importable without restarting the kernel.
# -q (quiet) suppresses verbose "Downloading…" lines; warnings still show.
%pip install -q \
  llama-index \
  llama-index-llms-replicate \
  llama-index-embeddings-huggingface \
  llama-index-readers-file \
  llama-index-packs-fusion-retriever \
  llama-index-vector-stores-chroma \
  chromadb \
  sentence-transformers \
  "huggingface_hub[hf_xet]" \
  hf_xet \
  certifi \
  python-certifi-win32 \
  truststore \
  nest-asyncio \
  requests \
  replicate \
  pytesseract \
  pdf2image \
  Pillow \
  PyMuPDF

# ── Async loop patch ──────────────────────────────────────────────────────────
import nest_asyncio   # Import the installed package into the current namespace
nest_asyncio.apply()  # Monkey-patch asyncio to allow re-entrant event loops.
                      # Jupyter already runs an event loop; without this patch,
                      # LlamaIndex's await calls raise "This event loop is already running".

print("✅ Installation complete (including OCR + persistent memory deps).")

# ── System dependencies note ──────────────────────────────────────────────────
# The Python packages above are not enough on their own — OCR also needs two
# system-level binaries that must be installed separately:
#
# 1. Tesseract OCR binary
#    The executable that performs actual character recognition on rasterised images.
#    Install from: https://github.com/tesseract-ocr/tesseract
#    Default Windows path: C:\Program Files\Tesseract-OCR\tesseract.exe
#    Cell 2 auto-discovers this path; Cell 3 validates the install.
#
# 2. Poppler utils (pdftoppm / pdftocairo)
#    pdf2image calls these Poppler binaries to rasterize PDF pages before OCR.
#    Install from: https://poppler.freedesktop.org/ or via conda / choco / scoop.
#    Cell 3 checks that poppler is on PATH and prints diagnostics if it is not.
#
# If either binary is missing, the pipeline continues using PyMuPDF native text
# extraction and prints a clear warning rather than crashing.


Note: you may need to restart the kernel to use updated packages.
✅ Installation complete (including OCR + persistent memory deps).



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:

# ═══════════════════════════════════════════════════════════
#  MEMORY PALACE RESET  —  Run this cell ONCE to wipe all
#  ingested data so you can start fresh with new PDFs.
#
#  ┌─────────────────────────────────────────────────────┐
#  │  👉 TO CONFIRM THE WIPE: change the line below to   │
#  │       CONFIRM_RESET = True                           │
#  │  Leave it False to do a dry-run / status check.     │
#  └─────────────────────────────────────────────────────┘
#
#  ⚠️  CRITICAL WARNING:
#  Running this cell with CONFIRM_RESET=True calls
#  _system.stop() which PERMANENTLY INVALIDATES ChromaDB's
#  Rust bindings for the entire kernel session.
#  You MUST restart the kernel before running Cell 3 again.
#
#  After running this cell with CONFIRM_RESET=True:
#    1. Restart the kernel (Kernel → Restart)
#    2. Re-run Cell 1, Cell 4, Cell 5
#    3. Run Cell 3 (batch ingest)
# ═══════════════════════════════════════════════════════════

CONFIRM_RESET = False   # ← change to True to execute the wipe

# ─────────────────────────────────────────────────────────
import gc, json, os, shutil, time

PALACE_DB_PATH       = "./mempalace_db"
PALACE_COLLECTION_   = "academic_palace"
PALACE_DOCSTORE_PATH = os.path.join(PALACE_DB_PATH, "docstore.json")

# ── Step 0: Count existing vectors via a fresh client ────
# The Rust bindings may already be dead in this session if
# _system.stop() was called previously.  We detect that and
# skip the count rather than crashing.
_vec_count   = None   # None = "could not determine"
_rust_dead   = False
_check_client = None

try:
    import chromadb as _chromadb
    _check_client = _chromadb.PersistentClient(path=PALACE_DB_PATH)
    _check_col    = _check_client.get_or_create_collection(PALACE_COLLECTION_)
    _vec_count    = _check_col.count()
except (AttributeError, ValueError) as _e:
    # AttributeError: 'RustBindingsAPI' object has no attribute 'bindings'
    # ValueError:     Could not connect to tenant default_tenant
    # Both mean the Rust layer is already dead — needs kernel restart.
    _rust_dead = True
except Exception as _e:
    print(f"ℹ️  Status check warning: {_e}")
finally:
    # ⚠️  DO NOT call _check_client._system.stop() here when CONFIRM_RESET=False.
    # Calling _system.stop() on ANY ChromaDB client kills the shared Rust DLL
    # for the entire kernel session — making all subsequent ChromaDB calls fail.
    # Simply drop the reference and let Python GC clean up the client safely.
    _check_client = None
    gc.collect()

# ── Report status ─────────────────────────────────────────
if _rust_dead:
    print("⚠️  ChromaDB Rust bindings are ALREADY INVALIDATED in this kernel session.")
    print("   This happens after _system.stop() has been called previously.")
    print()
    print("   ♻️  Please RESTART THE KERNEL, then re-run Cell 1, 4, 5, and Cell 3.")
    print("   No further action is needed here — the palace was already wiped.")
else:
    _status_str = f"{_vec_count} vector(s)" if _vec_count is not None else "status unknown"
    print(f"📊 Memory Palace status: {_status_str} in '{PALACE_COLLECTION_}'")

    # ── Guard: require explicit confirmation ──────────────
    if not CONFIRM_RESET:
        print()
        if _vec_count and _vec_count > 0:
            print("🛑 RESET NOT CONFIRMED — palace data is SAFE.")
            print()
            print("   To wipe all data, edit this cell and set:")
            print("       CONFIRM_RESET = True")
            print("   Then re-run the cell.")
        else:
            print("ℹ️  Palace is already empty — nothing to reset.")
        print()
        print("   ⚠️  After a reset you MUST restart the kernel before")
        print("      running the batch ingest cell again.")
    else:
        # ── Confirmed: proceed with wipe ──────────────────
        print(f"\n⚠️  Proceeding with full Memory Palace wipe ({_vec_count} vectors)...")
        print("   After this completes → RESTART KERNEL → re-run Cell 1, 4, 5, then Cell 3.\n")

        # 1. Use the EXISTING palace_client (if open in kernel) to delete the collection
        try:
            existing = [c.name for c in palace_client.list_collections()]
            if PALACE_COLLECTION_ in existing:
                palace_client.delete_collection(PALACE_COLLECTION_)
                print(f"✅ Deleted ChromaDB collection '{PALACE_COLLECTION_}'")
            else:
                print(f"ℹ️  Collection '{PALACE_COLLECTION_}' did not exist")
        except Exception as e:
            print(f"ℹ️  palace_client not available ({e}); proceeding to file-level wipe")

        # 2. Close the client — permanently invalidates Rust bindings
        try:
            palace_client._system.stop()
            print("✅ ChromaDB client closed (file locks released)")
            print("   ⚠️  Rust bindings are now invalid — RESTART THE KERNEL before running Cell 3.")
        except Exception as e:
            print(f"ℹ️  palace_client stop: {e}")

        gc.collect()
        time.sleep(0.5)

        # 3. Remove leftover HNSW segment folders
        removed = 0
        for entry in os.scandir(PALACE_DB_PATH):
            if entry.is_dir():
                try:
                    shutil.rmtree(entry.path)
                    print(f"  🗑️  Removed segment folder: {entry.name}")
                    removed += 1
                except Exception as e:
                    print(f"  ⚠️  Could not remove {entry.name}: {e}")
        if removed == 0:
            print("  ℹ️  No segment folders to remove")

        # 4. Remove SQLite files and reset docstore
        for fname in ("chroma.sqlite3", "chroma.sqlite3-wal", "chroma.sqlite3-shm"):
            fp = os.path.join(PALACE_DB_PATH, fname)
            if os.path.exists(fp):
                try:
                    os.remove(fp)
                    print(f"  🗑️  Removed {fname}")
                except PermissionError as e:
                    print(f"  ⚠️  Could not remove {fname}: {e}")
                    print("      File still locked — restart kernel and re-run this cell.")

        with open(PALACE_DOCSTORE_PATH, "w", encoding="utf-8") as f:
            json.dump({"docstore/data": {}}, f)
        print(f"✅ Docstore reset to empty")

        # 5. Null out in-kernel references
        for _vname in ("palace_client", "palace_index", "palace_vector_store",
                       "palace_collection", "palace_pipeline"):
            if _vname in globals():
                globals()[_vname] = None

        print("\n🟢 Memory Palace is now COMPLETELY EMPTY.")
        print("=" * 55)
        print("👉 NEXT STEPS — you MUST do ALL of these:")
        print("   1. ♻️  RESTART THE KERNEL  (Kernel → Restart Kernel)")
        print("   2. Run Cell 1  (environment setup)")
        print("   3. Run Cell 4  (console logger)")
        print("   4. Run Cell 5  (OCR diagnostics)")
        print("   5. Run Cell 3  (batch ingest)")
        print("=" * 55)
        print("\n   ← Reset CONFIRM_RESET back to False after restarting.")


📊 Memory Palace status: 4130 vector(s) in 'academic_palace'

🛑 RESET NOT CONFIRMED — palace data is SAFE.

   To wipe all data, edit this cell and set:
       CONFIRM_RESET = True
   Then re-run the cell.

   ⚠️  After a reset you MUST restart the kernel before
      running the batch ingest cell again.


In [2]:

# ═══════════════════════════════════════════════════════════════════════
#  BATCH INGEST — All individual PDFs in academic_data/
#  Labels applied to EVERY PDF:
#    Wing        : Academics
#    Room        : Adaptive Leadership
#    Hall        : Adaptive Leadership
#    Drawer Type : Adv. Dip
#
#  source_material.pdf is SKIPPED — it is the merged copy of all parts.
# ═══════════════════════════════════════════════════════════════════════
import os, glob, time
import chromadb
from llama_index.vector_stores.chroma   import ChromaVectorStore
from llama_index.core                   import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.ingestion         import IngestionPipeline, DocstoreStrategy
from llama_index.core.storage.docstore  import SimpleDocumentStore

# ── Fixed labels for every PDF ────────────────────────────────────────
BATCH_WING        = "Academics"
BATCH_ROOM        = "Adaptive Leadership"
BATCH_HALL        = "Adaptive Leadership"
BATCH_DRAWER_TYPE = "Adv. Dip"

# ── Palace paths ──────────────────────────────────────────────────────
PALACE_DB_PATH       = os.getenv("PALACE_DB_PATH", "./mempalace_db")
PALACE_COLLECTION    = os.getenv("PALACE_COLLECTION", "academic_palace")
PALACE_DOCSTORE_PATH = os.path.join(PALACE_DB_PATH, "docstore.json")
DATA_DIR_BATCH       = "./academic_data"

os.makedirs(PALACE_DB_PATH, exist_ok=True)

# ── Guard: catch dead Rust bindings on the real client creation ───────
# Do NOT use a probe that calls _system.stop() — that itself kills the
# Rust DLL for the whole process.  Instead, catch errors directly here.
_chroma_ok        = True
palace_client     = None
palace_collection = None
palace_vector_store = None
try:
    palace_client     = chromadb.PersistentClient(path=PALACE_DB_PATH)
    palace_collection = palace_client.get_or_create_collection(
        PALACE_COLLECTION,
        metadata={"hnsw:space": "cosine"},
    )
    palace_vector_store = ChromaVectorStore(chroma_collection=palace_collection)
except (AttributeError, ValueError):
    _chroma_ok = False

if not _chroma_ok:
    print("=" * 60)
    print("⚠️  ChromaDB Rust bindings are INVALIDATED in this kernel.")
    print()
    print("   The Memory Palace Reset cell (Cell 2) called _system.stop()")
    print("   which permanently kills the Rust layer for this session.")
    print()
    print("   ♻️  RESTART THE KERNEL, then run in order:")
    print("       Cell 1  → environment setup")
    print("       Cell 4  → console logger")
    print("       Cell 5  → OCR diagnostics")
    print("       Cell 3  → this batch ingest cell")
    print("=" * 60)
else:
    # ── Ensure embed_model and splitter are available ─────────────────
    if "embed_model" not in globals() or embed_model is None:
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        from llama_index.core import Settings
        embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
        Settings.embed_model = embed_model
        print("✅ embed_model loaded.")
    if "splitter" not in globals() or splitter is None:
        from llama_index.core.node_parser import SemanticSplitterNodeParser
        splitter = SemanticSplitterNodeParser(
            buffer_size=3,
            breakpoint_percentile_threshold=95,
            embed_model=embed_model,
        )
        print("✅ splitter ready.")

    # ── Discover PDFs — skip the big merged file ──────────────────────
    SKIP_FILES = {"source_material.pdf"}
    all_pdfs = sorted(glob.glob(os.path.join(DATA_DIR_BATCH, "*.pdf")))
    pdf_files = [p for p in all_pdfs if os.path.basename(p) not in SKIP_FILES]

    if not pdf_files:
        print(f"⚠️  No PDFs found in '{DATA_DIR_BATCH}'. Add your files and re-run.")
    else:
        print(f"📚 Found {len(pdf_files)} PDF(s) to ingest (skipping: {SKIP_FILES}):")
        for p in pdf_files:
            size_kb = os.path.getsize(p) // 1024
            print(f"   • {os.path.basename(p)}  ({size_kb} KB)")
        print()

        if os.path.exists(PALACE_DOCSTORE_PATH):
            palace_docstore = SimpleDocumentStore.from_persist_path(PALACE_DOCSTORE_PATH)
        else:
            palace_docstore = SimpleDocumentStore()

        print(f"🗄️  Palace opened. Current vectors: {palace_collection.count()}\n")

        # ── Helper: force-upsert nodes that the pipeline may miss ─────
        def _chroma_upsert(nodes_list, fname):
            ids, docs, metas, embs = [], [], [], []
            for idx, node in enumerate(nodes_list):
                try:
                    txt = node.get_content(metadata_mode="none")
                except Exception:
                    txt = node.get_content() if hasattr(node, "get_content") else ""
                if not txt or not txt.strip():
                    continue
                meta = dict(getattr(node, "metadata", {}) or {})
                meta.update({"source": fname, "file_name": fname,
                             "wing": BATCH_WING, "room": BATCH_ROOM,
                             "hall": BATCH_HALL, "drawer_type": BATCH_DRAWER_TYPE})
                node_id = (getattr(node, "node_id", None)
                           or getattr(node, "id_", None)
                           or f"{fname}::node::{idx}")
                emb = getattr(node, "embedding", None) or embed_model.get_text_embedding(txt)
                ids.append(str(node_id)); docs.append(txt)
                metas.append(meta);       embs.append(emb)
            if ids:
                palace_collection.upsert(ids=ids, documents=docs, metadatas=metas, embeddings=embs)
            return len(ids)

        # ── Ingest loop ───────────────────────────────────────────────
        total_nodes = 0
        failed      = []

        for i, pdf_path in enumerate(pdf_files, 1):
            fname = os.path.basename(pdf_path)
            t0    = time.time()
            print(f"[{i:02d}/{len(pdf_files)}] ⏳ {fname} ...")
            try:
                docs = SimpleDirectoryReader(input_files=[pdf_path]).load_data()
                for d in docs:
                    d.metadata.update({
                        "source": fname, "file_name": fname,
                        "wing": BATCH_WING, "room": BATCH_ROOM,
                        "hall": BATCH_HALL, "drawer_type": BATCH_DRAWER_TYPE,
                    })

                before = palace_collection.count()
                pipeline = IngestionPipeline(
                    transformations=[splitter],
                    docstore=palace_docstore,
                    vector_store=palace_vector_store,
                    docstore_strategy=DocstoreStrategy.UPSERTS,
                )
                p_nodes = pipeline.run(documents=docs)
                for n in p_nodes:
                    n.metadata.update({
                        "source": fname, "file_name": fname,
                        "wing": BATCH_WING, "room": BATCH_ROOM,
                        "hall": BATCH_HALL, "drawer_type": BATCH_DRAWER_TYPE,
                    })

                after = palace_collection.count()
                fallback_n = 0
                if p_nodes and after <= before:
                    fallback_n = _chroma_upsert(p_nodes, fname)
                    after = palace_collection.count()

                palace_docstore.persist(persist_path=PALACE_DOCSTORE_PATH)

                node_count   = len(p_nodes) or fallback_n
                total_nodes += node_count
                elapsed      = time.time() - t0
                fb_note      = f" (fallback: {fallback_n})" if fallback_n else ""
                print(f"         ✅ {node_count} nodes | palace {before}→{after}{fb_note} | {elapsed:.1f}s")

            except Exception as e:
                failed.append(fname)
                print(f"         ❌ FAILED — {e}")

        # ── Final save & rebuild index ────────────────────────────────
        palace_docstore.persist(persist_path=PALACE_DOCSTORE_PATH)

        palace_index = VectorStoreIndex.from_vector_store(
            palace_vector_store,
            embed_model=embed_model,
        )

        try:
            palace_query_engine = palace_index.as_query_engine(
                similarity_top_k=8,
                response_mode="compact",
                streaming=False,
            )
        except Exception:
            palace_query_engine = None

        wing_label  = BATCH_WING
        room_label  = BATCH_ROOM
        hall_label  = BATCH_HALL
        drawer_type = BATCH_DRAWER_TYPE

        print()
        print("=" * 60)
        print("🎓 BATCH INGEST COMPLETE")
        print(f"   PDFs ingested   : {len(pdf_files) - len(failed)} / {len(pdf_files)}")
        print(f"   Total nodes     : {total_nodes}")
        print(f"   Palace vectors  : {palace_collection.count()}")
        print(f"   wing={BATCH_WING} | room={BATCH_ROOM}")
        print(f"   hall={BATCH_HALL} | drawer={BATCH_DRAWER_TYPE}")
        if failed:
            print(f"   ❌ Failed       : {', '.join(failed)}")
        print("=" * 60)
        print("👉 palace_index is ready — run Cell 16 for Q&A.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ embed_model loaded.
✅ splitter ready.
📚 Found 16 PDF(s) to ingest (skipping: {'source_material.pdf'}):
   • Applied Research Methods and Statistics for Leadership - HADR400-1.pdf  (62 KB)
   • source_material_part1.pdf  (1557 KB)
   • source_material_part10.pdf  (294 KB)
   • source_material_part11.pdf  (371 KB)
   • source_material_part12.pdf  (2806 KB)
   • source_material_part13.pdf  (6256 KB)
   • source_material_part14.pdf  (431 KB)
   • source_material_part15.pdf  (170 KB)
   • source_material_part2.pdf  (816 KB)
   • source_material_part3.pdf  (347 KB)
   • source_material_part4.pdf  (409 KB)
   • source_material_part5.pdf  (188 KB)
   • source_material_part6.pdf  (4009 KB)
   • source_material_part7.pdf  (196 KB)
   • source_material_part8.pdf  (286 KB)
   • source_material_part9.pdf  (193 KB)

🗄️  Palace opened. Current vectors: 21037

[01/16] ⏳ Applied Research Methods and Statistics for Leadership - HADR400-1.pdf ...


2026-05-22 16:17:23,725 - WARNING - invalid pdf header: b'<!DOC'
2026-05-22 16:17:23,745 - WARNING - EOF marker not found
2026-05-22 16:17:23,750 - WARNING - invalid pdf header: b'<!DOC'
2026-05-22 16:17:23,770 - WARNING - EOF marker not found
2026-05-22 16:17:23,774 - WARNING - invalid pdf header: b'<!DOC'
2026-05-22 16:17:23,795 - WARNING - EOF marker not found


Failed to load file academic_data\Applied Research Methods and Statistics for Leadership - HADR400-1.pdf with error: RetryError[<Future at 0x27607366d50 state=finished raised PdfStreamError>]. Skipping...
         ✅ 0 nodes | palace 21037→21037 | 3.5s
[02/16] ⏳ source_material_part1.pdf ...
         ✅ 255 nodes | palace 21037→21292 (fallback: 255) | 283.8s
[03/16] ⏳ source_material_part10.pdf ...
         ✅ 49 nodes | palace 21292→21341 (fallback: 49) | 17.7s
[04/16] ⏳ source_material_part11.pdf ...
         ✅ 152 nodes | palace 21341→21493 (fallback: 152) | 150.1s
[05/16] ⏳ source_material_part12.pdf ...
         ✅ 81 nodes | palace 21493→21574 (fallback: 81) | 33.6s
[06/16] ⏳ source_material_part13.pdf ...
         ✅ 572 nodes | palace 21574→22139 (fallback: 565) | 281.0s
[07/16] ⏳ source_material_part14.pdf ...
         ✅ 210 nodes | palace 22139→22349 (fallback: 210) | 120.0s
[08/16] ⏳ source_material_part15.pdf ...
         ✅ 82 nodes | palace 22349→22431 (fallback: 82) | 36.6s
[0

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Console / Logger Helper (Widget-Free for VS Code Compatibility)
# Jupyter widgets (ipywidgets) don't always render in VS Code's notebook panel.
# This cell replaces them with simple print-based logging so every message
# appears inline in cell output with a timestamp and severity level.
# ─────────────────────────────────────────────────────────────────────────────

from datetime import datetime   # Used to generate human-readable timestamps for each log line
import logging                  # Standard Python logging framework — we hook into it below
import glob                     # Provides Unix-style wildcard path expansion (used in Tesseract search)
import os                       # OS interaction: env vars, file existence checks, path joins
import shutil                   # High-level file utilities — used here for `shutil.which()` PATH lookup


# ── console_log: the primary logging function used throughout the notebook ────
def console_log(msg, level='INFO'):
    # Build a timestamp string in YYYY-MM-DD HH:MM:SS format
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    # Print the message prefixed with timestamp and level (INFO, OK, WARN, ERROR, etc.)
    # Everything printed here appears directly in the cell output panel.
    print(f"[{ts}] {level}: {msg}")


# ── resolve_tesseract_cmd: locate Tesseract OCR executable automatically ──────
def resolve_tesseract_cmd() -> str | None:
    """Find and configure a usable Tesseract executable path.

    Returns the first valid path found, or None if Tesseract is not installed.
    Also sets the TESSERACT_CMD env var and pytesseract.tesseract_cmd so all
    downstream OCR calls use the correct binary without needing manual config.
    """
    candidates = []  # Ordered list of candidate paths to check, most preferred first

    # ── Candidate 1: explicit env-var override ────────────────────────────────
    # If the user has already set TESSERACT_CMD in the environment (e.g. in a
    # .env file or a launch script), honour it first without any guessing.
    env_cmd = os.getenv("TESSERACT_CMD", "").strip()
    if env_cmd and os.path.exists(env_cmd):
        candidates.append(env_cmd)

    # ── Candidate 2: PATH lookup via shutil.which ─────────────────────────────
    # `shutil.which("tesseract")` scans every directory in the system PATH and
    # returns the full path to the first match, exactly like `which` on Unix.
    path_cmd = shutil.which("tesseract")
    if path_cmd:
        candidates.append(path_cmd)

    # ── Candidates 3+: Windows-specific fixed install locations ──────────────
    if os.name == "nt":  # os.name == "nt" is True only on Windows
        # Standard installer paths for both 64-bit and 32-bit Program Files
        default_paths = [
            r"C:\Program Files\Tesseract-OCR\tesseract.exe",
            r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        ]
        for p in default_paths:
            if os.path.exists(p):
                candidates.append(p)

        # WinGet installs packages under a user-specific path with a hashed folder name.
        # We use a recursive glob to find any tesseract.exe under the WinGet packages tree.
        local_app_data = os.getenv("LOCALAPPDATA", "")  # e.g. C:\Users\You\AppData\Local
        if local_app_data:
            winget_pattern = os.path.join(
                local_app_data,
                "Microsoft",
                "WinGet",
                "Packages",
                "*",        # wildcard matches the hashed package folder name
                "**",       # recursive descent into any subdirectory
                "tesseract.exe",
            )
            # glob.glob with recursive=True expands ** to match any depth of subdirectories
            winget_matches = glob.glob(winget_pattern, recursive=True)
            # Sort for determinism — multiple WinGet versions would otherwise be random order
            candidates.extend(sorted(winget_matches))

    # ── Walk candidates and return the first valid path ───────────────────────
    for cmd in candidates:
        if os.path.exists(cmd):
            # Persist the confirmed path back into the environment so child processes see it
            os.environ["TESSERACT_CMD"] = cmd
            try:
                import pytesseract
                # Tell pytesseract explicitly which binary to call when performing OCR
                pytesseract.pytesseract.tesseract_cmd = cmd
            except Exception:
                pass  # pytesseract may not be installed yet during the first pass
            return cmd  # Return the first working path; no need to check further

    # None of the candidates were valid — Tesseract is not installed or not findable
    return None


# ── ConsoleHandler: bridge Python's logging module to console_log ─────────────
# Many libraries (e.g. LlamaIndex, ChromaDB) emit log records via Python's
# standard `logging` module.  By adding a custom handler that calls console_log,
# those library messages appear in the notebook output in the same format as
# our own console_log() calls instead of being silently dropped.
class ConsoleHandler(logging.Handler):
    def emit(self, record):
        # `self.format(record)` applies the handler's formatter to the log record,
        # producing the final message string (including any exception info).
        console_log(self.format(record), record.levelname)


# ── Attach ConsoleHandler to the root logger (once only) ─────────────────────
root_logger = logging.getLogger()  # The root logger receives all log records by default
# Guard against adding a duplicate handler on kernel re-runs (which would double-print every message)
if not any(isinstance(h, ConsoleHandler) for h in root_logger.handlers):
    handler = ConsoleHandler()
    # Format: "HH:MM:SS - LEVEL - message"
    # Note: console_log already prepends its own timestamp, so this format is
    # kept short to avoid double timestamps in output.
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

# Set the minimum severity to INFO so DEBUG records are suppressed by default
root_logger.setLevel(logging.INFO)

console_log("Console initialized - logs will appear in cell output.", "OK")


[2026-05-22 16:45:46] OK: Console initialized - logs will appear in cell output.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — OCR Diagnostics: Verify Tesseract and Poppler Availability
# Before spending time on PDF ingestion, this cell checks that every system-
# level binary the OCR fallback path needs is actually installed and reachable.
# It also runs a quick end-to-end pdf2image conversion test if a sample PDF
# already exists in the data directory, so you know the full OCR chain works.
# ─────────────────────────────────────────────────────────────────────────────

import shutil   # Used for `shutil.which()` — finds executables in PATH
import os       # Used for path checks and environment variable reads

console_log("Running diagnostics: checking system OCR dependencies...", "INFO")


# ── 1. Tesseract check ────────────────────────────────────────────────────────
# Call the resolver defined in Cell 2.  It checks env vars, PATH, common
# install locations, and WinGet packages in order, returning the first valid path.
tess_path = resolve_tesseract_cmd()

if tess_path:
    try:
        import pytesseract
        # get_tesseract_version() actually shells out to the binary and parses its
        # version string, so a success here confirms the executable is fully working.
        v = pytesseract.get_tesseract_version()
        console_log(f"Tesseract found: {tess_path} - version {v}", "OK")
    except Exception as e:
        # The path exists but the binary failed to run (missing DLLs, wrong arch, etc.)
        console_log(f"Tesseract found at {tess_path} but pytesseract error: {e}", "WARN")
else:
    # Tesseract is absent — OCR fallback will silently fail later without this warning.
    console_log("Tesseract executable not found in PATH or common install locations.", "ERROR")
    console_log(
        "Install Tesseract: https://github.com/tesseract-ocr/tesseract, `winget install UB-Mannheim.TesseractOCR`, or `choco install tesseract`",
        "INFO",
    )


# ── 2. Poppler check ──────────────────────────────────────────────────────────
# pdf2image (used to render PDF pages to images for OCR) requires Poppler's
# command-line tools.  `pdftoppm` and `pdftocairo` are both acceptable — we
# accept whichever one exists on PATH first.
poppler_bin = shutil.which("pdftoppm") or shutil.which("pdftocairo")

if poppler_bin:
    # At least one Poppler utility was found; pdf2image should work.
    console_log(f"Poppler utility found: {poppler_bin}", "OK")
else:
    # Without Poppler, pdf2image.convert_from_path() will raise an immediate error.
    console_log("Poppler utilities (pdftoppm/pdftocairo) not found in PATH.", "ERROR")
    console_log("Install Poppler: https://poppler.freedesktop.org/ or `choco install poppler`", "INFO")


# ── 3. Optional end-to-end conversion test ───────────────────────────────────
# If a source PDF was already downloaded (from a previous run), use it to verify
# the full pdf2image → PIL pipeline rather than just the Poppler binary check.
sample_pdf = os.path.join("academic_data", "source_material.pdf")

if os.path.exists(sample_pdf):
    if poppler_bin:
        try:
            from pdf2image import convert_from_path
            # Convert only the first page at very low DPI (50) — enough to test the
            # pipeline without burning time rendering the whole document at full quality.
            imgs = convert_from_path(sample_pdf, dpi=50, first_page=1, last_page=1)
            # If we reach this line, Poppler rendered the page and PIL received the image.
            console_log("pdf2image conversion test succeeded (Poppler working).", "OK")
        except Exception as e:
            # Conversion failed — common causes: Poppler not on PATH, corrupted PDF,
            # or a version mismatch between pdf2image and the installed Poppler.
            console_log(f"pdf2image conversion test failed: {e}", "ERROR")
    else:
        # Poppler is missing so we skip the test rather than producing a misleading traceback.
        console_log("Skipping pdf2image test because Poppler not found.", "WARN")
else:
    # No PDF yet — this is expected on the first run before Step 3 downloads one.
    console_log(f"No sample PDF at {sample_pdf}; skipping conversion test.", "INFO")

console_log("Diagnostics complete.", "OK")


[2026-05-22 17:14:39] INFO: Running diagnostics: checking system OCR dependencies...
[2026-05-22 17:14:41] OK: Tesseract found: C:\Users\SMANYEL\AppData\Local\Programs\Tesseract-OCR\tesseract.EXE - version 5.5.0.20241111
[2026-05-22 17:14:41] OK: Poppler utility found: C:\Program Files\poppler-25.12.0\Library\bin\pdftoppm.EXE
[2026-05-22 17:14:43] OK: pdf2image conversion test succeeded (Poppler working).
[2026-05-22 17:14:43] OK: Diagnostics complete.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Configure IBM Granite 3.1 & Security Guardrails
# This cell sets up two global LlamaIndex objects (Settings.llm and
# Settings.embed_model) that every later cell implicitly uses.
# It also hardens SSL/TLS networking so the pipeline works on corporate
# networks and Windows machines without manual certificate wrangling.
# ─────────────────────────────────────────────────────────────────────────────

import os               # Environment variable reads and writes
from getpass import getpass  # Secure password prompt — input stays hidden in the terminal
import certifi          # Provides an up-to-date CA bundle (certifi.where() returns its file path)
from llama_index.core import Settings                         # Global LlamaIndex configuration object
from llama_index.llms.replicate import Replicate              # Adapter that calls Replicate-hosted LLMs
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # Local sentence-embedding model


# ── Networking hardening ──────────────────────────────────────────────────────
# XetHub is a large-file transport layer used by newer HuggingFace Hub versions.
# Disabling it forces HF to fall back to standard HTTPS chunked downloads,
# which work reliably on all corporate proxies and firewalls.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Suppress an irrelevant warning about symlinks being disabled on Windows;
# setdefault only writes the value if the key is NOT already set, so an
# explicit user override in the environment is always respected.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# Point all three common SSL CA bundle environment variables to certifi's bundle.
# Different libraries (requests, urllib3, curl-based tools) each check their own
# variable, so we set all three for maximum compatibility.
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())  # Used by requests / urllib3
os.environ.setdefault("SSL_CERT_FILE",       certifi.where())  # Used by Python's ssl module
os.environ.setdefault("CURL_CA_BUNDLE",      certifi.where())  # Used by pycurl / libcurl tools

# ── Windows cert store bridge ─────────────────────────────────────────────────
# On managed Windows machines, corporate roots are stored in the Windows cert
# store (certmgr), not in certifi's bundle.  certifi_win32 reads the Windows
# store and adds those roots to certifi's CA file so HTTPS calls to internal
# servers (e.g. corporate Replicate proxies) don't fail with SSL errors.
try:
    import certifi_win32  # noqa: F401  (import for its side-effect only — no direct API needed)
    print("✅ Windows certificate store bridge enabled (certifi-win32).")
except Exception as e:
    # Not a hard failure — if certifi_win32 isn't installed the pipeline still works
    # on most networks; it's only needed for custom corporate CA roots.
    print(f"⚠️ certifi-win32 not active ({e}); using certifi defaults.")


# ── Replicate API token ───────────────────────────────────────────────────────
# Try the environment variable first so automated runs (CI/CD, .env files)
# work without an interactive prompt.
replicate_token = os.getenv("REPLICATE_API_TOKEN", "").strip()
if not replicate_token:
    try:
        # getpass hides the typed characters — use it in terminals that support it.
        replicate_token = getpass("Enter REPLICATE_API_TOKEN: ").strip()
    except Exception:
        # getpass fails in some non-interactive Jupyter environments; fall back to input().
        replicate_token = input("Enter REPLICATE_API_TOKEN: ").strip()

# Hard stop: without a token the Replicate API will reject every request.
if not replicate_token:
    raise ValueError("REPLICATE_API_TOKEN is required to continue.")

# Write the token into the environment so the `replicate` SDK picks it up
# automatically without needing to pass it explicitly to each call.
os.environ["REPLICATE_API_TOKEN"] = replicate_token


# ── IBM Granite 3.1 LLM configuration ────────────────────────────────────────
# ACADEMIC SHIELD FIX: Granite 3.1 with Extended Patience
# The Replicate adapter wraps the model behind an HTTP request.
# temperature=0.1 keeps answers factually tight with very little randomness.
# context_window=128000 tells LlamaIndex how many tokens Granite can process
# in a single call — Granite 3.1 supports up to 128k tokens.
# is_chat_model=True routes requests through the chat-style API endpoint.
# request_timeout=600.0 gives the model up to 10 minutes to respond — complex
# multi-PDF synthesis prompts can take several minutes on Replicate's queue.
# system_prompt anchors the model's behaviour: it must only use provided
# evidence, refuse to hallucinate, and always write in formal academic prose.
llm = Replicate(
    model="ibm-granite/granite-3.1-8b-instruct",  # Official Granite 3.1 8B instruct model ID on Replicate
    temperature=0.1,           # Near-zero temperature = deterministic, factual responses
    context_window=128000,     # Maximum token budget per request (Granite 3.1 native limit)
    is_chat_model=True,        # Use the chat-format API (system + user turn structure)
    request_timeout=600.0,     # 10-minute timeout: handles slow starts on Replicate's cold containers
    system_prompt=(
        "You are an academic research assistant. "
        # Evidence-first mandate: every claim must come from the supplied context.
        "Answer ONLY using the evidence provided in the user's message and in the provided context. "
        # Hallucination block: explicitly forbids adding external facts.
        "Do NOT introduce external knowledge or assumptions not present in the provided text. "
        # Graceful degradation: if evidence is missing, say so rather than guessing.
        "If the answer cannot be found in the provided evidence, state exactly what is missing. "
        # Output style: formal academic prose structured as a thorough essay.
        "Write in formal academic prose. Produce thorough, well-structured essays "
        "that are fully grounded in the supplied source material."
    )
)


# ── Sentence embedding model with fallback chain ──────────────────────────────
# The embedding model converts text into dense vectors for similarity search.
# We try three options in order: preferred → fallback → mock (offline).
try:
    # Primary model: BAAI/bge-small-en-v1.5 — fast, high-quality English embeddings.
    # Outputs 384-dimensional vectors.  Downloaded from HuggingFace on first use.
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print("✅ Primary embedding model loaded: BAAI/bge-small-en-v1.5")
except Exception as e1:
    print(f"⚠️ Primary embedding load failed: {e1}")
    try:
        # Fallback model: all-MiniLM-L6-v2 — slightly older but very widely cached.
        # Often available even when HuggingFace Hub connectivity is restricted.
        embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("✅ Fallback embedding model loaded: all-MiniLM-L6-v2")
    except Exception as e2:
        # Last resort: MockEmbedding returns random 384-dim vectors.
        # Retrieval quality is meaningless but the pipeline won't crash.
        # ONLY use this to test the plumbing when HuggingFace is unreachable.
        from llama_index.core.embeddings import MockEmbedding
        embed_model = MockEmbedding(embed_dim=384)  # 384 matches bge-small and MiniLM output dims
        print(f"⚠️ HF downloads unavailable; using MockEmbedding fallback ({e2}).")


# ── Register models globally in LlamaIndex Settings ──────────────────────────
# Any LlamaIndex component created after this line (indices, retrievers, pipelines)
# will automatically use these models unless overridden locally.
Settings.llm = llm
Settings.embed_model = embed_model


# ── Patch Replicate SDK client to enforce 600-second read timeout ─────────────
# LlamaIndex's request_timeout=600.0 is silently ignored in replicate SDK >=0.25
# because the timeout must now be set on the Client object, not passed to run().
# Replacing default_client here ensures ALL replicate calls (both LlamaIndex-
# routed via llm and direct replicate.run() calls in Cell 14) use the correct
# 600 s read timeout so cold-start containers have time to warm up.
import httpx as _httpx
import replicate as _rep_sdk
try:
    _rep_sdk.default_client = _rep_sdk.Client(
        api_token=replicate_token,
        timeout=_httpx.Timeout(connect=10.0, read=600.0, write=60.0, pool=10.0),
    )
    print("✅ Replicate SDK client patched: 600 s read timeout enforced.")
except Exception as _te:
    print(f"⚠️ Could not patch Replicate client timeout: {_te}")

print("🚀 Granite 3.1 Ready with Academic Shield (Timeout: 600s | Essay Mode)")


2026-05-22 17:21:12,753 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


✅ Windows certificate store bridge enabled (certifi-win32).
[2026-05-22 17:21:12] INFO: 17:21:12 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


2026-05-22 17:21:13,333 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 17:21:13,403 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


[2026-05-22 17:21:13] INFO: 17:21:13 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-22 17:21:13] INFO: 17:21:13 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


2026-05-22 17:21:13,757 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 17:21:13,835 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

[2026-05-22 17:21:18] INFO: 17:21:18 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


2026-05-22 17:21:18,860 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 17:21:18,933 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


[2026-05-22 17:21:18] INFO: 17:21:18 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-05-22 17:21:18] INFO: 17:21:18 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


2026-05-22 17:21:19,276 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-05-22 17:21:19,426 - INFO - 1 prompt is loaded, with the key: query


[2026-05-22 17:21:19] INFO: 17:21:19 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
[2026-05-22 17:21:19] INFO: 17:21:19 - INFO - 1 prompt is loaded, with the key: query
✅ Primary embedding model loaded: BAAI/bge-small-en-v1.5
✅ Replicate SDK client patched: 600 s read timeout enforced.
🚀 Granite 3.1 Ready with Academic Shield (Timeout: 600s | Essay Mode)


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Step 3: Automated PDF Ingestion from Google Drive
# Accepts one or more comma-separated Drive share links.
# Each PDF is downloaded via gdown (primary) with a requests fallback.
# Failed links are skipped with a warning; successful PDFs are merged into
# a single source_material.pdf for the rest of the pipeline.
# ─────────────────────────────────────────────────────────────────────────────

import os
import re
import requests


# ── Helper 1: extract_drive_file_id ──────────────────────────────────────────
def extract_drive_file_id(drive_url: str) -> str:
    """Extract Google Drive file id from common share URL formats."""
    patterns = [
        r"/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, drive_url)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract a Google Drive file id from: {drive_url!r}")


# ── Helper 2: download_pdf_from_drive ────────────────────────────────────────
def download_pdf_from_drive(drive_url: str, save_path: str) -> None:
    """Download a Drive-shared PDF to *save_path*.

    Strategy (in order):
      1. gdown  — handles virus-scan interstitials and auth cookies automatically.
      2. requests (two endpoints) — manual confirm-token fallback.

    Raises ValueError if the saved artifact is not a valid PDF after all attempts.
    """
    file_id = extract_drive_file_id(drive_url)

    # ── Attempt 1: gdown ─────────────────────────────────────────────────────
    gdown_ok = False
    try:
        import gdown  # pip install gdown  (already in .venv)
        # gdown 6.x removed the fuzzy parameter; use id= directly.
        result = gdown.download(id=file_id, output=save_path, quiet=True)
        if result is not None and os.path.exists(save_path):
            with open(save_path, "rb") as f:
                header = f.read(5)
            if header == b"%PDF-":
                console_log(f"[gdown] Document secured: {save_path}", "OK")
                gdown_ok = True
            else:
                # gdown returned something but it isn't a PDF (auth page, etc.)
                os.remove(save_path)
                console_log(f"[gdown] Non-PDF response for {file_id}; trying requests fallback.", "WARN")
    except Exception as exc:
        console_log(f"[gdown] Error ({exc}); trying requests fallback.", "WARN")

    if gdown_ok:
        return  # Done — skip the requests path

    # ── Attempt 2: requests (two endpoints, confirm-token aware) ─────────────
    session = requests.Session()
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept": "text/html,application/pdf,application/octet-stream,*/*",
        "Referer": "https://drive.google.com/",
    }
    endpoints = [
        ("https://drive.usercontent.google.com/download", {"id": file_id, "export": "download", "confirm": "t"}),
        ("https://drive.google.com/uc",                   {"export": "download", "id": file_id, "confirm": "t"}),
    ]

    last_error = None
    for base_url, params in endpoints:
        try:
            response = session.get(
                base_url, params=params, headers=headers,
                stream=True, allow_redirects=True, timeout=90,
            )

            # Confirm-token from cookie (large-file warning page)
            token = None
            for key, val in response.cookies.items():
                if key.startswith("download_warning"):
                    token = val
                    break

            # Confirm-token from HTML body
            if not token and "text/html" in response.headers.get("Content-Type", "").lower():
                html = response.content.decode("utf-8", errors="ignore")
                m = re.search(r"confirm=([0-9A-Za-z_-]+)", html)
                if m:
                    token = m.group(1)

            if token:
                p2 = dict(params)
                p2["confirm"] = token
                response = session.get(
                    base_url, params=p2, headers=headers,
                    stream=True, allow_redirects=True, timeout=90,
                )

            if response.status_code in (401, 403):
                last_error = PermissionError(
                    f"HTTP {response.status_code}: Drive blocked download for file id={file_id}. "
                    "Set sharing to 'Anyone with the link: Viewer'."
                )
                continue

            response.raise_for_status()

            with open(save_path, "wb") as fh:
                for chunk in response.iter_content(chunk_size=32768):
                    if chunk:
                        fh.write(chunk)

            with open(save_path, "rb") as fh:
                header = fh.read(5)

            if header == b"%PDF-":
                console_log(f"[requests] Document secured: {save_path}", "OK")
                return

            # Not a PDF — save a preview for debugging and try next endpoint.
            with open(save_path, "rb") as fh:
                preview = fh.read(300).decode("utf-8", errors="ignore")
            os.remove(save_path)
            last_error = ValueError(
                f"Non-PDF response for file id={file_id} from {base_url}. "
                "Ensure sharing is 'Anyone with the link: Viewer'. "
                f"Preview: {preview[:120]!r}"
            )

        except Exception as exc:
            last_error = exc

    # All attempts failed.
    raise last_error or RuntimeError(f"All download attempts failed for file id={file_id}.")


# ── Helper 3: extract_text_or_ocr ────────────────────────────────────────────
def extract_text_or_ocr(pdf_path: str, dpi: int = 300) -> str:
    """PyMuPDF native extraction with Tesseract OCR fallback.

    Returns the path that downstream cells should read:
      - The original PDF path if it contains extractable text.
      - A .ocr.txt path if OCR was required.
    """
    try:
        import fitz
    except Exception:
        fitz = None
        console_log("PyMuPDF not available; OCR fallback may be required.", "WARN")

    extracted_text = ""
    if fitz is not None:
        try:
            with fitz.open(pdf_path) as doc:
                for page in doc:
                    extracted_text += page.get_text() + "\n"
        except Exception as exc:
            console_log(f"PyMuPDF extraction error: {exc}", "WARN")
            extracted_text = ""

    if len(extracted_text.strip()) > 50:
        console_log("PDF contains extractable text (PyMuPDF).", "OK")
        return pdf_path

    console_log("No extractable text found; using OCR fallback.", "WARN")
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except Exception:
        console_log("OCR packages missing (pdf2image/pytesseract/Poppler/Tesseract).", "ERROR")
        return pdf_path

    tess_cmd = resolve_tesseract_cmd() if "resolve_tesseract_cmd" in globals() else None
    if tess_cmd:
        console_log(f"Using Tesseract executable: {tess_cmd}", "INFO")

    try:
        _ = pytesseract.get_tesseract_version()
    except Exception:
        console_log("Tesseract executable not found. Install it and rerun Step 3.", "ERROR")
        return pdf_path

    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        ocr_text = "".join(pytesseract.image_to_string(img) + "\n" for img in images)
        txt_path = pdf_path + ".ocr.txt"
        with open(txt_path, "w", encoding="utf-8") as fh:
            fh.write(ocr_text)
        console_log(f"OCR complete: {txt_path}", "OK")
        return txt_path
    except Exception as exc:
        console_log(f"OCR failed: {exc}", "ERROR")
        return pdf_path


# ── Main ──────────────────────────────────────────────────────────────────────
raw_input_links = input(
    "📌 Paste Google Drive Link(s) — separate multiple links with commas: "
).strip()

drive_links = [lnk.strip() for lnk in raw_input_links.split(",") if lnk.strip()]
console_log(f"Received {len(drive_links)} link(s).", "INFO")

DATA_DIR = "academic_data"
os.makedirs(DATA_DIR, exist_ok=True)
pdf_path = os.path.join(DATA_DIR, "source_material.pdf")
source_file = None

if len(drive_links) == 1:
    # ── Single link ───────────────────────────────────────────────────────────
    download_pdf_from_drive(drive_links[0], pdf_path)

else:
    # ── Multiple links — download each with per-link error isolation ──────────
    part_paths = []
    failed_links = []

    for idx, link in enumerate(drive_links, start=1):
        part_path = os.path.join(DATA_DIR, f"source_material_part{idx}.pdf")
        console_log(f"Downloading document {idx}/{len(drive_links)} …", "INFO")
        try:
            download_pdf_from_drive(link, part_path)
            part_paths.append(part_path)
        except Exception as exc:
            console_log(
                f"Skipping link {idx}/{len(drive_links)} — {exc} "
                "(set sharing to 'Anyone with the link: Viewer' and retry this link).",
                "ERROR",
            )
            failed_links.append((idx, link, str(exc)))

    if not part_paths:
        raise RuntimeError(
            "All links failed. Verify that every file is shared as "
            "'Anyone with the link: Viewer' and re-run this cell."
        )

    if failed_links:
        console_log(
            f"{len(failed_links)} link(s) failed and were skipped: "
            + ", ".join(str(i) for i, *_ in failed_links),
            "WARN",
        )

    # ── Merge successful PDFs ─────────────────────────────────────────────────
    merged_ok = False
    try:
        import fitz
        merged_doc = fitz.open()
        for path in part_paths:
            with fitz.open(path) as part_doc:
                merged_doc.insert_pdf(part_doc)
        merged_doc.save(pdf_path)
        merged_doc.close()
        console_log(f"Merged {len(part_paths)} PDF(s) → {pdf_path}", "OK")
        merged_ok = True
    except Exception as exc:
        console_log(f"PDF merge failed ({exc}); falling back to text concatenation.", "WARN")

    if not merged_ok:
        txt_path = pdf_path + ".ocr.txt"
        with open(txt_path, "w", encoding="utf-8") as out_f:
            for path in part_paths:
                part_src = extract_text_or_ocr(path)
                with open(part_src, "r", encoding="utf-8", errors="ignore") as in_f:
                    out_f.write(in_f.read() + "\n")
        source_file = txt_path
        console_log(f"Using concatenated text: {source_file}", "OK")

# Extract text / OCR unless it was already set by the text-concatenation fallback.
if source_file is None:
    source_file = extract_text_or_ocr(pdf_path)

console_log(f"Using source file: {source_file}", "OK")
2

[2026-05-22 17:23:23] INFO: Received 31 link(s).
[2026-05-22 17:23:23] INFO: Downloading document 1/31 …
[2026-05-22 17:23:30] OK: [gdown] Document secured: academic_data\source_material_part1.pdf
[2026-05-22 17:23:30] INFO: Downloading document 2/31 …
[2026-05-22 17:23:36] OK: [gdown] Document secured: academic_data\source_material_part2.pdf
[2026-05-22 17:23:36] INFO: Downloading document 3/31 …
[2026-05-22 17:23:42] OK: [gdown] Document secured: academic_data\source_material_part3.pdf
[2026-05-22 17:23:42] INFO: Downloading document 4/31 …
[2026-05-22 17:23:48] OK: [gdown] Document secured: academic_data\source_material_part4.pdf
[2026-05-22 17:23:48] INFO: Downloading document 5/31 …
[2026-05-22 17:23:53] OK: [gdown] Document secured: academic_data\source_material_part5.pdf
[2026-05-22 17:23:53] INFO: Downloading document 6/31 …
[2026-05-22 17:24:00] OK: [gdown] Document secured: academic_data\source_material_part6.pdf
[2026-05-22 17:24:00] INFO: Downloading document 7/31 …
[2026-0

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Step 4: Semantic Chunking + Memory Palace Metadata Labelling
# This cell takes the ingested source file, splits it into semantically
# coherent chunks (nodes), attaches four-level hierarchy metadata to each
# node, and builds a lightweight current-document query engine for fast
# in-session retrieval without touching the persistent Memory Palace.
# ─────────────────────────────────────────────────────────────────────────────

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
# SemanticSplitterNodeParser splits documents at natural topic boundaries
# rather than fixed character counts.  It uses the embedding model to detect
# where the semantic meaning shifts and places chunk boundaries there.
from llama_index.core.node_parser import SemanticSplitterNodeParser


# ── Load document(s) ─────────────────────────────────────────────────────────
# `source_file` comes from the previous cell and can be either a PDF path or
# a .ocr.txt path depending on what extract_text_or_ocr() returned.
# SimpleDirectoryReader understands both formats automatically.
documents = SimpleDirectoryReader(input_files=[source_file]).load_data()


# ── Build default metadata labels from the filename ──────────────────────────
# These defaults give every ingested document a sensible home in the Memory
# Palace hierarchy even if the user skips the prompts below.
file_name = os.path.basename(source_file)  # e.g. "source_material.pdf" or "source_material.pdf.ocr.txt"

# Top-level grouping: defaults to "Projects" — change to "People", "Papers", etc.
default_wing = "Projects"

# Mid-level grouping: derived from the filename (without extension) using
# spaces-to-underscores so it works cleanly as a ChromaDB metadata value.
default_room = os.path.splitext(file_name)[0].replace(" ", "_") or "General_Topic"

# Evidence container inside the room: defaults to "Research_Evidence".
default_hall = "Research_Evidence"

# Content type tag: "Raw_Text" for OCR output, "PDF_Text" for native PDF text.
default_drawer_type = "Raw_Text" if source_file.endswith(".txt") else "PDF_Text"


# ── Interactive label prompts (press Enter to accept defaults) ────────────────
# The user can customise all four hierarchy levels before ingestion.
# This is useful when ingesting multiple PDFs into different rooms.
wing_label  = input(f"🪽 Wing label [{default_wing}]: ").strip()  or default_wing
room_label  = input(f"🚪 Room label [{default_room}]: ").strip()  or default_room
hall_label  = input(f"🏛️ Hall label [{default_hall}]: ").strip()  or default_hall
drawer_type = input(f"🗄️ Drawer type [{default_drawer_type}]: ").strip() or default_drawer_type


# ── Attach stable document IDs and metadata to each raw Document object ───────
# Stable doc IDs (file_name::doc::N) are required for the UPSERT ingestion
# strategy in Step 4.5.  Without stable IDs, re-running this cell would
# insert duplicate vectors in ChromaDB instead of updating existing ones.
for i, doc in enumerate(documents):
    doc.doc_id = f"{file_name}::doc::{i}"  # Unique, deterministic ID per document chunk
    # The metadata dict is preserved on every node derived from this document,
    # making it queryable later via ChromaDB metadata filters.
    doc.metadata["source"]      = file_name     # Original filename (for citation display)
    doc.metadata["file_name"]   = file_name     # Alias — some LlamaIndex paths use this key
    doc.metadata["wing"]        = wing_label    # Top-level palace hierarchy label
    doc.metadata["room"]        = room_label    # Mid-level palace hierarchy label
    doc.metadata["hall"]        = hall_label    # Evidence-level palace hierarchy label
    doc.metadata["drawer_type"] = drawer_type   # Content-type tag


# ── Semantic splitter configuration ──────────────────────────────────────────
splitter = SemanticSplitterNodeParser(
    buffer_size=3,  # Number of sentences to consider on each side of a potential boundary.
                    # A buffer of 3 gives the model enough context to detect topic shifts
                    # without creating overly large intermediate comparison windows.

    breakpoint_percentile_threshold=95,  # Only split when the cosine-distance between
                                         # adjacent sentence groups exceeds the 95th percentile
                                         # of all distances in the document.
                                         # Higher = fewer, larger chunks.
                                         # Lower  = more, smaller chunks.

    embed_model=embed_model,  # The same embedding model registered in Cell 4 — using the
                              # same model for chunking and retrieval ensures the vector space
                              # is consistent across both operations.
)


# ── Split documents into semantic nodes ──────────────────────────────────────
# Each node is a coherent text chunk with its own embedding and metadata.
# Nodes are the atomic retrieval unit — what gets stored in ChromaDB and
# returned by the retriever during the Q&A loop.
nodes = splitter.get_nodes_from_documents(documents)

# Re-apply the hierarchy metadata to every node because SemanticSplitterNodeParser
# propagates document metadata by default, but explicit re-application ensures
# it stays consistent even if the splitter is updated or replaced.
for n in nodes:
    n.metadata["source"]      = file_name
    n.metadata["file_name"]   = file_name
    n.metadata["wing"]        = wing_label
    n.metadata["room"]        = room_label
    n.metadata["hall"]        = hall_label
    n.metadata["drawer_type"] = drawer_type


# ── Build a current-document in-memory vector index ──────────────────────────
# This index holds only the nodes from the current PDF (not the full palace).
# It is used in the Q&A loop to provide fast, precise citations from the
# document the user just uploaded, before cross-document palace retrieval kicks in.
current_doc_index = VectorStoreIndex(nodes, embed_model=embed_model)

# Wrap the index in a query engine for direct question answering.
# similarity_top_k=5  — retrieve the 5 most similar nodes per query.
# response_mode="compact" — LlamaIndex condenses multiple node texts into one
#   response rather than answering once per node (reduces token usage).
# streaming=False — return the complete answer string; don't stream tokens.
current_doc_query_engine = current_doc_index.as_query_engine(
    similarity_top_k=5,
    response_mode="compact",
    streaming=False,
)

console_log(
    (
        f"Created {len(nodes)} semantic nodes from {file_name} "
        f"in wing='{wing_label}', room='{room_label}', hall='{hall_label}', drawer='{drawer_type}'."
    ),
    "OK",
)


[2026-05-22 17:59:22] OK: Created 3126 semantic nodes from source_material.pdf in wing='Academics', room='Adaptive Leadership', hall='Adaptive Leadership', drawer='Adv. Dip'.


In [10]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Step 4.5: MemPalace Bridge (Persistent Long-Term Memory + UPSERTS)
# This cell persists the current document's nodes into ChromaDB — the
# "Memory Palace" — so they survive kernel restarts and can be retrieved
# alongside nodes from every previously ingested PDF.
#
# The UPSERT strategy prevents duplicate vectors: if the same source file is
# ingested again, existing vectors are updated in-place rather than appended.
#
# Two automatic fallback modes handle edge cases where the LlamaIndex
# ingestion pipeline doesn't move the collection count:
#   Case A — pipeline ran but ChromaDB count didn't increase → explicit upsert
#   Case B — pipeline produced zero nodes AND collection is empty → bootstrap
#             from the Step 4 semantic nodes as a safety net.
# ─────────────────────────────────────────────────────────────────────────────

import os
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline, DocstoreStrategy
from llama_index.core.storage.docstore import SimpleDocumentStore

# ── Configuration ─────────────────────────────────────────────────────────────
PALACE_DB_PATH       = os.getenv("PALACE_DB_PATH", "./mempalace_db")
PALACE_COLLECTION    = os.getenv("PALACE_COLLECTION", "academic_palace")
PALACE_DOCSTORE_PATH = os.path.join(PALACE_DB_PATH, "docstore.json")

# ── Guard: catch dead Rust bindings on the real client creation ───────────────
# Do NOT use a probe that calls _system.stop() — that itself kills the
# Rust DLL for the whole process.  Instead, catch errors directly here.
_chroma_ok          = True
palace_client       = None
palace_collection   = None
palace_vector_store = None
try:
    palace_client     = chromadb.PersistentClient(path=PALACE_DB_PATH)
    palace_collection = palace_client.get_or_create_collection(
        PALACE_COLLECTION,
        metadata={"hnsw:space": "cosine"},
    )
    palace_vector_store = ChromaVectorStore(chroma_collection=palace_collection)
except (AttributeError, ValueError):
    _chroma_ok = False

if not _chroma_ok:
    print("=" * 60)
    print("⚠️  ChromaDB Rust bindings are INVALIDATED in this kernel.")
    print()
    print("   The Memory Palace Reset cell (Cell 2) called _system.stop()")
    print("   which permanently kills the Rust layer for this session.")
    print()
    print("   ♻️  RESTART THE KERNEL, then run in order:")
    print("       Cell 1  → environment setup")
    print("       Cell 4  → console logger")
    print("       Cell 5  → OCR diagnostics")
    print("       Cell 3  → batch ingest  (or re-run the single-doc flow)")
    print("=" * 60)
else:
    if os.path.exists(PALACE_DOCSTORE_PATH):
        palace_docstore = SimpleDocumentStore.from_persist_path(PALACE_DOCSTORE_PATH)
    else:
        palace_docstore = SimpleDocumentStore()

    # ── Lazy-init splitter (Cell 8 may have been skipped) ────────────────────
    if "splitter" not in globals() or splitter is None:
        from llama_index.core.node_parser import SemanticSplitterNodeParser
        splitter = SemanticSplitterNodeParser(
            buffer_size=3,
            breakpoint_percentile_threshold=95,
            embed_model=embed_model,
        )
        console_log("splitter created (Cell 8 was skipped — lazy init).", "WARN")

    # ── Check whether Cell 8 produced documents to ingest ────────────────────
    _has_documents = "documents" in globals() and documents
    _has_source    = "source_file" in globals() and source_file

    if not _has_documents:
        # Cell 8 was skipped (e.g. the user ran batch ingest instead).
        # Skip the pipeline and just reconnect palace_index to the existing store.
        console_log(
            "No documents from Cell 8 — skipping pipeline, reconnecting palace_index only.",
            "WARN",
        )
        before_count         = palace_collection.count()
        after_pipeline_count = before_count
        after_final_count    = before_count
        pipeline_nodes       = []
        fallback_used        = False
        fallback_mode        = "none"
        fallback_upserts     = 0

        palace_index = VectorStoreIndex.from_vector_store(
            palace_vector_store,
            embed_model=embed_model,
        )
        palace_query_engine = palace_index.as_query_engine(
            similarity_top_k=8,
            response_mode="compact",
            streaming=False,
        )
        console_log(
            f"MemPalace connected at '{PALACE_DB_PATH}' "
            f"(collection: '{PALACE_COLLECTION}', vectors: {before_count}).",
            "OK",
        )
    else:
        # ── _build_chroma_payload: convert LlamaIndex nodes to raw ChromaDB format
        def _build_chroma_payload(nodes_for_payload):
            payload_ids        = []
            payload_docs       = []
            payload_metas      = []
            payload_embeddings = []

            for idx, node in enumerate(nodes_for_payload):
                try:
                    node_text = node.get_content(metadata_mode="none")
                except Exception:
                    node_text = node.get_content() if hasattr(node, "get_content") else ""

                if not node_text or not node_text.strip():
                    continue

                meta = dict(getattr(node, "metadata", {}) or {})
                meta["source"]      = meta.get("source",      os.path.basename(source_file))
                meta["file_name"]   = meta.get("file_name",   os.path.basename(source_file))
                meta["wing"]        = meta.get("wing",        globals().get("wing_label",  "Projects"))
                meta["room"]        = meta.get("room",        globals().get("room_label",  "General_Topic"))
                meta["hall"]        = meta.get("hall",        globals().get("hall_label",  "Research_Evidence"))
                meta["drawer_type"] = meta.get("drawer_type", globals().get("drawer_type", "PDF_Text"))

                node_id = (
                    getattr(node, "node_id", None)
                    or getattr(node, "id_", None)
                    or f"{os.path.basename(source_file)}::node::{idx}"
                )

                embedding = getattr(node, "embedding", None)
                if embedding is None:
                    embedding = embed_model.get_text_embedding(node_text)

                payload_ids.append(str(node_id))
                payload_docs.append(node_text)
                payload_metas.append(meta)
                payload_embeddings.append(embedding)

            return payload_ids, payload_docs, payload_metas, payload_embeddings

        # ── Record palace size before ingestion ───────────────────────────────
        before_count = palace_collection.count()

        # ── Run the ingestion pipeline with UPSERT deduplication ──────────────
        palace_pipeline = IngestionPipeline(
            transformations=[splitter],
            docstore=palace_docstore,
            vector_store=palace_vector_store,
            docstore_strategy=DocstoreStrategy.UPSERTS,
        )

        pipeline_nodes = palace_pipeline.run(documents=documents)

        for n in pipeline_nodes:
            n.metadata["source"]      = n.metadata.get("source",      os.path.basename(source_file))
            n.metadata["file_name"]   = n.metadata.get("file_name",   os.path.basename(source_file))
            n.metadata["wing"]        = n.metadata.get("wing",        globals().get("wing_label",  "Projects"))
            n.metadata["room"]        = n.metadata.get("room",        globals().get("room_label",  "General_Topic"))
            n.metadata["hall"]        = n.metadata.get("hall",        globals().get("hall_label",  "Research_Evidence"))
            n.metadata["drawer_type"] = n.metadata.get("drawer_type", globals().get("drawer_type", "PDF_Text"))

        # ── Persist the docstore to disk ──────────────────────────────────────
        os.makedirs(PALACE_DB_PATH, exist_ok=True)
        palace_docstore.persist(persist_path=PALACE_DOCSTORE_PATH)

        # ── Check whether the pipeline actually wrote vectors ─────────────────
        after_pipeline_count = palace_collection.count()
        fallback_used    = False
        fallback_mode    = "none"
        fallback_upserts = 0

        # ── Fallback Case A: pipeline ran but count didn't increase ───────────
        if pipeline_nodes and after_pipeline_count <= before_count:
            f_ids, f_docs, f_metas, f_embeddings = _build_chroma_payload(pipeline_nodes)
            if f_ids:
                palace_collection.upsert(
                    ids=f_ids, documents=f_docs,
                    metadatas=f_metas, embeddings=f_embeddings,
                )
                fallback_used    = True
                fallback_mode    = "pipeline-nodes"
                fallback_upserts = len(f_ids)

        # ── Fallback Case B: zero nodes AND collection is empty ───────────────
        if not fallback_used and not pipeline_nodes and after_pipeline_count == 0 and globals().get("nodes"):
            b_ids, b_docs, b_metas, b_embeddings = _build_chroma_payload(globals()["nodes"])
            if b_ids:
                palace_collection.upsert(
                    ids=b_ids, documents=b_docs,
                    metadatas=b_metas, embeddings=b_embeddings,
                )
                fallback_used    = True
                fallback_mode    = "bootstrap-step4-nodes"
                fallback_upserts = len(b_ids)

        after_final_count = palace_collection.count()

        # ── Re-open a queryable index from the populated vector store ─────────
        palace_index = VectorStoreIndex.from_vector_store(
            palace_vector_store,
            embed_model=embed_model,
        )

        palace_query_engine = palace_index.as_query_engine(
            similarity_top_k=8,
            response_mode="compact",
            streaming=False,
        )

        # ── Summary logging ───────────────────────────────────────────────────
        console_log(
            f"MemPalace connected at '{PALACE_DB_PATH}' (collection: '{PALACE_COLLECTION}').",
            "OK",
        )
        console_log(
            (
                f"UPSERT ingestion complete: {len(pipeline_nodes)} nodes processed "
                f"for wing='{globals().get('wing_label', 'Projects')}', "
                f"room='{globals().get('room_label', 'General_Topic')}', "
                f"hall='{globals().get('hall_label', 'Research_Evidence')}', "
                f"drawer='{globals().get('drawer_type', 'PDF_Text')}'."
            ),
            "OK",
        )
        console_log(
            f"Palace count check: before={before_count}, after_pipeline={after_pipeline_count}, final={after_final_count}",
            "INFO",
        )
        if fallback_used:
            console_log(
                f"Fallback '{fallback_mode}' upsert executed for {fallback_upserts} nodes.",
                "WARN",
            )


KeyboardInterrupt: 

In [10]:

# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Step 4.6: Memory Stats Dashboard (Hierarchy Expanded + Lightweight Scan)
# Prints a summary of everything currently stored in the Memory Palace so you
# can verify ingestion was successful and understand the distribution of your
# academic material across wings, rooms, halls, and drawer types.
#
# To avoid performance issues on very large collections, the scan is capped by
# PALACE_DASHBOARD_MAX_SCAN (default 1500 records).  Set the env var higher
# if you want full coverage on a large palace.
# ─────────────────────────────────────────────────────────────────────────────

from collections import Counter
import os

# ── Guard: palace_collection must be defined by Cell 9 ───────────────────────
# If Cell 9 detected dead Rust bindings it skipped setting palace_collection.
# In that case we bail out with a clear message rather than a NameError.
if "palace_collection" not in globals() or palace_collection is None:
    print("=" * 60)
    print("⚠️  palace_collection is not available.")
    print()
    print("   This means Cell 9 (MemPalace Bridge) could not connect to")
    print("   ChromaDB — most likely because the Rust bindings are")
    print("   invalidated in this kernel session.")
    print()
    print("   ♻️  RESTART THE KERNEL, then run in order:")
    print("       Cell 1  → environment setup")
    print("       Cell 4  → console logger")
    print("       Cell 5  → OCR diagnostics")
    print("       Cell 3  → batch ingest (or re-run the single-doc flow)")
    print("       Cell 9  → MemPalace Bridge")
    print("       Cell 10 → this dashboard")
    print("=" * 60)
else:
    # ── Total vector count ────────────────────────────────────────────────────
    palace_count = palace_collection.count()
    print(f"🏰 Palace Stats: {palace_count} semantic memories stored.")
    print(f"📂 Current Collection: {palace_collection.name}")

    # ── Scan limits ───────────────────────────────────────────────────────────
    DASHBOARD_MAX_SCAN     = int(os.getenv("PALACE_DASHBOARD_MAX_SCAN",    "1500"))
    DASHBOARD_PREVIEW_LIMIT = int(os.getenv("PALACE_DASHBOARD_PREVIEW_LIMIT", "8"))

    try:
        scan_limit    = min(palace_count, DASHBOARD_MAX_SCAN) if palace_count > 0 else 0
        all_metadatas = []

        if scan_limit > 0:
            all_meta_payload = palace_collection.get(limit=scan_limit, include=["metadatas"])
            if isinstance(all_meta_payload, dict):
                all_metadatas = all_meta_payload.get("metadatas", []) or []

        # ── Count distribution across each hierarchy level ────────────────────
        wing_counter   = Counter((m or {}).get("wing",        "Projects")          for m in all_metadatas)
        room_counter   = Counter((m or {}).get("room",        "General_Topic")     for m in all_metadatas)
        hall_counter   = Counter((m or {}).get("hall",        "Research_Evidence") for m in all_metadatas)
        drawer_counter = Counter((m or {}).get("drawer_type", "PDF_Text")          for m in all_metadatas)

        if palace_count > scan_limit:
            print(
                f"ℹ️ Dashboard scanned {scan_limit}/{palace_count} memories "
                f"(set PALACE_DASHBOARD_MAX_SCAN env var for deeper scans)."
            )

        if wing_counter:
            print("\n🪽 Wing distribution:")
            for wing, cnt in wing_counter.most_common():
                print(f"  - {wing}: {cnt}")

        if room_counter:
            print("\n🚪 Top rooms:")
            for room, cnt in room_counter.most_common(10):
                print(f"  - {room}: {cnt}")

        if hall_counter:
            print("\n🏛️ Hall distribution:")
            for hall, cnt in hall_counter.most_common():
                print(f"  - {hall}: {cnt}")

        if drawer_counter:
            print("\n🗄️ Drawer types:")
            for drawer, cnt in drawer_counter.most_common():
                print(f"  - {drawer}: {cnt}")

        # ── Sample memories preview ───────────────────────────────────────────
        preview   = palace_collection.peek(limit=DASHBOARD_PREVIEW_LIMIT)
        metadatas = preview.get("metadatas", []) if isinstance(preview, dict) else []

        if metadatas:
            print("\n🧾 Sample memories:")
            for i, meta in enumerate(metadatas[:DASHBOARD_PREVIEW_LIMIT], start=1):
                m = meta or {}
                print(
                    f"  {i}. file={m.get('file_name', 'Unknown')} | "
                    f"wing={m.get('wing', 'Projects')} | "
                    f"room={m.get('room', 'General_Topic')} | "
                    f"hall={m.get('hall', 'Research_Evidence')} | "
                    f"drawer={m.get('drawer_type', 'PDF_Text')}"
                )

    except Exception as e:
        print(f"⚠️ Could not preview memory metadata: {e}")


⚠️  palace_collection is not available.

   This means Cell 9 (MemPalace Bridge) could not connect to
   ChromaDB — most likely because the Rust bindings are
   invalidated in this kernel session.

   ♻️  RESTART THE KERNEL, then run in order:
       Cell 1  → environment setup
       Cell 4  → console logger
       Cell 5  → OCR diagnostics
       Cell 3  → batch ingest (or re-run the single-doc flow)
       Cell 9  → MemPalace Bridge
       Cell 10 → this dashboard


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Step 4.6b: Label Index Retriever
# Two capabilities in one cell:
#   1. get_label_index()     — scans ChromaDB and prints all unique wing/room/hall
#                              combinations, so you know what's in the palace.
#   2. retrieve_by_label()   — fetches stored chunks filtered by any combination
#                              of wing, room, and/or hall labels, with an optional
#                              semantic similarity query on top of the filter.
# Use this cell to explore palace contents and verify retrieval before running
# the full research loop.
# ─────────────────────────────────────────────────────────────────────────────

from collections import defaultdict  # A dict subclass that auto-initialises missing keys
import os                            # Used inside retrieve_by_label for query fallback


# ── 1. Build the full label index ─────────────────────────────────────────────
def get_label_index(collection, max_scan: int = 5000) -> dict:
    """Return all unique (wing, room, hall, file_name) combinations in the palace.

    Scans up to max_scan metadata records from ChromaDB and aggregates them
    into a dict keyed by (wing, room, hall) tuples, where each value is the
    set of file names that live at that hierarchical address.

    Args:
        collection: The ChromaDB collection object to scan.
        max_scan:   Maximum number of records to inspect.  5000 covers most
                    real-world palace sizes without excessive memory usage.
    Returns:
        dict mapping (wing, room, hall) → set of file_name strings.
        Returns an empty dict if the palace is empty.
    """
    count = collection.count()  # Total vectors currently in the palace
    if count == 0:
        print("⚠️ Palace is empty. Run Steps 4 and 4.5 first.")
        return {}

    # Never request more records than actually exist.
    limit = min(count, max_scan)

    # Request only the metadata — we don't need the raw text or embeddings here.
    payload = collection.get(limit=limit, include=["metadatas"])
    metadatas = (payload or {}).get("metadatas", []) or []

    # defaultdict(set) auto-creates a set for any new key, so we can .add() without
    # first checking whether the key exists.
    index = defaultdict(set)
    for m in metadatas:
        m = m or {}  # Guard against None entries in the metadata list
        wing  = m.get("wing",        "Projects")
        room  = m.get("room",        "General_Topic")
        hall  = m.get("hall",        "Research_Evidence")
        fname = m.get("file_name",   "Unknown")
        # Key: the three-level address in the hierarchy.  Value: all files at that address.
        index[(wing, room, hall)].add(fname)

    return dict(index)  # Convert to plain dict for clean printing and serialisation


# Run the index scan against the already-connected palace_collection.
label_index = get_label_index(palace_collection)

print(f"\n📚 LABEL INDEX — {len(label_index)} unique wing/room/hall combination(s) in palace:\n")
for (wing, room, hall), files in sorted(label_index.items()):  # sorted() gives alphabetical output
    print(f"  Wing: {wing}  |  Room: {room}  |  Hall: {hall}")
    for f in sorted(files):  # Sort filenames too for consistent output
        print(f"    └─ {f}")  # Tree-style formatting makes the hierarchy visually clear


# ── 2. Retrieve stored chunks by label ────────────────────────────────────────
def retrieve_by_label(
    wing: str | None = None,
    room: str | None = None,
    hall: str | None = None,
    query: str = "",
    top_k: int = 5,
    show_text: bool = True,
) -> list[dict]:
    """Retrieve stored chunks from the palace filtered by label(s).

    Combines ChromaDB metadata filtering with LlamaIndex semantic similarity search.
    Any combination of wing/room/hall can be supplied — unspecified dimensions
    are not filtered (i.e. all values pass).  An empty query falls back to the
    concatenation of supplied labels as the semantic search string.

    Args:
        wing:      Filter by wing label.  None means no wing filter.
        room:      Filter by room label.  None means no room filter.
        hall:      Filter by hall label.  None means no hall filter.
        query:     Semantic search string.  Leave empty to rely solely on metadata filters.
        top_k:     Number of results to return (passed to the retriever).
        show_text: Whether to print results to cell output.

    Returns:
        List of dicts, each with: file_name, wing, room, hall, drawer_type, text.
    """
    # Import filter primitives from LlamaIndex's vector store module.
    # MetadataFilter: a single key=value constraint.
    # MetadataFilters: a container that AND-combines multiple MetadataFilter objects.
    from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

    # Build the list of active filters from whichever labels were supplied.
    active_filters = []
    if wing:
        active_filters.append(MetadataFilter(key="wing", value=wing))
    if room:
        active_filters.append(MetadataFilter(key="room", value=room))
    if hall:
        active_filters.append(MetadataFilter(key="hall", value=hall))

    # Base retriever arguments — always include the result count limit.
    retriever_kwargs = {"similarity_top_k": top_k}

    # Only pass filters if at least one label was specified.
    # Passing an empty MetadataFilters object would cause an error in some versions.
    if active_filters:
        retriever_kwargs["filters"] = MetadataFilters(filters=active_filters)

    # Build the retriever from the global palace_index using the assembled kwargs.
    retriever = palace_index.as_retriever(**retriever_kwargs)

    # If no query was provided, synthesise one from the supplied labels.
    # This gives the embedding model something meaningful to compare against
    # even when the caller only wants to explore by label.
    search_query = query.strip() or (
        f"{wing or ''} {room or ''} {hall or ''}".strip() or "general"
    )

    # Run the retrieval — returns NodeWithScore objects.
    nodes = retriever.retrieve(search_query)

    results = []
    for i, item in enumerate(nodes, start=1):
        # item may be a NodeWithScore wrapper; extract the underlying node.
        node = getattr(item, "node", item)
        meta = getattr(node, "metadata", {}) or {}

        # Pull the plain text content from the node.
        try:
            text = node.get_content().strip()
        except Exception:
            text = ""

        # Build a clean result dict with all useful fields.
        result = {
            "file_name":   meta.get("file_name",   "Unknown"),
            "wing":        meta.get("wing",         "Projects"),
            "room":        meta.get("room",         "General_Topic"),
            "hall":        meta.get("hall",         "Research_Evidence"),
            "drawer_type": meta.get("drawer_type",  "PDF_Text"),
            "text":        text,
        }
        results.append(result)

        if show_text:
            print(f"\n── Result {i} ──────────────────────────────────────")
            print(f"  File:   {result['file_name']}")
            print(f"  Labels: wing={result['wing']} | room={result['room']} | hall={result['hall']} | drawer={result['drawer_type']}")
            # Truncate long texts to 400 chars in the preview; add ellipsis if clipped.
            print(f"  Text:   {text[:400]}{'...' if len(text) > 400 else ''}")

    if show_text and not results:
        print("⚠️ No chunks found for the given label combination.")

    return results  # Always return the full list for programmatic use


# ── 3. Usage examples ─────────────────────────────────────────────────────────
# These are NOT executed automatically — they demonstrate the API.
# Edit the label values to match the wing/room/hall names shown in the label index above.
print("\n" + "=" * 55)
print("retrieve_by_label() is ready. Example usage:")
print("=" * 55)
print("""
# Retrieve all chunks from a specific room:
retrieve_by_label(room="Adaptive_Leadership", top_k=5)

# Retrieve with a semantic query inside a wing:
retrieve_by_label(wing="Projects", query="What is adaptive leadership?", top_k=3)

# Retrieve by full path (most specific — narrows to one hall):
retrieve_by_label(wing="Projects", room="Adaptive_Leadership", hall="Theory", top_k=5)

# Retrieve without printing (for programmatic use in other cells):
chunks = retrieve_by_label(room="Case_Studies", show_text=False)
""")


📚 LABEL INDEX — 1 unique wing/room/hall combination(s) in palace:

  Wing: Academics  |  Room: Adaptive Leadership  |  Hall: Adaptive Leadership
    └─ source_material.pdf
    └─ source_material_part1.pdf
    └─ source_material_part10.pdf
    └─ source_material_part11.pdf
    └─ source_material_part12.pdf
    └─ source_material_part13.pdf
    └─ source_material_part14.pdf
    └─ source_material_part2.pdf
    └─ source_material_part3.pdf
    └─ source_material_part4.pdf
    └─ source_material_part5.pdf
    └─ source_material_part6.pdf
    └─ source_material_part7.pdf
    └─ source_material_part8.pdf
    └─ source_material_part9.pdf

retrieve_by_label() is ready. Example usage:

# Retrieve all chunks from a specific room:
retrieve_by_label(room="Adaptive_Leadership", top_k=5)

# Retrieve with a semantic query inside a wing:
retrieve_by_label(wing="Projects", query="What is adaptive leadership?", top_k=3)

# Retrieve by full path (most specific — narrows to one hall):
retrieve_by_label(w

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Step 4.7: Load awesome-notebookLM Style Library
# awesome-notebookLM is a local repository of visual/layout style prompts.
# This cell reads its README.md, extracts the fenced code blocks from each
# section heading, and curates a set of named styles that the slide brief
# generator (Cell 11) uses to format exported JSON briefs.
#
# Styles map names like "editorial", "neon-tech", or "artifact" to multi-line
# prompt strings that describe a visual aesthetic for a presentation tool.
#
# If the repository is not installed, this cell raises FileNotFoundError.
# Set NOTEBOOKLM_REPO_PATH env var to point at the checkout location.
# ─────────────────────────────────────────────────────────────────────────────

import os       # Environment variable for repo path override
import re       # Regular expressions for parsing the Markdown README
from pathlib import Path  # Object-oriented path handling — cleaner than os.path for complex paths


# ── Repository location ───────────────────────────────────────────────────────
# Default path is C:\Users\SMANYEL\awesome-notebookLM.  Override via env var
# if the repo is checked out to a different location.
NOTEBOOKLM_REPO_PATH = Path(os.getenv("NOTEBOOKLM_REPO_PATH", r"C:\Users\SMANYEL\awesome-notebookLM"))

# Path to the README file that contains all the style definitions.
NOTEBOOKLM_README_PATH = NOTEBOOKLM_REPO_PATH / "README.md"


# ── _normalize_heading: clean a Markdown heading for comparison ───────────────
def _normalize_heading(text: str) -> str:
    """Strip Markdown formatting from a heading string for case-insensitive lookup.

    Removes surrounding whitespace, bold markers (*), colons, and lowercases
    the result so headings like '**Modern Newspaper:**' become 'modern newspaper'.
    """
    return text.strip().strip("*").strip(":").lower()


# ── _extract_style_blocks: parse README and extract fenced code prompts ───────
def _extract_style_blocks(markdown_text: str) -> dict:
    """Extract style sections and their first fenced prompt blocks from the README.

    The README is structured as ## Heading sections, each containing one or more
    fenced code blocks (``` ... ```) that hold the visual style prompt text.
    We take only the FIRST fenced block per section as the canonical style.

    Args:
        markdown_text: Full text of the README.md file.
    Returns:
        Dict mapping normalised heading string → first code block content string.
    """
    # Match every level-2 Markdown heading (##) and capture everything until the next ## or end-of-file.
    # re.MULTILINE makes ^ match at the start of every line.
    # re.DOTALL makes . match newlines so the body can span multiple lines.
    section_pattern = re.compile(r"^##\s+(.+?)\n(.*?)(?=^##\s+|\Z)", re.MULTILINE | re.DOTALL)

    # Match fenced code blocks: anything between ``` markers, non-greedy.
    fence_pattern = re.compile(r"```\n(.*?)```", re.DOTALL)

    style_blocks = {}
    for heading, body in section_pattern.findall(markdown_text):
        heading_clean = _normalize_heading(heading)  # Normalise for consistent dict keys
        code_blocks = fence_pattern.findall(body)    # Find all fenced blocks in this section
        if code_blocks:
            # Use the first code block — subsequent blocks are usually variant examples.
            style_blocks[heading_clean] = code_blocks[0].strip()
    return style_blocks


# ── _pick_style: match a curated alias to a parsed heading ────────────────────
def _pick_style(extracted: dict, *keywords: str) -> str:
    """Return the style block whose heading contains ALL of the given keywords.

    Used to create human-friendly aliases ("editorial", "artifact") that map to
    the actual (often long) heading names from the awesome-notebookLM README.
    Returns an empty string if no heading matches all keywords.
    """
    for key, val in extracted.items():
        # all() checks that every keyword appears somewhere in the normalised heading.
        if all(kw in key for kw in keywords):
            return val
    return ""  # No match — caller should handle the empty string gracefully


# ── load_notebooklm_style_library: main entry point ──────────────────────────
def load_notebooklm_style_library(readme_path: Path = NOTEBOOKLM_README_PATH) -> dict:
    """Parse the awesome-notebookLM README and return a curated style library dict.

    Returns a dict with three keys:
      raw_sections: all sections found in the README (for debugging or browsing).
      curated:      a filtered dict of the six named styles used by the ATE pipeline.
      readme_path:  the absolute path that was read (for traceability in logs).

    Raises FileNotFoundError if the README doesn't exist at readme_path.
    """
    if not readme_path.exists():
        raise FileNotFoundError(f"awesome-notebookLM README not found at: {readme_path}")

    # Read the entire README as plain text.
    # errors="ignore" silently skips any bytes that aren't valid UTF-8.
    markdown_text = readme_path.read_text(encoding="utf-8", errors="ignore")

    # Parse all ## sections and their code blocks.
    extracted = _extract_style_blocks(markdown_text)

    # Map six human-friendly style names to parsed sections by keyword matching.
    # Each _pick_style() call finds the README section whose normalised heading
    # contains all of the listed keyword fragments.
    curated = {
        "editorial":   _pick_style(extracted, "modern", "newspaper"),  # Broadsheet / newspaper layout
        "minimal":     _pick_style(extracted, "sharp-edged", "minimalism"),  # Clean, stripped-back design
        "magazine":    _pick_style(extracted, "magazine"),              # Magazine-style visual hierarchy
        "neon-tech":   _pick_style(extracted, "tech", "neon"),          # Neon-accented tech aesthetic
        "digital-pop": _pick_style(extracted, "digital", "pop"),        # Bold, high-contrast pop art style
        "artifact":    _pick_style(extracted, "anti-gravity", "artifact"),  # Avant-garde artifact layout
    }

    return {
        "raw_sections": extracted,                             # All parsed sections (unfiltered)
        "curated": {k: v for k, v in curated.items() if v},   # Only include styles that were matched
        "readme_path": str(readme_path),                       # Absolute path for log traceability
    }


# ── Load the library and announce which styles were found ────────────────────
style_library = load_notebooklm_style_library()
console_log(
    f"Loaded awesome-notebookLM styles from {style_library['readme_path']}. "
    f"Curated styles: {', '.join(style_library['curated'].keys())}",  # List found style names
    "OK",
)


[2026-05-22 18:42:49] OK: Loaded awesome-notebookLM styles from C:\Users\SMANYEL\awesome-notebookLM\README.md. Curated styles: editorial, minimal, magazine, neon-tech, digital-pop, artifact


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Step 4.8: Visual Answer Renderer + NotebookLM Slide Brief Export
# Three helper functions used by the Q&A loop in Cell 14:
#
#   render_answer_panel()  — renders a styled HTML card in the notebook output
#                            showing the question, answer, retrieval mode, and sources.
#
#   build_slide_brief()    — packages a Q&A result into a structured JSON dict
#                            suitable for slide-generation tools, including the
#                            awesome-notebookLM style prompt and source citations.
#
#   export_slide_brief()   — writes a brief dict to a timestamped JSON file in
#                            academic_data/visual_briefs/ for external use.
#
# These functions are called automatically in the research loop when the
# ATE_VISUAL_PANEL / ATE_AUTO_BRIEF / ATE_AUTO_BRIEF_EXPORT env toggles are ON.
# ─────────────────────────────────────────────────────────────────────────────

import json                               # Serialises the slide brief dict to a JSON file
from datetime import datetime             # Generates ISO-format timestamps for export filenames
from IPython.display import HTML, display # Renders raw HTML inside the Jupyter notebook output panel


# ── render_answer_panel: display a styled HTML card ──────────────────────────
def render_answer_panel(question: str, result: dict, max_sources: int = 8):
    """Render a clean, dashboard-like answer panel in the notebook output.

    Args:
        question:    The user's original question string.
        result:      The dict returned by run_academic_query(), containing
                     'answer', 'retrieval_mode', and 'sources'.
        max_sources: Maximum number of source entries to show in the panel.
                     Higher values can make the panel very long.
    """
    # Safely extract each field from the result dict using .get() with defaults.
    answer         = (result or {}).get("answer", "")
    retrieval_mode = (result or {}).get("retrieval_mode", "unknown")
    # Slice sources to max_sources before building the HTML to avoid enormous panels.
    sources        = (result or {}).get("sources", [])[:max_sources]

    # Build one <li> HTML fragment per source.
    source_items = []
    for src in sources:
        source_items.append(
            # .format() is used instead of an f-string because the HTML contains
            # curly braces that would confuse Python's f-string parser.
            "<li><b>{file}</b> | wing={wing} | room={room} | hall={hall} | drawer={drawer}<br>"
            "<span style='color:#8ab0c8'>{snippet}</span></li>".format(
                file=src.get("file_name", "Unknown"),
                wing=src.get("wing",      "Projects"),
                room=src.get("room",      "General_Topic"),
                hall=src.get("hall",      "Research_Evidence"),
                drawer=src.get("drawer_type", "PDF_Text"),
                # Fallback text if the source has no snippet stored.
                snippet=(src.get("snippet", "") or "No snippet available"),
            )
        )

    # Join all source <li> items into a single string, or show a placeholder if empty.
    sources_html = "".join(source_items) if source_items else "<li>No sources returned.</li>"

    # Build the full HTML panel as an f-string.
    # The outer <div> uses inline CSS so the panel renders correctly without
    # an external stylesheet — important for VS Code's notebook renderer.
    panel = f"""
    <div style='
        font-family: Segoe UI, Consolas, Arial, sans-serif;
        border-top: 3px solid #CFB53B;
        border-bottom: 3px solid #CFB53B;
        border-left: 3px solid #1A6FBF;
        border-right: 3px solid #1A6FBF;
        border-radius: 12px;
        padding: 20px;
        background: #0a0a0a;
        color: #f0f0f0;
        box-shadow: 0 0 18px rgba(26,111,191,0.35), 0 0 8px rgba(207,181,59,0.2);
    '>
      <div style='font-size:11px; letter-spacing:2px; color:#CFB53B; margin-bottom:4px; text-transform:uppercase;'>&#9670; Academic Truth Engine v3 &#9670;</div>
      <div style='font-size:19px; font-weight:700; color:#ffffff; border-bottom:1px solid #1A6FBF; padding-bottom:8px; margin-bottom:12px;'>Evidence-first Answer</div>
      <div style='font-size:12px; color:#a8c4e0; margin-bottom:6px;'><span style="color:#CFB53B; font-weight:600;">&#9658; Question:</span> {question}</div>
      <div style='font-size:11px; color:#6a8fa8; margin-bottom:12px;'><span style="color:#CFB53B;">Mode:</span> {retrieval_mode}</div>
      <div style='
          padding: 14px;
          border-radius: 8px;
          background: #111111;
          border-left: 3px solid #CFB53B;
          border-right: 1px solid #1A6FBF;
          white-space: pre-wrap;
          line-height: 1.6;
          color: #e8e8e8;
          font-size: 13px;
      '>{answer}</div>
      <div style='margin-top:14px; font-size:13px; font-weight:700; color:#CFB53B; letter-spacing:1px;'>&#128218; Sources</div>
      <ol style='margin-top:6px; padding-left:18px; line-height:1.6; color:#9ab8cc; font-size:12px;'>{sources_html}</ol>
    </div>
    """
    # display(HTML(...)) renders the string as real HTML inside the cell output.
    # This is the correct Jupyter API — printing the HTML string would show raw tags.
    display(HTML(panel))


# ── build_slide_brief: package an answer into a structured JSON brief ─────────
def build_slide_brief(question: str, result: dict, style_key: str = "artifact") -> dict:
    """Create a structured brief using local awesome-notebookLM style templates.

    The brief is a plain Python dict (JSON-serialisable) containing:
      - The question and full answer text.
      - The retrieval mode label.
      - A style prompt string from the style library (for use by slide tools).
      - A list of source summaries with file/hierarchy metadata and text snippets.
      - Strict usage instructions to prevent unsupported synthesis downstream.

    Args:
        question:  The user's question.
        result:    The run_academic_query() result dict.
        style_key: Key into style_library['curated']. Defaults to 'artifact'.
                   Falls back gracefully if the key is not in the library.
    Returns:
        A dict that can be passed directly to export_slide_brief() or used inline.
    """
    # Look up the style prompt; fall back to empty string if the key doesn't exist.
    curated      = (style_library or {}).get("curated", {})
    style_prompt = curated.get(style_key, "")  # Empty string = no style guidance

    sources = (result or {}).get("sources", [])

    # Build a compact source summary list (capped at 10 to keep the JSON manageable).
    source_summary = []
    for src in sources[:10]:
        source_summary.append({
            "file":    src.get("file_name",   "Unknown"),
            "wing":    src.get("wing",        "Projects"),
            "room":    src.get("room",        "General_Topic"),
            "hall":    src.get("hall",        "Research_Evidence"),
            "drawer":  src.get("drawer_type", "PDF_Text"),
            "snippet": src.get("snippet",     ""),  # The 160-char text preview from the source node
        })

    brief = {
        # ISO 8601 timestamp (seconds precision) for sorting and deduplication.
        "timestamp":      datetime.now().isoformat(timespec="seconds"),
        "question":       question,
        "answer":         (result or {}).get("answer", ""),
        "retrieval_mode": (result or {}).get("retrieval_mode", "unknown"),
        "style_key":      style_key,      # Which style was requested
        "style_prompt":   style_prompt,   # The full style prompt text from awesome-notebookLM
        "source_summary": source_summary,
        # Downstream instructions: tell any slide-generation tool how to use this brief.
        "instructions": [
            "Use source_summary as the only evidence base.",        # Evidence-only mandate
            "One slide should carry one claim.",                    # Atomic slide structure
            "Preserve strict citation traceability to source snippets.",  # No unsourced claims
            "Avoid unsupported synthesis beyond provided evidence."  # No hallucination
        ]
    }
    return brief


# ── export_slide_brief: write the brief dict to a timestamped JSON file ───────
def export_slide_brief(brief: dict, output_dir: str = "academic_data/visual_briefs") -> str:
    """Persist a slide brief dict to disk as a JSON file.

    Args:
        brief:      The dict produced by build_slide_brief().
        output_dir: Directory to write into.  Created automatically if absent.
    Returns:
        The absolute path of the written JSON file.
    """
    # Create the output directory tree if any part of it is missing.
    os.makedirs(output_dir, exist_ok=True)

    # Build a timestamp-based filename so successive exports don't overwrite each other.
    safe_ts  = datetime.now().strftime("%Y%m%d_%H%M%S")  # e.g. 20260429_153042
    out_path = os.path.join(output_dir, f"slide_brief_{safe_ts}.json")

    # Write the brief as pretty-printed JSON.
    # ensure_ascii=False preserves non-Latin characters (e.g. accented academic terms).
    # indent=2 makes the file human-readable.
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(brief, f, ensure_ascii=False, indent=2)

    console_log(f"Slide brief exported: {out_path}", "OK")
    return out_path  # Return the path so the caller can display or link it


# ── Confirm helpers are ready ─────────────────────────────────────────────────
print("✅ Visual helpers ready:")
print("- render_answer_panel(question, result)")           # Call after run_academic_query()
print("- build_slide_brief(question, result, style_key='artifact')")  # Package result for slides
print("- export_slide_brief(brief)")                      # Save the packaged brief to disk

✅ Visual helpers ready:
- render_answer_panel(question, result)
- build_slide_brief(question, result, style_key='artifact')
- export_slide_brief(brief)


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Step 5: Advanced Query Fusion (Sequential Stability Mode)
# Loads the local QueryRewritingRetrieverPack from the query_rewriting_pack/
# directory in this workspace.  The pack rewrites the user's question into
# multiple paraphrased variants, retrieves documents for each variant, and
# fuses the ranked results — improving recall over a single-pass retrieval.
#
# Sequential mode (num_queries=1) is used by default for stability: running
# multiple parallel async retrievals inside a Jupyter kernel can trigger
# event loop conflicts.  Increase num_queries once concurrency is confirmed safe.
# ─────────────────────────────────────────────────────────────────────────────

import os               # Used for PATH manipulation
import sys              # sys.path: controls where Python looks for importable modules
from pathlib import Path  # Object-oriented path resolution
import importlib.util   # importlib.util.spec_from_file_location: loads a module directly from a file path


# ── 1. Add the local pack root to the Python module search path ───────────────
# The query_rewriting_pack/ folder contains the local fork of the LlamaIndex
# fusion retriever pack.  We need it on sys.path so the import below resolves
# to the local version instead of the pip-installed upstream package.
local_pack_root = Path("query_rewriting_pack").resolve()  # Absolute path for safety

# Guard: only insert once even if this cell is re-run.
if str(local_pack_root) not in sys.path:
    sys.path.insert(0, str(local_pack_root))  # insert at position 0 = highest priority


# ── 2. Import with fallback to direct file loading ────────────────────────────
# Try the standard namespace import first.  On a clean environment where
# sys.path was correctly set, this will work immediately.
try:
    from llama_index.packs.fusion_retriever.query_rewrite.base import QueryRewritingRetrieverPack

except Exception:
    # The namespace import failed (e.g. the package's __init__.py chain is broken
    # or there is a naming conflict with the pip-installed upstream version).
    # Fall back to loading the module directly from its file path — this bypasses
    # all namespace resolution and always loads the local version.
    base_file = (
        local_pack_root
        / "llama_index"
        / "packs"
        / "fusion_retriever"
        / "query_rewrite"
        / "base.py"    # The file that contains the QueryRewritingRetrieverPack class
    )
    # spec_from_file_location creates a module spec from an absolute file path.
    spec = importlib.util.spec_from_file_location("local_query_rewrite_base", str(base_file))
    # module_from_spec creates the actual module object from the spec.
    module = importlib.util.module_from_spec(spec)
    # exec_module executes the module's code, populating its namespace.
    spec.loader.exec_module(module)
    # Extract the class from the loaded module.
    QueryRewritingRetrieverPack = module.QueryRewritingRetrieverPack


# ── 3. Instantiate the pack on the current document's nodes ──────────────────
# The pack builds its own internal VectorStoreIndex from the provided nodes.
# chunk_size=256: maximum token width of each chunk the pack considers.
#   Smaller chunks give more precise retrieval but may lose context.
# vector_similarity_top_k=5: retrieve the 5 nearest neighbours per query variant.
# num_queries=1: generate only 1 query variant (the original question) per call.
#   This is "sequential stability mode" — no parallel async work, no event loop issues.
#   To enable full fusion: set num_queries=3 or higher and ensure nest_asyncio is active.
query_rewriting_pack = QueryRewritingRetrieverPack(
    nodes,                          # The semantic nodes produced in Cell 6 (current document only)
    chunk_size=256,                 # Token chunk width for the pack's internal index
    vector_similarity_top_k=5,     # Top-K candidates per query variant
    num_queries=1,                  # Sequential mode: 1 variant = 0 concurrency
)

print("🚀 Search Engine set to SINGLE-STREAM mode.")
print("✅ Stability confirmed: Sequential processing enabled.")
console_log("Advanced Query Fusion Engine Ready (Sequential Fix)", "OK")
console_log("Hybrid mode ready: short-term fusion + long-term MemPalace retrieval.", "OK")


resource module not available on Windows
🚀 Search Engine set to SINGLE-STREAM mode.
✅ Stability confirmed: Sequential processing enabled.
[2026-05-22 18:48:52] OK: Advanced Query Fusion Engine Ready (Sequential Fix)
[2026-05-22 18:48:52] OK: Hybrid mode ready: short-term fusion + long-term MemPalace retrieval.


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Step 5.5: Runtime Visual Preset Switcher
# Run this cell immediately before Cell 14 (the research loop) to configure
# which visual and export features are active during the Q&A session.
#
# Change ATE_PRESET to one of three named presets:
#   'visual_only'          — render the HTML answer panel, no brief generation
#   'visual_brief'         — render panel + generate in-memory slide brief
#   'visual_brief_export'  — render panel + generate brief + save brief to disk
#
# The preset writes four environment variables that Cell 14 reads at startup:
#   ATE_VISUAL_PANEL       — "1" = render HTML panel after each answer
#   ATE_AUTO_BRIEF         — "1" = auto-generate a slide brief per answer
#   ATE_AUTO_BRIEF_EXPORT  — "1" = auto-save the brief to academic_data/visual_briefs/
#   ATE_BRIEF_STYLE        — which awesome-notebookLM style to use for the brief
# ─────────────────────────────────────────────────────────────────────────────

import os  # Environment variable writes — the only dependency needed here


# ── Choose a preset by changing this one variable ────────────────────────────
# 'visual_only'         — fastest; only the HTML panel renders, no JSON output
# 'visual_brief'        — panel + brief in memory; brief not saved unless you call export manually
# 'visual_brief_export' — panel + brief + automatic save after each query
ATE_PRESET = "visual_brief"


# ── Preset definitions ────────────────────────────────────────────────────────
# Each preset maps to a dict of environment variable name → value ("1" or "0").
# All four env vars are explicitly set so Cell 14 always reads a defined value,
# even if a previous session left stale values in the environment.
PRESETS = {
    "visual_only": {
        "ATE_VISUAL_PANEL":       "1",  # HTML panel ON
        "ATE_AUTO_BRIEF":         "0",  # Brief generation OFF
        "ATE_AUTO_BRIEF_EXPORT":  "0",  # Disk export OFF
    },
    "visual_brief": {
        "ATE_VISUAL_PANEL":       "1",  # HTML panel ON
        "ATE_AUTO_BRIEF":         "1",  # Brief generation ON (in-memory)
        "ATE_AUTO_BRIEF_EXPORT":  "0",  # Disk export OFF
    },
    "visual_brief_export": {
        "ATE_VISUAL_PANEL":       "1",  # HTML panel ON
        "ATE_AUTO_BRIEF":         "1",  # Brief generation ON
        "ATE_AUTO_BRIEF_EXPORT":  "1",  # Disk export ON (saves JSON per query)
    },
}

# Guard: fail fast with a clear message if ATE_PRESET was set to an invalid value.
if ATE_PRESET not in PRESETS:
    raise ValueError(f"Unknown ATE_PRESET='{ATE_PRESET}'. Choose one of: {', '.join(PRESETS)}")

# Write every env var in the chosen preset.
# os.environ writes persist for the lifetime of the kernel process, so Cell 14
# reads these values without needing them passed as arguments.
for key, value in PRESETS[ATE_PRESET].items():
    os.environ[key] = value

# ATE_BRIEF_STYLE controls which awesome-notebookLM style is applied to auto-generated briefs.
# setdefault only writes the value if the variable is NOT already set, so an explicit
# override from a previous cell or a .env file is always preserved.
os.environ.setdefault("ATE_BRIEF_STYLE", "artifact")  # Default: avant-garde artifact layout

print("✅ ATE runtime preset applied:", ATE_PRESET)
# Print a summary of all four toggle states so the user can confirm before running Cell 14.
print(
    f"ATE_VISUAL_PANEL={os.getenv('ATE_VISUAL_PANEL')} | "
    f"ATE_AUTO_BRIEF={os.getenv('ATE_AUTO_BRIEF')} | "
    f"ATE_AUTO_BRIEF_EXPORT={os.getenv('ATE_AUTO_BRIEF_EXPORT')} | "
    f"ATE_BRIEF_STYLE={os.getenv('ATE_BRIEF_STYLE')}"
)


✅ ATE runtime preset applied: visual_brief
ATE_VISUAL_PANEL=1 | ATE_AUTO_BRIEF=1 | ATE_AUTO_BRIEF_EXPORT=0 | ATE_BRIEF_STYLE=artifact


In [19]:

# ── Bridge: when batch ingest was used, current_doc_index is not set ──────────
# Cell 16 expects current_doc_index for "local doc" retrieval.  After a batch
# ingest all PDFs are already in the palace, so we point both retrievers at the
# same palace_index — every query gets full cross-document coverage.
if "current_doc_index" not in globals() or current_doc_index is None:
    current_doc_index = palace_index
    print("ℹ️  current_doc_index → palace_index (batch-ingest mode: all PDFs in palace)")
else:
    print(f"ℹ️  current_doc_index already set — single-doc mode active")


ℹ️  current_doc_index already set — single-doc mode active


In [20]:
# Step 6: The Research Loop (Q&A) - Hierarchical Hybrid Memory + Semantic Citations
import os
import time
import json
import pathlib
from datetime import datetime  # restore class-level name in kernel scope
import replicate as _replicate_sdk

# ── Monkey-patch replicate.run + replicate.stream (bypass broken Rust SDK v1.x) ──
def _http_run_replicate(model_ref, input=None, **kwargs):
    """HTTP drop-in for replicate.run() — bypasses broken SDK."""
    import requests as _req2, time as _t2, os as _os2
    _tok = _os2.environ.get("REPLICATE_API_TOKEN", "") or globals().get("replicate_token", "")
    _hdrs = {
        "Authorization": f"Bearer {_tok}",
        "Content-Type": "application/json",
        "Prefer": "wait=60",
    }
    _inp = dict(input or {})
    _inp.update(kwargs.get("input", {}))
    _url = f"https://api.replicate.com/v1/models/{model_ref}/predictions"
    _resp = _req2.post(_url, headers=_hdrs, json={"input": _inp}, timeout=90)
    _resp.raise_for_status()
    _pred = _resp.json()
    if _pred.get("status") == "succeeded":
        _out = _pred.get("output", "")
        return _out if isinstance(_out, list) else ([str(_out)] if _out else [])
    _poll_url = (_pred.get("urls") or {}).get("get") or _pred.get("url")
    if not _poll_url:
        raise RuntimeError(f"No poll URL in response: {_pred}")
    _dl = _t2.time() + 300.0
    while _t2.time() < _dl:
        _t2.sleep(3.0)
        _pr = _req2.get(_poll_url, headers=_hdrs, timeout=30)
        _pr.raise_for_status()
        _pd = _pr.json()
        _st = _pd.get("status", "")
        if _st == "succeeded":
            _out = _pd.get("output", "")
            return _out if isinstance(_out, list) else ([str(_out)] if _out else [])
        if _st in ("failed", "canceled"):
            raise RuntimeError(f"Prediction {_st}: {_pd.get('error')}")
    raise TimeoutError("Replicate prediction timed out after 300s")

def _http_stream_replicate(model_ref, *, input=None, use_file_output=True, **params):
    """HTTP drop-in for replicate.stream() — used by LlamaIndex Replicate LLM."""
    _chunks = _http_run_replicate(model_ref, input=input)
    if isinstance(_chunks, list):
        for _chunk in _chunks:
            yield str(_chunk)
    elif _chunks:
        yield str(_chunks)

_replicate_sdk.run = _http_run_replicate
_replicate_sdk.stream = _http_stream_replicate


import datetime as _dt_mod
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

WING_KEYWORD_MAP = {
    "Projects": ["project", "assignment", "capstone", "research", "analysis"],
    "People": ["stakeholder", "leader", "manager", "person", "team"],
}
ROOM_KEYWORD_MAP = {
    "Adaptive_Leadership": ["adaptive", "leadership", "capacity"],
    "Case_Studies": ["case study", "case", "scenario", "example"],
    "System_Stats": ["stat", "statistics", "variance", "mean", "regression", "dataset"],
    "Systems_Thinking": ["system", "systems thinking", "feedback loop", "causal"],
}
HALL_KEYWORD_MAP = {
    "Research_Evidence": ["evidence", "citation", "source", "proof"],
    "Theory": ["theory", "framework", "model", "concept"],
    "Application": ["apply", "application", "implementation", "practice"],
}
RUNTIME_MODE = os.getenv("ATE_RUNTIME_MODE", "deep").strip().lower()
if RUNTIME_MODE not in {"fast", "deep"}:
    RUNTIME_MODE = "deep"
PALACE_TOP_K = 8 if RUNTIME_MODE == "fast" else 12
MAX_FILTER_ATTEMPTS = 3 if RUNTIME_MODE == "fast" else 7
ENABLE_LLM_SYNTHESIS = os.getenv("ATE_ENABLE_SYNTHESIS", "1").strip() in {"1", "true", "True"}
ENABLE_NETWORK_LLM = os.getenv("ATE_ENABLE_NETWORK_LLM", "1").strip() in {"1", "true", "True"}
USE_FUSION_PACK = os.getenv("ATE_USE_FUSION_PACK", "0").strip() in {"1", "true", "True"}
SOURCE_PRINT_LIMIT = int(os.getenv("ATE_SOURCE_PRINT_LIMIT", "10"))
VISUAL_PANEL_ON = os.getenv("ATE_VISUAL_PANEL", "1").strip() not in {"0", "false", "False"}
AUTO_BRIEF_ON = os.getenv("ATE_AUTO_BRIEF", "0").strip() in {"1", "true", "True"}
AUTO_BRIEF_EXPORT_ON = os.getenv("ATE_AUTO_BRIEF_EXPORT", "0").strip() in {"1", "true", "True"}
BRIEF_STYLE_KEY = os.getenv("ATE_BRIEF_STYLE", "artifact").strip() or "artifact"
# Max tokens for Granite LLM synthesis.
SYNTHESIS_MAX_TOKENS = int(os.getenv("ATE_SYNTHESIS_MAX_TOKENS", "800" if RUNTIME_MODE == "fast" else "1200"))

# ── Session persistence ────────────────────────────────────────────────────────
_SESSION_DIR = pathlib.Path("./sessions")
_SESSION_DIR.mkdir(parents=True, exist_ok=True)
_session_data: dict = {}
_session_path: pathlib.Path | None = None
_palace_retriever_cache: dict = {}

# ── Logging helper ─────────────────────────────────────────────────────────────
def console_log(msg, level='INFO'):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{ts}] {level}: {msg}")

# ── Node helper utilities ──────────────────────────────────────────────────────
def _node_obj(item):
    return getattr(item, "node", item)

def _extract_text_from_node(item) -> str:
    node = _node_obj(item)
    try:
        return node.get_content().strip()
    except Exception:
        return ""

# ── Network / timeout error detection ─────────────────────────────────────────
def _is_network_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(k in msg for k in ("getaddrinfo", "name or service not known",
                                   "nodename nor servname", "network unreachable",
                                   "connection refused", "errno 11004",
                                   "errno 11001", "temporary failure in name resolution"))

def _is_timeout_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(m in msg for m in ("read operation timed out", "timed out", "timeout", "readtimeout"))

# ── Replicate API wrapper ──────────────────────────────────────────────────────
def _replicate_create_and_poll(
    prompt: str,
    max_tokens: int = 800,
    poll_interval: float = 3.0,
    timeout: float = 300.0,
) -> str:
    '''Call Replicate HTTP API directly (bypasses broken SDK).'''
    import requests as _req, time as _t, os as _os
    _os.environ["REPLICATE_API_TOKEN"] = replicate_token
    _headers = {
        "Authorization": f"Bearer {replicate_token}",
        "Content-Type": "application/json",
        "Prefer": "wait=60",
    }
    _payload = {
        "input": {
            "prompt": prompt,
            "max_new_tokens": max_tokens,
            "temperature": 0.4,
            "top_p": 0.9,
        },
    }
    _url = "https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions"
    _r = _req.post(_url, headers=_headers, json=_payload, timeout=90)
    _r.raise_for_status()
    _pred = _r.json()
    # Prefer:wait may have resolved it immediately
    _status = _pred.get("status", "")
    if _status == "succeeded":
        _out = _pred.get("output", "")
        if isinstance(_out, list): return "".join(str(x) for x in _out).strip()
        return str(_out).strip() if _out else ""
    # Poll until done
    _poll_url = (_pred.get("urls") or {}).get("get") or _pred.get("url")
    if not _poll_url:
        raise RuntimeError(f"No poll URL in prediction response: {_pred}")
    _deadline = _t.time() + timeout
    while _t.time() < _deadline:
        _t.sleep(poll_interval)
        _pr = _req.get(_poll_url, headers=_headers, timeout=30)
        _pr.raise_for_status()
        _pd = _pr.json()
        _st = _pd.get("status", "")
        if _st == "succeeded":
            _out = _pd.get("output", "")
            if isinstance(_out, list): return "".join(str(x) for x in _out).strip()
            return str(_out).strip() if _out else ""
        if _st in ("failed", "canceled"):
            raise RuntimeError(f"Prediction {_st}: {_pd.get('error')}")
    raise TimeoutError(f"Replicate prediction timed out after {timeout}s")

# ── Prompt mode detection ──────────────────────────────────────────────────────
def _detect_prompt_mode(question: str) -> tuple[str, str]:
    q = question.strip()
    ql = q.lower()

    for prefix in (
        "summarise this content", "summarize this content",
        "summarise this text",    "summarize this text",
        "summarise this",         "summarize this",
        "summarise:",             "summarize:",
    ):
        if ql.startswith(prefix):
            return "summarise", q[len(prefix):].strip().lstrip(":").strip()

    for prefix in (
        "paraphrase this content", "paraphrase this text",
        "paraphrase this",         "paraphrase:",
        "rewrite this content",    "rewrite this text", "rewrite this",
    ):
        if ql.startswith(prefix):
            return "paraphrase", q[len(prefix):].strip().lstrip(":").strip()

    for prefix in (
        "given this text",           "given this content",
        "from this text",            "from this content",
        "formulate from",            "formulate a",
        "generate a hypothesis",     "generate a research question",
        "write an introduction for",
    ):
        if ql.startswith(prefix):
            return "formulate", q[len(prefix):].strip().lstrip(":,").strip()

    for starter in (
        "tell me everything about", "tell me all about", "tell me about",
        "what is ", "what are ", "explain ", "describe ", "define ",
        "give me information on", "give me an overview of",
        "overview of", "list everything about",
    ):
        if ql.startswith(starter):
            return "retrieval", ""

    _chat_starters = (
        "do you ", "can you ", "will you ", "did you ", "are you ",
        "have you ", "what do you ", "how do you ", "does the engine",
        "does it ", "is it ", "is there ", "why do you ", "when do you ",
        "could you ", "would you ", "should i ", "should the ",
    )
    if any(ql.startswith(s) for s in _chat_starters) or (len(q) <= 80 and q.endswith("?")):
        return "chat", ""

    return "essay", ""

# ── Meta question patterns ─────────────────────────────────────────────────────
_META_PATTERNS = [
    (["keep the last", "remember the last", "retain the last", "store the last", "keep my last", "remember my last", "do you keep"],
     "No -- the engine is stateless between queries. Each question is processed independently: your prompt goes through retrieval, synthesis, then an answer is returned. The result is then discarded. Previous answers, prompts, and conversations are not stored or referenced in subsequent queries."),
    (["conversation history", "chat history", "previous answer", "previous prompt", "previous question", "earlier answer", "last answer", "last prompt", "last question", "session", "session history", "what have i asked", "my questions"],
     "Yes -- session history IS stored. Every Q&A exchange is saved to a JSON file in the sessions/ folder. On restart you can continue the previous session or start a new one. Type 'show history' at any prompt to review the current session."),
    (["what is your name", "who are you", "what are you"],
     "Academic Truth Engine v3 -- a PDF-grounded research assistant. It retrieves evidence from a ChromaDB Memory Palace (your ingested PDFs) and uses IBM Granite 3.1 via Replicate to synthesise answers."),
    (["how does the engine work", "how do you work", "how do you answer"],
     "The engine works in three stages: (1) Retrieval -- your question is matched against the ChromaDB Memory Palace using hierarchical metadata filters (wing/room/hall) and embedding similarity. (2) Synthesis -- retrieved chunks are assembled into a prompt and sent to IBM Granite 3.1 via Replicate. (3) Output -- the answer is returned and displayed with source citations."),
    (["what modes", "what prompt modes", "what can you do", "what types of questions"],
     "The engine supports five prompt modes detected automatically from how you phrase your question: (1) essay -- full 12-paragraph academic essay (default); (2) chat -- short direct factual answer; (3) summarise -- concise summary of pasted text; (4) paraphrase -- human rewrite of pasted text; (5) formulate -- generates introduction, hypothesis, and research question; (6) retrieval -- structured knowledge overview (definition, theories, applications)."),
]

def _check_meta_question(question: str) -> str | None:
    ql = question.lower()
    for patterns, answer in _META_PATTERNS:
        if any(p in ql for p in patterns):
            return answer
    return None

# ── Text extraction helpers ────────────────────────────────────────────────────
def _nodes_to_text(items, max_items: int = 10, max_chars: int = 6000) -> str:
    snippets = []
    total = 0
    for item in items[:max_items]:
        text = _extract_text_from_node(item)
        if not text:
            continue
        text = text.replace("\n", " ").strip()
        if not text:
            continue
        remaining = max_chars - total
        if remaining <= 0:
            break
        clipped = text[:remaining]
        snippets.append(clipped)
        total += len(clipped)
    return "\n\n".join(snippets)

def _to_text(obj) -> str:
    if obj is None:
        return ""
    if hasattr(obj, "response") and isinstance(obj.response, str):
        return obj.response.strip()
    if hasattr(obj, "text") and isinstance(obj.text, str):
        return obj.text.strip()
    return str(obj).strip()

def _extract_sources_from_nodes(label: str, items) -> list[dict]:
    refs = []
    for item in items or []:
        node = _node_obj(item)
        meta = getattr(node, "metadata", {}) or {}
        snippet = ""
        try:
            snippet = node.get_content()[:160].replace("\n", " ").strip()
        except Exception:
            snippet = ""
        refs.append(
            {
                "retriever": label,
                "file_name": meta.get("file_name") or meta.get("source") or "Unknown",
                "wing": meta.get("wing", "Projects"),
                "room": meta.get("room", "General_Topic"),
                "hall": meta.get("hall", "Research_Evidence"),
                "drawer_type": meta.get("drawer_type", "PDF_Text"),
                "snippet": snippet,
            }
        )
    return refs

def _dedupe_sources(refs: list[dict]) -> list[dict]:
    seen = set()
    unique_refs = []
    for ref in refs:
        key = (
            ref["retriever"],
            ref["file_name"],
            ref["wing"],
            ref["room"],
            ref["hall"],
            ref["snippet"][:80],
        )
        if key in seen:
            continue
        seen.add(key)
        unique_refs.append(ref)
    return unique_refs

# ── Synthesis prompt builder ───────────────────────────────────────────────────
def _build_synthesis_prompt(
    mode: str,
    question: str,
    inline_text: str,
    current_doc_text: str,
    palace_text: str,
    short_term_text: str,
) -> str:
    evidence_parts = []
    if current_doc_text:
        evidence_parts.append(f"SOURCE — Current Document:\n{current_doc_text}")
    if palace_text:
        evidence_parts.append(f"SOURCE — Memory Palace:\n{palace_text}")
    if short_term_text:
        evidence_parts.append(f"SOURCE — Fusion Retriever:\n{short_term_text}")
    evidence = "\n\n".join(evidence_parts) or "No additional source material retrieved."
    subject = inline_text.strip() or question.strip()

    if mode == "summarise":
        return (
            "You are an academic summarisation assistant. "
            "Read the following text carefully and produce a clear, concise summary. "
            "Preserve every main point, key argument, and critical detail. "
            "Remove repetition and padding. "
            "Write in continuous flowing prose (2-4 paragraphs). "
            "Do not use bullet points or numbered lists.\n\n"
            f"TEXT TO SUMMARISE:\n{subject}\n\n"
            f"ADDITIONAL CONTEXT FROM SOURCE MATERIAL (use only if directly relevant):\n{evidence}\n"
        )

    if mode == "paraphrase":
        return (
            "You are a skilled academic writer. "
            "Rewrite the following text entirely in your own words. "
            "Preserve every main idea, argument, and key point exactly - nothing should be lost. "
            "Make the writing feel natural, human, and original. "
            "Avoid robotic or formulaic phrasing. "
            "Do not add new information. Do not omit any core ideas. "
            "Write in clear, engaging academic prose.\n\n"
            f"TEXT TO PARAPHRASE:\n{subject}\n\n"
            f"ADDITIONAL CONTEXT FROM SOURCE MATERIAL (use only if directly relevant):\n{evidence}\n"
        )

    if mode == "formulate":
        return (
            "You are an academic research design assistant. "
            "Read the following source text and produce all three items below. "
            "Base everything strictly on the provided text - do not invent information.\n\n"
            "DELIVERABLES:\n"
            "1. INTRODUCTION (1 paragraph, 4-6 sentences): A compelling academic introduction "
            "that contextualises the topic, establishes its significance, and signals what will be explored.\n"
            "2. HYPOTHESIS (1-2 sentences): A clear, testable hypothesis derived from the text.\n"
            "3. RESEARCH QUESTION (1-2 sentences): A focused research question that the text raises or implies.\n\n"
            f"SOURCE TEXT:\n{subject}\n\n"
            f"ADDITIONAL CONTEXT FROM SOURCE MATERIAL (use only if directly relevant):\n{evidence}\n"
        )

    if mode == "retrieval":
        return (
            "You are an academic knowledge assistant. "
            "Provide a comprehensive, well-organised overview of the topic below. "
            "Use the following structure:\n\n"
            "1. DEFINITION: What is this topic? (2-3 sentences)\n"
            "2. KEY CONCEPTS: The most important ideas, terms, or principles\n"
            "3. MAIN THEORIES OR FRAMEWORKS: Relevant academic models or schools of thought\n"
            "4. PRACTICAL APPLICATIONS: How this topic is applied in practice\n"
            "5. DEBATES AND LIMITATIONS: Key disagreements or gaps in the field\n\n"
            "Use ONLY the provided source material. Do not invent facts. "
            "Write in clear academic prose under each heading.\n\n"
            f"TOPIC: {question}\n\n"
            f"SOURCE MATERIAL:\n{evidence}\n"
        )

    if mode == "chat":
        return (
            "You are a helpful academic research assistant. "
            "Answer the following question directly, concisely, and factually. "
            "Use ONLY the provided source material as evidence. "
            "If the source material does not contain enough information to answer the question, "
            "say so clearly -- do not guess or invent anything. "
            "Write in plain, clear prose. Aim for 1-3 short paragraphs maximum. "
            "Do NOT write a full essay. Do NOT use headings or numbered sections. "
            "Just answer the question directly and stop.\n\n"
            f"QUESTION: {question}\n\n"
            f"SOURCE MATERIAL:\n{evidence}\n"
        )

    return (
        "You are an academic writing assistant. Your task is to write a thorough, "
        "well-structured academic essay that fully answers the question below. "
        "You MUST use ONLY the evidence provided in the source material sections. "
        "Do NOT introduce any external knowledge, assumptions, or information not "
        "present in the evidence. Every claim must be grounded in the provided text.\n\n"

        "ESSAY STRUCTURE - follow this exactly:\n\n"

        "INTRODUCTION (3 paragraphs):\n"
        "  Paragraph 1: Define all key concepts and theories mentioned in the question, "
        "drawing directly from the source material.\n"
        "  Paragraph 2: Explain the academic context and importance of the topic as "
        "evidenced in the source material.\n"
        "  Paragraph 3: State your thesis - a precise statement of what the essay will "
        "cover and how it answers every part of the question.\n\n"

        "BODY (6 paragraphs):\n"
        "  Paragraph 4: Address the first major concept or sub-question using evidence "
        "from the source material with direct quotations or paraphrases.\n"
        "  Paragraph 5: Address the second major concept or sub-question using evidence "
        "from the source material.\n"
        "  Paragraph 6: Address the third major concept or sub-question, providing "
        "theoretical grounding from the source material.\n"
        "  Paragraph 7: Apply the concepts to the practical/scenario context raised in "
        "the question, supported by source material evidence.\n"
        "  Paragraph 8: Compare, contrast, or extend the concepts using additional "
        "evidence from the source material.\n"
        "  Paragraph 9: Critically evaluate any limitations, gaps, or nuances found in "
        "the source material's treatment of the topic.\n\n"

        "CONCLUSION (3 paragraphs):\n"
        "  Paragraph 10: Summarise the key arguments made in the body and how they "
        "collectively answer the question.\n"
        "  Paragraph 11: Reflect on the broader significance of the findings as "
        "supported by the source material.\n"
        "  Paragraph 12: Final closing statement - tie all parts of the question "
        "together and restate the essay's core answer.\n\n"

        "STRICT RULES:\n"
        "- Write in formal academic prose. No bullet points or numbered lists in the essay.\n"
        "- Each paragraph must be at least 4 sentences long.\n"
        "- Use direct quotes or close paraphrases from the evidence where possible.\n"
        "- After the essay, include a SOURCES section listing the file names referenced.\n"
        "- If the evidence is insufficient to fill a paragraph, state what is missing "
        "and work with what is available.\n\n"

        f"QUESTION:\n{question}\n\n"
        f"SOURCE MATERIAL - Current Document:\n{current_doc_text or 'No current-doc evidence returned.'}\n\n"
        f"SOURCE MATERIAL - Long-Term Palace:\n{palace_text or 'No long-term palace evidence returned.'}\n\n"
        f"SOURCE MATERIAL - Fusion Retriever:\n{short_term_text or 'Fusion pack disabled or no evidence returned.'}\n"
    )

# ── Keyword inference for hierarchy routing ────────────────────────────────────
def _infer_by_keywords(question: str, mapping: dict[str, list[str]]) -> str | None:
    q = question.lower()
    best_label = None
    best_score = 0
    for label, keywords in mapping.items():
        score = sum(1 for kw in keywords if kw in q)
        if score > best_score:
            best_score = score
            best_label = label
    return best_label if best_score > 0 else None

def infer_hierarchy_targets(question: str) -> dict:
    return {
        "wing": _infer_by_keywords(question, WING_KEYWORD_MAP),
        "room": _infer_by_keywords(question, ROOM_KEYWORD_MAP),
        "hall": _infer_by_keywords(question, HALL_KEYWORD_MAP),
    }

def _filters_key(filters: list[MetadataFilter] | None) -> tuple:
    if not filters:
        return tuple()
    return tuple(sorted((f.key, str(f.value)) for f in filters))

def _get_palace_retriever(filters: list[MetadataFilter] | None = None):
    key = _filters_key(filters)
    if key in _palace_retriever_cache:
        return _palace_retriever_cache[key]
    kwargs = {"similarity_top_k": PALACE_TOP_K}
    if filters:
        kwargs["filters"] = MetadataFilters(filters=filters)
    retriever = palace_index.as_retriever(**kwargs)
    _palace_retriever_cache[key] = retriever
    return retriever

def _retrieve_with_filters(question: str, filters: list[MetadataFilter]):
    palace_retriever = _get_palace_retriever(filters)
    return palace_retriever.retrieve(question)

def _build_filter_candidates(targets: dict) -> list[list[MetadataFilter]]:
    filter_candidates = []
    if targets["wing"] and targets["room"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="room", value=targets["room"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"] and targets["room"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="room", value=targets["room"]),
        ])
    if targets["room"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="room", value=targets["room"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"] and targets["hall"]:
        filter_candidates.append([
            MetadataFilter(key="wing", value=targets["wing"]),
            MetadataFilter(key="hall", value=targets["hall"]),
        ])
    if targets["wing"]:
        filter_candidates.append([MetadataFilter(key="wing", value=targets["wing"])])
    if targets["room"]:
        filter_candidates.append([MetadataFilter(key="room", value=targets["room"])])
    if targets["hall"]:
        filter_candidates.append([MetadataFilter(key="hall", value=targets["hall"])])
    return filter_candidates

def hierarchical_palace_retrieve(question: str):
    targets = infer_hierarchy_targets(question)
    filter_candidates = _build_filter_candidates(targets)[:MAX_FILTER_ATTEMPTS]
    for filters in filter_candidates:
        nodes = _retrieve_with_filters(question, filters)
        if nodes:
            route = ", ".join(f"{f.key}={f.value}" for f in filters)
            return nodes, f"Hierarchy-routed ({route})"
    general_nodes = _get_palace_retriever().retrieve(question)
    return general_nodes, "General palace retrieval"

# ── Fast answer (offline mode) ─────────────────────────────────────────────────
def _compose_fast_answer(question: str, short_term_text: str, current_doc_text: str, palace_text: str) -> str:
    combined = "\n\n".join(filter(None, [short_term_text, current_doc_text, palace_text]))
    if not combined.strip():
        return "\u26a0\ufe0f No evidence found in uploaded source material for this question."
    return (
        "\u26a0\ufe0f OFFLINE MODE \u2014 LLM synthesis disabled. Raw evidence retrieved from your PDFs:\n\n"
        + combined
        + "\n\n[Enable ATE_ENABLE_SYNTHESIS=1 and ATE_ENABLE_NETWORK_LLM=1 for full essay output.]"
    )

# ── Main query function ────────────────────────────────────────────────────────
def run_academic_query(question, max_attempts=3):
    _meta_ans = _check_meta_question(question)
    if _meta_ans:
        return {
            "answer": _meta_ans,
            "sources": [],
            "retrieval_mode": "system-knowledge | prompt=meta",
        }
    for attempt in range(1, max_attempts + 1):
        try:
            short_term_text = ""
            if USE_FUSION_PACK:
                try:
                    short_term_answer = query_rewriting_pack.run(question)
                    short_term_text = _to_text(short_term_answer)
                except Exception as fusion_error:
                    console_log(f"Fusion retriever unavailable: {fusion_error}", "WARN")
                    short_term_text = ""

            current_doc_nodes = current_doc_retriever.retrieve(question)
            current_doc_text = _nodes_to_text(current_doc_nodes)

            palace_nodes, retrieval_mode_note = hierarchical_palace_retrieve(question)
            palace_text = _nodes_to_text(palace_nodes)

            if not short_term_text and not current_doc_text and not palace_text:
                return {
                    "answer": "\u26a0\ufe0f Information not found in source material.",
                    "sources": [],
                    "retrieval_mode": retrieval_mode_note,
                }

            prompt_mode, inline_text = _detect_prompt_mode(question)
            if not ENABLE_LLM_SYNTHESIS or not ENABLE_NETWORK_LLM:
                final_text = _compose_fast_answer(question, short_term_text, current_doc_text, palace_text)
            else:
                synthesis_prompt = _build_synthesis_prompt(
                    prompt_mode, question, inline_text,
                    current_doc_text, palace_text, short_term_text,
                )
                final_text = _replicate_create_and_poll(
                    synthesis_prompt,
                    max_tokens=SYNTHESIS_MAX_TOKENS,
                ) or "\u26a0\ufe0f Information not found in source material."

            refs = []
            refs.extend(_extract_sources_from_nodes("current-doc", current_doc_nodes))
            refs.extend(_extract_sources_from_nodes("palace", palace_nodes))
            refs = _dedupe_sources(refs)

            return {
                "answer": final_text,
                "sources": refs,
                "retrieval_mode": f"{retrieval_mode_note} | mode={RUNTIME_MODE} | prompt={prompt_mode}",
            }

        except Exception as e:
            error_msg = str(e).lower()
            if _is_timeout_error(e):
                if attempt < max_attempts:
                    wait_seconds = min(2 * attempt, 8)
                    print(
                        f"\u26a0\ufe0f Timeout detected (attempt {attempt}/{max_attempts}). "
                        f"Retrying in {wait_seconds}s..."
                    )
                    time.sleep(wait_seconds)
                    continue
                return {
                    "answer": (
                        f"\u26a0\ufe0f CONNECTION TIMEOUT: The network/API stream timed out after "
                        f"{max_attempts} attempts. Please retry in a few seconds."
                    ),
                    "sources": [],
                    "retrieval_mode": "timeout",
                }
            if _is_network_error(e):
                return {
                    "answer": (
                        "\U0001f310 OFFLINE \u2014 Cannot reach the Replicate API (DNS failure: getaddrinfo failed).\n"
                        "Your internet connection appears to be down. Please:\n"
                        "  1. Check your network / Wi-Fi connection\n"
                        "  2. Re-run this cell once connectivity is restored\n"
                        "Your question has been saved and can be re-asked in the next session."
                    ),
                    "sources": [],
                    "retrieval_mode": "offline",
                }
            if "422" in error_msg:
                return {
                    "answer": "\u26a0\ufe0f MODEL ERROR: Granite 3.1 is currently busy. Wait 10 seconds and retry.",
                    "sources": [],
                    "retrieval_mode": "model-error",
                }
            return {
                "answer": f"\u26a0\ufe0f SYSTEM ERROR: {e}",
                "sources": [],
                "retrieval_mode": "system-error",
            }

# ── Session management ─────────────────────────────────────────────────────────
def _list_sessions() -> list[pathlib.Path]:
    return sorted(_SESSION_DIR.glob("session_*.json"), reverse=True)

def _load_session(path: pathlib.Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

def _new_session() -> dict:
    ts = _dt_mod.datetime.now().strftime("%Y%m%d_%H%M%S")
    return {
        "session_id": ts,
        "started":    _dt_mod.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "exchanges":  [],
    }

def _save_session(data: dict, path: pathlib.Path) -> None:
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def _append_exchange(question: str, result: dict) -> None:
    global _session_data, _session_path
    if not _session_data:
        return
    _session_data["exchanges"].append({
        "timestamp":      _dt_mod.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "question":       question,
        "mode":           result.get("retrieval_mode", ""),
        "answer_snippet": (result.get("answer") or "")[:400],
    })
    if _session_path:
        _save_session(_session_data, _session_path)

def _start_session_prompt() -> None:
    global _session_data, _session_path
    sessions = _list_sessions()
    if sessions:
        last_path = sessions[0]
        last      = _load_session(last_path)
        n_ex      = len(last.get("exchanges", []))
        started   = last.get("started", "unknown")
        print(f"\n\U0001f4c2 Previous session found: {last.get('session_id', '?')} | started {started} | {n_ex} exchange(s)")
        choice = input("   Continue previous session or start new? [continue / new]: ").strip().lower()
        if choice in ("c", "continue", "cont"):
            _session_data = last
            _session_path = last_path
            _session_data.setdefault("exchanges", [])
            print(f"\u2705 Resumed session {_session_data['session_id']} ({n_ex} previous exchange(s) loaded)")
            if n_ex > 0:
                print("\n\U0001f4dc Last 3 exchanges in this session:")
                for ex in _session_data["exchanges"][-3:]:
                    print(f"  [{ex['timestamp']}] Q: {ex['question'][:80]}")
                    print(f"               A: {ex['answer_snippet'][:100]}...")
            return
    _session_data = _new_session()
    ts            = _session_data["session_id"]
    _session_path = _SESSION_DIR / f"session_{ts}.json"
    _save_session(_session_data, _session_path)
    print(f"\n\U0001f195 New session started: {ts}")

# ── Startup HTML banner ────────────────────────────────────────────────────────
from IPython.display import display as _ate_disp, HTML as _ate_HTML, clear_output as _ate_clear
_ate_disp(_ate_HTML(f"""
<div style='font-family:Segoe UI,Arial,sans-serif; background:#0a0a0a;
            border-top:3px solid #CFB53B; border-bottom:3px solid #CFB53B;
            border-left:3px solid #1A6FBF; border-right:3px solid #1A6FBF;
            border-radius:12px; padding:20px 26px; color:#d4e8f8;
            box-shadow:0 0 18px rgba(26,111,191,0.35),0 0 8px rgba(207,181,59,0.2);
            max-width:900px; margin:10px 0;'>
  <div style='font-size:10px; letter-spacing:2px; color:#CFB53B; text-transform:uppercase;'>
    &#9670; Academic Truth Engine v3 &#9670;</div>
  <div style='font-size:20px; font-weight:700; color:#d4e8f8; margin:6px 0 14px;'>
    &#127979; Research Engine Online</div>
  <div style='display:grid; grid-template-columns:1fr 1fr; gap:8px 20px;
              font-size:12px; line-height:1.8; color:#a0b8c8; margin-bottom:14px;'>
    <div><span style='color:#CFB53B;'>Runtime</span> &nbsp; {RUNTIME_MODE.upper()}</div>
    <div><span style='color:#CFB53B;'>Synthesis</span> &nbsp;
      {"ON &#9654; Essay Mode" if ENABLE_LLM_SYNTHESIS and ENABLE_NETWORK_LLM else "OFF &#9658; Local Only"}</div>
    <div><span style='color:#CFB53B;'>Visual Panel</span> &nbsp; {"ON" if VISUAL_PANEL_ON else "OFF"}</div>
    <div><span style='color:#CFB53B;'>Auto Brief</span> &nbsp; {"ON" if AUTO_BRIEF_ON else "OFF"}</div>
    <div><span style='color:#CFB53B;'>Fusion Pack</span> &nbsp; {"ON" if USE_FUSION_PACK else "OFF"}</div>
    <div><span style='color:#CFB53B;'>Brief Style</span> &nbsp; {BRIEF_STYLE_KEY}</div>
  </div>
  <div style='font-size:12px; color:#a0b8c8; border-top:1px solid #1a2a3a;
              padding-top:10px; line-height:1.9;'>
    <b style='color:#CFB53B;'>Modes</b> &mdash; auto-detected from phrasing:<br>
    &nbsp;&nbsp;<code style='color:#CFB53B;'>essay</code> full academic essay (default) &nbsp;&middot;&nbsp;
    <code style='color:#CFB53B;'>chat</code> direct Q&amp;A &nbsp;&middot;&nbsp;
    <code style='color:#CFB53B;'>summarise</code> concise summary<br>
    &nbsp;&nbsp;<code style='color:#CFB53B;'>paraphrase</code> human rewrite &nbsp;&middot;&nbsp;
    <code style='color:#CFB53B;'>formulate</code> intro + hypothesis + RQ &nbsp;&middot;&nbsp;
    <code style='color:#CFB53B;'>retrieval</code> structured overview<br>
    <b style='color:#CFB53B;'>Attachments</b> &mdash;
    PDF &nbsp;&middot;&nbsp; DOCX &nbsp;&middot;&nbsp; XLSX &nbsp;&middot;&nbsp;
    JPG / PNG / BMP / GIF (OCR) &nbsp;&middot;&nbsp; SVG
  </div>
</div>"""))

_start_session_prompt()

# ── File attachment text extractor ────────────────────────────────────────────
def _extract_attachment_text(name: str, content: bytes) -> str:
    """Extract plain text from an uploaded attachment.
    Supports: .pdf  .docx  .xlsx  .jpg/.jpeg/.png/.bmp/.gif (OCR)  .svg
    Returns extracted text, or an error note if extraction fails.
    """
    ext = pathlib.Path(name).suffix.lower()
    try:
        # ── PDF ───────────────────────────────────────────────────────────────
        if ext == ".pdf":
            import io
            import pdfplumber
            _parts: list[str] = []
            with pdfplumber.open(io.BytesIO(content)) as _pdf:
                for _pg in _pdf.pages:
                    _t = _pg.extract_text()
                    if _t and _t.strip():
                        _parts.append(_t.strip())
            return "\n\n".join(_parts)

        # ── DOCX ──────────────────────────────────────────────────────────────
        if ext == ".docx":
            import io
            import docx as _docx_mod
            _doc = _docx_mod.Document(io.BytesIO(content))
            return "\n".join(p.text for p in _doc.paragraphs if p.text.strip())

        # ── XLSX ──────────────────────────────────────────────────────────────
        if ext == ".xlsx":
            import io
            import openpyxl as _xl
            _wb = _xl.load_workbook(io.BytesIO(content), read_only=True, data_only=True)
            _rows: list[str] = []
            for _ws in _wb.worksheets:
                for _row in _ws.iter_rows(values_only=True):
                    _rt = "\t".join("" if c is None else str(c) for c in _row)
                    if _rt.strip():
                        _rows.append(_rt)
            return "\n".join(_rows)

        # ── Raster images — OCR via pytesseract ───────────────────────────────
        if ext in (".jpg", ".jpeg", ".png", ".bmp", ".gif"):
            import io
            from PIL import Image as _PILImg
            import pytesseract as _tess
            if "tess_path" in globals() and tess_path:
                _tess.pytesseract.tesseract_cmd = tess_path
            _img = _PILImg.open(io.BytesIO(content))
            return _tess.image_to_string(_img).strip()

        # ── SVG — extract text nodes via ElementTree ──────────────────────────
        if ext == ".svg":
            import xml.etree.ElementTree as _ET
            _root = _ET.fromstring(content.decode("utf-8", errors="replace"))
            _texts: list[str] = []
            for _el in _root.iter():
                if _el.text and _el.text.strip():
                    _texts.append(_el.text.strip())
                if _el.tail and _el.tail.strip():
                    _texts.append(_el.tail.strip())
            return "\n".join(_texts)

    except Exception as _ex:
        return f"[Could not extract text from {name}: {_ex}]"
    return ""

# ── Helper: normalise ipywidgets FileUpload value across versions ──────────────
def _get_upload_files() -> list[tuple[str, bytes]]:
    """Return list of (filename, bytes) pairs from the FileUpload widget.
    Handles both ipywidgets 7.x (dict) and 8.x (tuple-of-dicts) APIs.
    """
    val = _qa_upload.value
    files: list[tuple[str, bytes]] = []
    if isinstance(val, (list, tuple)):
        for item in val:
            if isinstance(item, dict):
                files.append((item.get("name", "file"), bytes(item.get("content", b""))))
    elif isinstance(val, dict):
        for fname, fdata in val.items():
            files.append((fname, bytes(fdata.get("content", b""))))
    return files

# ── Rich Q&A widget UI ────────────────────────────────────────────
import ipywidgets as _widgets

# ── Left-column widgets ────────────────────────────────────────────
_qa_text = _widgets.Textarea(
    placeholder=(
        "Start typing your research inquiry or topic here for real-time analysis...\n"
        "Use natural language. Paste direct text for summarization.\n\n"
        "Type your research question here...\n"
        "For summarise / paraphrase modes, paste your text directly."
    ),
    layout=_widgets.Layout(width="100%", min_height="220px"),
)
_qa_text.add_class('ate-research-input')

_qa_upload = _widgets.FileUpload(
    accept=".pdf,.docx,.xlsx,.jpeg,.gif,.jpg,.png,.bmp,.svg",
    multiple=True,
    description="📎 Attach (0)",
    style=_widgets.ButtonStyle(button_color="#0d2a4a", font_color="#d4e8f8", font_weight="bold"),
    layout=_widgets.Layout(width="148px"),
)

_qa_submit_btn = _widgets.Button(
    description="🔍  Research",
    button_style="primary",
    style=_widgets.ButtonStyle(button_color="#1A6FBF", font_weight="bold"),
    layout=_widgets.Layout(width="142px", height="34px"),
    tooltip="Submit your research question",
)
_qa_clear_btn = _widgets.Button(
    description="✕ Clear",
    button_style="",
    style=_widgets.ButtonStyle(button_color="#CFB53B", font_color="#000000", font_weight="bold"),
    layout=_widgets.Layout(width="90px"),
    tooltip="Clear question and attachments",
)
_qa_history_btn = _widgets.Button(
    description="📑 History",
    button_style="",
    style=_widgets.ButtonStyle(button_color="#1A4A7F", font_color="#d4e8f8", font_weight="bold"),
    layout=_widgets.Layout(width="110px"),
    tooltip="Show session history",
)
_qa_end_btn = _widgets.Button(
    description="○ End",
    button_style="",
    style=_widgets.ButtonStyle(button_color="#CFB53B", font_color="#000000", font_weight="bold"),
    layout=_widgets.Layout(width="80px"),
    tooltip="Close the Research Engine",
)

_qa_status = _widgets.HTML(
    value="<span style='color:#6a8fa8;font-size:12px;font-family:Consolas,monospace;'>"
          "Ready — enter a question or attach a file.</span>"
)
_qa_output = _widgets.Output()
_qa_side   = _widgets.Output()   # right-column live panel

# ── Side-panel builder ─────────────────────────────────────────────────
def _qa_build_side_html() -> str:
    """Return HTML for the right-hand Live Source Overview + Status Monitor."""
    _src_rows = ""
    _seen = set()
    _pdf_list = [s for s in (PDF_SOURCES if "PDF_SOURCES" in globals() else []) if s]
    for _p in _pdf_list[:5]:
        _short = str(_p).replace("\\", "/").split("/")[-1]
        if _short in _seen:
            continue
        _seen.add(_short)
        _src_rows += (
            "<div style='display:flex;align-items:flex-start;gap:8px;"
            "margin-bottom:10px;'>"
            "<span style='font-size:15px;'>🗎️</span>"
            "<span style='font-size:12px;color:#d4e8f8;line-height:1.4;'>"
            f"<strong>{_short[:34]}</strong><br>"
            "<span style='color:#CFB53B;font-size:10px;'>Linked (Deep Synthesis)</span>"
            "</span></div>"
        )
    _pal_count = 0
    try:
        if "palace_collection" in globals():
            _pal_count = palace_collection.count()
    except Exception:
        pass
    if _pal_count:
        _src_rows += (
            "<div style='display:flex;align-items:flex-start;gap:8px;margin-bottom:10px;'>"
            "<span style='font-size:15px;'>🌐</span>"
            "<span style='font-size:12px;color:#d4e8f8;line-height:1.4;'>"
            f"<strong>Memory Palace ({_pal_count:,} vectors)</strong><br>"
            "<span style='color:#50c878;font-size:10px;'>Ready for Query</span>"
            "</span></div>"
        )
    if not _src_rows:
        _src_rows = (
            "<div style='color:#6a8fa8;font-size:11px;"
            "font-family:Consolas,monospace;'>(no sources loaded)</div>"
        )

    def _bar(filled: int, total: int = 8) -> str:
        filled = max(0, min(filled, total))
        return "█" * filled + "░" * (total - filled)

    _cov_fill  = min(8, int((_pal_count / 3000) * 8)) if _pal_count else 2
    _cit_fill  = min(8, len(_pdf_list) * 2)
    _cov_label = "(low, build required)" if _cov_fill < 3 else "(good coverage)"

    _status_html = (
        "<div style='font-family:Consolas,monospace;font-size:11px;"
        "color:#a0c4dc;line-height:1.9;'>"
        f"<div>Source Coverage <span style='color:#CFB53B;'>[{_bar(_cov_fill)}]</span></div>"
        f"<div style='font-size:9px;color:#6a8fa8;margin-top:-4px;margin-bottom:6px;'>{_cov_label}</div>"
        f"<div>Citations Found&nbsp;&nbsp;<span style='color:#CFB53B;'>[{_bar(_cit_fill)}]</span></div>"
        "<div style='font-size:9px;color:#6a8fa8;margin-top:-4px;margin-bottom:6px;'>&nbsp;</div>"
        f"<div>Contradictions&nbsp;&nbsp;&nbsp;<span style='color:#CFB53B;'>[{_bar(1)}]</span></div>"
        "</div>"
    )

    return (
        "<div style='font-family:Segoe UI,Arial,sans-serif;display:flex;"
        "flex-direction:column;gap:12px;'>"
        "<div style='background:#0d1117;border:1px solid #1A6FBF;border-radius:10px;"
        "padding:14px 16px;'>"
        "<div style='font-size:13px;font-weight:700;color:#d4e8f8;margin-bottom:12px;'>"
        "📚 Live Source Overview</div>"
        + _src_rows +
        "</div>"
        "<div style='background:#0d1117;border:1px solid #1A6FBF;border-radius:10px;"
        "padding:14px 16px;'>"
        "<div style='font-size:13px;font-weight:700;color:#d4e8f8;margin-bottom:10px;'>"
        "⚙️ Status Monitor</div>"
        + _status_html +
        "</div>"
        "</div>"
    )

def _qa_refresh_side() -> None:
    with _qa_side:
        _ate_clear(wait=False)
        _ate_disp(_ate_HTML(_qa_build_side_html()))

# ── Widget helpers ─────────────────────────────────────────────────────
def _qa_set_status(msg: str, colour: str = "#6a8fa8") -> None:
    _qa_status.value = (
        f"<span style='color:{colour};font-size:12px;"
        f"font-family:Consolas,monospace;'>{msg}</span>"
    )

def _qa_clear_upload() -> None:
    try:
        _qa_upload.value = ()
        _qa_upload.description = "📎 Attach (0)"
    except Exception:
        try:
            _qa_upload.value = {}
        except Exception:
            pass

# ── Button callbacks ──────────────────────────────────────────────────────
def _on_clear(_b=None) -> None:
    _qa_text.value = ""
    _qa_clear_upload()
    _qa_set_status("Cleared — enter a new question.")
    with _qa_output:
        _ate_clear()

def _show_history(_b=None) -> None:
    exs = (_session_data or {}).get("exchanges", [])
    sid = (_session_data or {}).get("session_id", "?")
    if not exs:
        _hist_rows = "<p style='color:#9ab8cc;'>No exchanges in the current session yet.</p>"
    else:
        _hist_rows = "".join(
            f"<div style='border-left:3px solid #CFB53B;padding:10px 14px;"
            f"background:#0d0d0d;border-radius:4px;margin-bottom:12px;'>"
            f"<div style='font-size:11px;color:#6a8fa8;margin-bottom:3px;'>"
            f"[{idx2}] {ex['timestamp']}</div>"
            f"<div style='color:#a8c4e0;font-size:12px;margin-bottom:4px;'>"
            f"<span style='color:#CFB53B;font-weight:600;'>Q:</span> {ex['question'][:120]}</div>"
            f"<div style='font-size:13px;color:#a0b8c8;background:#111;padding:8px 12px;"
            f"border-radius:4px;border-left:2px solid #1A6FBF;white-space:pre-wrap;'>"
            f"<span style='color:#CFB53B;font-weight:600;'>A:</span> {ex['answer_snippet'][:300]}…</div></div>"
            for idx2, ex in enumerate(exs, 1)
        )
    with _qa_output:
        _ate_clear(wait=True)
        _ate_disp(_ate_HTML(
            "<div style='font-family:Segoe UI,Consolas,Arial,sans-serif;"
            "border-top:3px solid #CFB53B;border-bottom:3px solid #CFB53B;"
            "border-left:3px solid #1A6FBF;border-right:3px solid #1A6FBF;"
            "border-radius:12px;padding:20px;background:#0a0a0a;color:#d4e8f8;"
            "box-shadow:0 0 18px rgba(26,111,191,0.35);'>"
            "<div style='font-size:11px;letter-spacing:2px;color:#CFB53B;"
            "text-transform:uppercase;margin-bottom:4px;'>&#9670; Session History &#9670;</div>"
            "<div style='font-size:16px;font-weight:700;color:#d4e8f8;"
            "border-bottom:1px solid #1A6FBF;padding-bottom:8px;margin-bottom:14px;'>"
            f"Session {sid} &mdash; {len(exs)} exchange(s)</div>"
            + _hist_rows +
            "</div>"
        ))

def _on_end(_b=None) -> None:
    for _btn in (_qa_submit_btn, _qa_end_btn, _qa_history_btn, _qa_clear_btn):
        _btn.disabled = True
    _qa_set_status("🛑 Research Engine closed.", "#e07070")
    with _qa_output:
        _ate_clear(wait=True)
        print("🛑 Closing Research Engine. Good luck with your assignment!")

def _on_submit(_b=None) -> None:
    question = _qa_text.value.strip()
    uploads  = _get_upload_files()

    attachment_text  = ""
    attachment_label = ""
    if uploads:
        _qa_upload.description = f"📎 Attach ({len(uploads)})"
        _parts = []
        for _fn, _fc in uploads:
            try:
                _parts.append(f"[Attachment: {_fn}]\n{_extract_attachment_text(_fn, _fc)}")
            except Exception as _ae:
                _parts.append(f"[Attachment: {_fn}] (extraction failed: {_ae})")
        attachment_text  = "\n\n".join(_parts)
        attachment_label = ", ".join(fn for fn, _ in uploads)

    if not question and not attachment_text:
        _qa_set_status("⚠️ Please enter a question or attach a file.", "#e0a000")
        return

    full_question = question
    if attachment_text:
        full_question = (question + "\n\n" + attachment_text).strip() if question else attachment_text

    _mode, _ = _detect_prompt_mode(full_question)
    _label = _mode
    _depth = f"{RUNTIME_MODE.upper()} · "
    _qa_set_status(
        f"🔍 Mode: {_label} | {_depth}retrieval running… (30–120 s)",
        "#1A6FBF",
    )
    _qa_submit_btn.disabled = True

    with _qa_output:
        _ate_clear(wait=True)
        if attachment_label:
            print(f"📎 Attachments: {attachment_label}")
        print(f"🔍 Mode: {_label} | {_depth}retrieval running… (30–120 s)")

    try:
        _start = time.time()
        result = run_academic_query(full_question, max_attempts=3)
        _append_exchange(full_question, result)
        _dur = time.time() - _start

        _answer   = result.get("answer", "")
        _sources  = result.get("sources", [])
        _ret_mode = result.get("retrieval_mode", "unknown")

        with _qa_output:
            _ate_clear(wait=True)
            if VISUAL_PANEL_ON and "render_answer_panel" in globals():
                try:
                    render_answer_panel(full_question, result)
                except Exception as _rpe:
                    print(f"⚠️ Visual panel render failed: {_rpe}")
                print(f"⏱️  {_dur:.1f}s  |  {_ret_mode}")
            else:
                print("\n" + "=" * 60)
                print("📝 ACADEMIC ESSAY (PDF-Grounded | Granite 3.1 + MemPalace)")
                print("=" * 60)
                print(_answer)
                print("=" * 60)
                print(f"⏱️  Essay completed in {_dur:.1f} seconds.")
                print(f"🧭 Retrieval Mode: {_ret_mode}")
                print("\n📍 SOURCES:")
                if not _sources:
                    print("- No traceable source nodes returned for this answer.")
                else:
                    for _ref in _sources[:SOURCE_PRINT_LIMIT]:
                        _snip = _ref["snippet"] or "No snippet available"
                        print(
                            f"- [{_ref['retriever']}] {_ref['file_name']} | "
                            f"wing={_ref['wing']} | room={_ref['room']} | hall={_ref['hall']} | "
                            f"drawer={_ref['drawer_type']} | \"{_snip}...\""
                        )

        if AUTO_BRIEF_ON and "build_slide_brief" in globals():
            try:
                _cs = BRIEF_STYLE_KEY
                if "style_library" in globals():
                    _ck = list((style_library or {}).get("curated", {}).keys())
                    if _ck and _cs not in _ck:
                        _cs = _ck[0]
                _brief = build_slide_brief(full_question, result, style_key=_cs)
                with _qa_output:
                    print(f"🧾 Slide brief generated (style={_cs}).")
                if AUTO_BRIEF_EXPORT_ON and "export_slide_brief" in globals():
                    try:
                        _bpath = export_slide_brief(_brief)
                        with _qa_output:
                            print(f"💾 Slide brief saved: {_bpath}")
                    except Exception as _ee:
                        with _qa_output:
                            print(f"⚠️ Slide brief export failed: {_ee}")
            except Exception as _be:
                with _qa_output:
                    print(f"⚠️ Slide brief generation failed: {_be}")

        _qa_set_status(f"✅ Done in {_dur:.1f}s | {_ret_mode}", "#50c878")
        _qa_refresh_side()

    except Exception as _submit_err:
        with _qa_output:
            print(f"⚠️ SYSTEM ERROR: {_submit_err}")
        _qa_set_status(f"⚠️ Error: {_submit_err}", "#e07070")

    finally:
        _qa_submit_btn.disabled = False
        _qa_clear_upload()

# ── Wire up callbacks ──────────────────────────────────────────────────────
_qa_clear_btn.on_click(_on_clear)
_qa_history_btn.on_click(_show_history)
_qa_end_btn.on_click(_on_end)
_qa_submit_btn.on_click(_on_submit)

# ── Build initial side panel ────────────────────────────────────────────────
_qa_refresh_side()

# ── Top header bar ───────────────────────────────────────────────────────────
_top_bar = _widgets.HTML(value=
    f"<div style='font-family:Segoe UI,Arial,sans-serif;background:#0a0a0a;"
    f"border-bottom:1px solid #1A4A7F;border-radius:10px 10px 0 0;"
    f"padding:18px 22px 14px;'>"
    f"<div style='font-size:10px;letter-spacing:2px;color:#CFB53B;"
    f"text-transform:uppercase;margin-bottom:4px;'>&#9670; Academic Truth Engine v3 &#9670;</div>"
    f"<div style='font-size:20px;font-weight:700;color:#d4e8f8;margin:4px 0 14px;'>"
    f"&#127979; Research Engine Online</div>"
    f"<div style='display:grid;grid-template-columns:1fr 1fr;gap:4px 20px;"
    f"font-size:12px;line-height:1.9;color:#a0b8c8;margin-bottom:12px;'>"
    f"<div><span style='color:#CFB53B;'>Runtime</span>&nbsp;&nbsp;{RUNTIME_MODE.upper()}</div>"
    f"<div><span style='color:#CFB53B;'>Synthesis</span>&nbsp;&nbsp;"
    f"{'ON &#9654; Essay Mode' if ENABLE_LLM_SYNTHESIS and ENABLE_NETWORK_LLM else 'OFF &#9658; Local Only'}</div>"
    f"<div><span style='color:#CFB53B;'>Visual Panel</span>&nbsp;&nbsp;{'ON' if VISUAL_PANEL_ON else 'OFF'}</div>"
    f"<div><span style='color:#CFB53B;'>Fusion Pack</span>&nbsp;&nbsp;{'ON' if USE_FUSION_PACK else 'OFF'}</div>"
    f"</div>"
    f"</div>"
)

# ── Modes / attachments info bar ─────────────────────────────────────────────
_modes_bar = _widgets.HTML(value=
    "<div style='font-family:Segoe UI,Arial,sans-serif;font-size:12px;"
    "color:#a0b8c8;line-height:1.9;border-top:1px solid #1a2a3a;padding:8px 0 4px;'>"
    "<b style='color:#CFB53B;'>Modes</b> &mdash; auto-detected from phrasing:<br>"
    "&nbsp;&nbsp;<code style='color:#CFB53B;background:transparent;border:none;'>essay</code>"
    " full academic essay (default) &nbsp;&middot;&nbsp;"
    "<code style='color:#CFB53B;background:transparent;border:none;'>chat</code>"
    " direct Q&amp;A &nbsp;&middot;&nbsp;"
    "<code style='color:#CFB53B;background:transparent;border:none;'>summarise</code>"
    " concise summary<br>"
    "&nbsp;&nbsp;<code style='color:#CFB53B;background:transparent;border:none;'>paraphrase</code>"
    " human rewrite &nbsp;&middot;&nbsp;"
    "<code style='color:#CFB53B;background:transparent;border:none;'>formulate</code>"
    " intro + hypothesis + RQ &nbsp;&middot;&nbsp;"
    "<code style='color:#CFB53B;background:transparent;border:none;'>retrieval</code>"
    " structured overview<br>"
    "<b style='color:#CFB53B;'>Attachments</b> &mdash;"
    " PDF &nbsp;&middot;&nbsp; DOCX &nbsp;&middot;&nbsp; XLSX &nbsp;&middot;&nbsp;"
    " JPG / PNG / BMP / GIF (OCR) &nbsp;&middot;&nbsp; SVG"
    "</div>"
)

# ── Left column ────────────────────────────────────────────────────────────────────────
_left_col = _widgets.VBox(
    [
        _widgets.HBox(
            [_qa_upload, _qa_history_btn, _qa_clear_btn, _qa_end_btn],
            layout=_widgets.Layout(align_items="center", gap="6px", flex_flow="row wrap"),
        ),
        _qa_text,
        _modes_bar,
        _widgets.HBox(
            [_qa_submit_btn, _qa_status],
            layout=_widgets.Layout(align_items="center", gap="10px"),
        ),
    ],
    layout=_widgets.Layout(gap="6px", flex="1 1 0", min_width="320px"),
)

# ── Two-column layout ──────────────────────────────────────────────────────────────
_main_cols = _widgets.HBox(
    [_left_col, _qa_side],
    layout=_widgets.Layout(gap="14px", align_items="flex-start", width="100%"),
)

# ── Full UI display ───────────────────────────────────────────────────────────────
# ── Inject full widget CSS (banner-matched) ───────────────────────────────
_ate_disp(_ate_HTML("""
<style>
/* ── Outer panel: banner-style gold+blue quad border ── */
.ate-main-panel {
    border-top: 3px solid #CFB53B !important;
    border-bottom: 3px solid #CFB53B !important;
    border-left: 3px solid #1A6FBF !important;
    border-right: 3px solid #1A6FBF !important;
    border-radius: 12px !important;
    box-shadow: 0 0 18px rgba(26,111,191,0.35), 0 0 8px rgba(207,181,59,0.2) !important;
}

/* ── Textarea: shiny black + blue text ── */
.ate-research-input textarea {
    background: linear-gradient(160deg, #141414 0%, #0a0a0a 55%, #080c10 100%) !important;
    box-shadow: inset 0 2px 8px rgba(0,0,0,0.7), inset 0 0 0 1px rgba(26,111,191,0.35) !important;
    border: 1px solid #1A4A7F !important;
    border-radius: 8px !important;
    color: #d4e8f8 !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
    font-size: 13px !important;
    line-height: 1.7 !important;
    caret-color: #CFB53B !important;
    padding: 12px 14px !important;
    resize: vertical !important;
    transition: border-color 0.2s, box-shadow 0.2s !important;
}
.ate-research-input textarea:focus {
    border-color: #1A6FBF !important;
    box-shadow: inset 0 2px 8px rgba(0,0,0,0.7),
                0 0 10px rgba(26,111,191,0.45),
                inset 0 0 0 1px rgba(26,111,191,0.6) !important;
    outline: none !important;
}
.ate-research-input textarea::placeholder {
    color: #4a7a9a !important;
    font-style: italic !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
    font-size: 12px !important;
}
.ate-research-input { background: #0a0a0a !important; }

/* ── HTML widgets: banner body text colour ── */
.widget-html-content, .widget-html-content * {
    font-family: 'Segoe UI', Arial, sans-serif !important;
}

/* ── code tags inside widgets match banner gold ── */
.widget-html-content code {
    color: #CFB53B !important;
    background: #0a0a0a !important;
    border: none !important;
    font-family: Consolas, monospace !important;
    font-size: 11px !important;
}

/* ── Widget containers: transparent over black base ── */
.widget-vbox, .widget-hbox, .widget-box {
    background: #0a0a0a !important;
}

/* ── Output area: black bg + banner body text ── */
.jp-OutputArea-output, .jp-OutputArea {
    background: #0a0a0a !important;
}
.jp-OutputArea-output p, .jp-OutputArea-output div,
.jp-OutputArea-output span, .jp-OutputArea-output pre {
    color: #a0b8c8 !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
}

/* ── All buttons: shared base ── */
.widget-button {
    font-family: 'Segoe UI', Arial, sans-serif !important;
    font-size: 12px !important;
    border-radius: 6px !important;
    border: none !important;
    transition: filter 0.15s !important;
    letter-spacing: 0.3px !important;
}
.widget-button:hover { filter: brightness(1.18) !important; }
.widget-button:active { filter: brightness(0.88) !important; }

/* ── Upload (FileUpload) button: blue ── */
.widget-upload .jupyter-button, .widget-upload button {
    background: #0d2a4a !important;
    color: #d4e8f8 !important;
    border: 1px solid #1A6FBF !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
    font-size: 12px !important;
    border-radius: 6px !important;
}

/* ── Output widget: black background ── */
.widget-output {
    background: #0a0a0a !important;
}
body, .jp-RenderedHTMLCommon {
    background-color: #0a0a0a !important;
    color: #a0b8c8 !important;
}
pre, code {
    background: #0a0a0a !important;
    color: #a0b8c8 !important;
}
</style>
"""))

_outer_box = _widgets.VBox(
    [_top_bar, _main_cols, _qa_output],
    layout=_widgets.Layout(
        gap="0px",
        width="100%",
        max_width="960px",
        background_color="#0a0a0a",
    ),
)
_outer_box.add_class('ate-main-panel')
_ate_disp(_outer_box)



📂 Previous session found: 20260522_120305 | started 2026-05-22 12:03:05 | 1 exchange(s)

🆕 New session started: 20260522_185335


In [23]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 18 — Rapid Paper Q&A  ◆  Academic Truth Engine v3
# ───────────────────────────────────────────────────────────────────────────────
# Phase 1 → paste/upload PDFs in widget panel  →  click Load & Index
# Phase 2 → widget Q&A interface styled identically to cell 17
#
# ANSWER MODES  (prefix your question to choose, or just ask naturally):
#   !short / !long / !summarise / !bullet / !compare / !define / !critique
# COMMANDS:  help · history · clear · end
# ✦ No ChromaDB · Ephemeral in-memory index only
# ✦ Reuses REPLICATE_API_TOKEN, embed_model, splitter from the main engine
# ═══════════════════════════════════════════════════════════════════════════════

# ─── ① CONFIG ────────────────────────────────────────────────────────────────
PDF_SOURCES = []   # leave empty → Phase-1 widget collects at runtime

PR_TOP_K       = 6
PR_MAX_TOKENS  = {
    "short":    350,  "long":     950,  "summarise": 900,
    "bullet":   700,  "compare":  800,  "define":    400,
    "critique": 850,  "auto":     700,
}
PR_TEMPERATURE    = 0.15
PR_HISTORY_CTX    = 3
PR_VERBATIM_NGRAM = 8
PR_VERBATIM_WARN  = 0.05
PR_VERBATIM_ALERT = 0.15
# ─────────────────────────────────────────────────────────────────────────────

import os, re, time, tempfile, pathlib
import ipywidgets as _widgets
from IPython.display import display as _pr_display, HTML as _pr_HTML, clear_output as _pr_clear
import replicate as _pr_replicate
from datetime import datetime as _pr_dt

# ── Token guard ───────────────────────────────────────────────────────────────
_pr_token = globals().get("replicate_token") or os.getenv("REPLICATE_API_TOKEN", "")
if not _pr_token:
    raise RuntimeError(
        "No REPLICATE_API_TOKEN found. Run Cell 3 first, or set\n"
        "  os.environ['REPLICATE_API_TOKEN'] = '<your_token>'"
    )
os.environ["REPLICATE_API_TOKEN"] = _pr_token

_PR_MODEL   = "ibm-granite/granite-3.1-8b-instruct"
_PR_TIMEOUT = 300
_PR_POLL    = 3

# ── Shared mutable state (closures need a container, not bare variables) ──────
_pr_state = {
    "nodes": [], "retriever": None, "labels": [],
    "history": [], "ctx_mem": [], "q_num": 0,
}

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS — download, extract, mode, verbatim, LLM, HTML panels
# ══════════════════════════════════════════════════════════════════════════════
def _pr_is_gdrive(s):
    return "drive.google.com" in s or "drive.usercontent.google.com" in s

def _pr_file_id(url):
    for pat in [r"/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    raise ValueError(f"Cannot parse Google Drive file ID from: {url!r}")

def _pr_download_gdrive(url, dest):
    import requests as _req
    fid = _pr_file_id(url)
    try:
        import gdown
        r = gdown.download(id=fid, output=dest, quiet=True)
        if r and os.path.exists(dest):
            with open(dest, "rb") as _f:
                if _f.read(5) == b"%PDF-":
                    print(f"    [gdown] \u2705 {pathlib.Path(dest).name}")
                    return
            os.remove(dest)
    except Exception as _e:
        print(f"    [gdown] \u26a0\ufe0f  {_e}")
    _sess = _req.Session()
    _hdrs = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36",
        "Accept": "application/pdf,application/octet-stream,*/*",
        "Referer": "https://drive.google.com/",
    }
    for _base, _params in [
        ("https://drive.usercontent.google.com/download", {"id": fid, "export": "download", "confirm": "t"}),
        ("https://drive.google.com/uc", {"export": "download", "id": fid, "confirm": "t"}),
    ]:
        try:
            _r = _sess.get(_base, params=_params, headers=_hdrs, stream=True, timeout=90)
            _tok2 = None
            for _k, _v in _r.cookies.items():
                if _k.startswith("download_warning"):
                    _tok2 = _v; break
            if not _tok2 and "text/html" in _r.headers.get("Content-Type", ""):
                _mx = re.search(r"confirm=([0-9A-Za-z_-]+)",
                                _r.content.decode("utf-8", errors="ignore"))
                if _mx:
                    _tok2 = _mx.group(1)
            if _tok2:
                _r = _sess.get(_base, params={**_params, "confirm": _tok2},
                               headers=_hdrs, stream=True, timeout=90)
            with open(dest, "wb") as _fh:
                for _chunk in _r.iter_content(32768):
                    _fh.write(_chunk)
            with open(dest, "rb") as _fh:
                if _fh.read(5) == b"%PDF-":
                    print(f"    [requests] \u2705 {_base}")
                    return
            os.remove(dest)
        except Exception as _e2:
            print(f"    [requests] \u26a0\ufe0f  {_base}: {_e2}")
    raise RuntimeError(f"All download attempts failed for Drive ID: {fid}")

def _pr_extract_text(path):
    try:
        import fitz
        _doc = fitz.open(path)
        _pages = [pg.get_text("text").strip() for pg in _doc if pg.get_text("text").strip()]
        _doc.close()
        if _pages:
            return "\n\n".join(_pages)
    except Exception as _e:
        print(f"    [fitz] \u26a0\ufe0f  {_e}")
    _SDR = globals().get("SimpleDirectoryReader")
    if _SDR is None:
        from llama_index.core import SimpleDirectoryReader as _SDR
    return "\n\n".join(d.text for d in _SDR(input_files=[path]).load_data() if d.text.strip())

_ANTI_PLAGIARISM = (
    "ZERO-PLAGIARISM RULE: You must NOT copy or reproduce any sentence verbatim "
    "from the source excerpts. Synthesize, paraphrase, and express every idea in "
    "your own academic words. If a specific technical term or very short phrase is "
    "unavoidable, enclose it in quotation marks and attribute it. "
    "Producing copied text constitutes plagiarism and is strictly forbidden. "
)

_MODE_PREFIXES = {
    "!short": "short", "!long": "long",
    "!summarise": "summarise", "!summarize": "summarise",
    "!bullet": "bullet", "!compare": "compare",
    "!define": "define", "!critique": "critique",
}

_MODE_AUTO_RULES = [
    (r"\b(define|what is|what are|meaning of)\b",                    "define"),
    (r"\b(compare|contrast|differ|versus|vs\.?)\b",                  "compare"),
    (r"\b(critique|evaluate|assess|strength|weakness|flaw)\b",       "critique"),
    (r"\b(list|outline|key points|main points|steps|factors)\b",     "bullet"),
    (r"\b(briefly|quick|short|tldr|one sentence)\b",                 "short"),
    (r"\b(summarise|summarize|summary|overview of the document)\b",  "summarise"),
    (r"\b(explain in detail|elaborate|discuss at length|full analysis)\b", "long"),
]

_MODE_INSTRUCTIONS = {
    "short":    (_ANTI_PLAGIARISM + "Answer in ONE concise paragraph (3–5 sentences). Cite inline as [SOURCE n]."),
    "long":     (_ANTI_PLAGIARISM + "Full academic answer across multiple paragraphs. Cover context, argument, evidence, implications. Cite as [SOURCE n]."),
    "summarise":(_ANTI_PLAGIARISM + "4–6 paragraph structured summary: (1) Overview, (2) Key themes, (3) Evidence, (4) Conclusions, (5) Significance. Full paragraphs only. Cite as [SOURCE n]."),
    "bullet":   (_ANTI_PLAGIARISM + "Structured bullet/numbered points, each paraphrased in your own words with an inline citation [SOURCE n]. One-sentence summary at the end."),
    "compare":  (_ANTI_PLAGIARISM + "Compare and contrast concepts/frameworks: similarities, differences, implications. Cite as [SOURCE n]."),
    "define":   (_ANTI_PLAGIARISM + "Precise academic definition as the source presents it, paraphrased in your own words. Then plain-language explanation. Cite [SOURCE n]. Under 200 words."),
    "critique": (_ANTI_PLAGIARISM + "Critical analysis: strengths, weaknesses, limitations. Balanced, evidence-grounded. Cite as [SOURCE n]."),
    "auto":     (_ANTI_PLAGIARISM + "Answer thoroughly, paraphrasing source evidence. Cite inline as [SOURCE n]. Be academically precise."),
}

def _pr_detect_mode(q):
    ql = q.lower()
    for pfx, mode in _MODE_PREFIXES.items():
        if ql.startswith(pfx):
            return mode, q[len(pfx):].strip()
    for pattern, mode in _MODE_AUTO_RULES:
        if re.search(pattern, ql):
            return mode, q
    return "auto", q

def _pr_mode_label(mode):
    return {
        "short": "⚡ Short", "long": "📖 Long", "summarise": "📄 Summarise",
        "bullet": "📋 Bullet", "compare": "⚖️  Compare",
        "define": "📌 Define", "critique": "🔬 Critique", "auto": "🔍 Auto",
    }.get(mode, mode.title())

def _pr_verbatim_check(answer, hits):
    ans_words = re.findall(r'\w+', answer.lower())
    n = PR_VERBATIM_NGRAM
    max_ratio, worst = 0.0, ""
    for h in hits:
        src_words = re.findall(r'\w+', h.node.get_content().lower())
        src_grams = set()
        for i in range(max(0, len(src_words) - n + 1)):
            src_grams.add(tuple(src_words[i:i + n]))
        if not src_grams:
            continue
        total   = max(1, len(ans_words) - n + 1)
        matches = 0
        for i in range(len(ans_words) - n + 1):
            gram = tuple(ans_words[i:i + n])
            if gram in src_grams:
                matches += 1
                if not worst:
                    worst = " ".join(gram)
        ratio = matches / total
        if ratio > max_ratio:
            max_ratio = ratio
    pct = round(max_ratio * 100, 1)
    if max_ratio < PR_VERBATIM_WARN:
        badge = (f"<span style='background:#1a4a1a;color:#4CAF50;font-size:11px;"
                 f"padding:3px 10px;border-radius:4px;font-weight:600;'>"
                 f"\u2705 Original synthesis \u00b7 {pct}% overlap</span>")
    elif max_ratio < PR_VERBATIM_ALERT:
        badge = (f"<span style='background:#3a2e00;color:#FFB300;font-size:11px;"
                 f"padding:3px 10px;border-radius:4px;font-weight:600;'>"
                 f"\u26a0\ufe0f Review citations \u00b7 {pct}% overlap</span>")
    else:
        badge = (f"<span style='background:#3a0000;color:#EF5350;font-size:11px;"
                 f"padding:3px 10px;border-radius:4px;font-weight:600;'>"
                 f"\U0001f6a8 Verbatim risk \u00b7 {pct}% overlap — rephrase before using</span>")
    return max_ratio, worst, badge

def _pr_granite(prompt, mode="auto"):
    _max  = PR_MAX_TOKENS.get(mode, 700)
    _poll = globals().get("_replicate_create_and_poll")
    if _poll is not None:
        try:
            return _poll(prompt, max_tokens=_max)
        except Exception:
            pass
    _pred = _pr_replicate.predictions.create(
        model=_PR_MODEL,
        input={"prompt": prompt, "max_tokens": _max,
               "temperature": PR_TEMPERATURE, "top_p": 0.9, "repetition_penalty": 1.1},
    )
    _t0 = time.time()
    while True:
        _pred.reload()
        if _pred.status in ("succeeded", "failed", "canceled"):
            break
        if time.time() - _t0 > _PR_TIMEOUT:
            raise TimeoutError(f"Timed out after {_PR_TIMEOUT}s.")
        time.sleep(_PR_POLL)
    if _pred.status != "succeeded":
        raise RuntimeError(f"Prediction {_pred.status}: {_pred.error}")
    out = _pred.output
    return "".join(str(x) for x in out) if isinstance(out, list) else str(out or "")

def _pr_network_err(exc):
    m = str(exc).lower()
    return any(k in m for k in ("getaddrinfo", "name or service not known",
                                 "network unreachable", "connection refused",
                                 "errno 11004", "errno 11001"))

def _pr_render(q, answer, hits, q_num, mode):
    def _esc(s):
        return (str(s).replace("&","&amp;").replace("<","&lt;")
                      .replace(">","&gt;").replace("\n","<br>"))
    _ts   = _pr_dt.now().strftime("%H:%M:%S")
    _mlbl = _pr_mode_label(mode)
    _, _, _ibadge = _pr_verbatim_check(answer, hits)
    _src_rows = ""
    for i, h in enumerate(hits):
        _src  = h.metadata.get("source", "?")
        _scr  = getattr(h, "score", None)
        _scrs = f"{_scr:.3f}" if _scr is not None else "—"
        _snip = h.node.get_content()[:300].strip()
        _src_rows += (
            f"<li style='margin-bottom:12px;'>"
            f"<span style='color:#CFB53B;font-size:11px;font-weight:600;'>[{i+1}] {_esc(_src)}</span>"
            f"<span style='color:#5a8aab;font-size:10px;'> · score: {_scrs}</span><br>"
            f"<span style='color:#7a9ab0;font-size:12px;font-style:italic;'>{_esc(_snip)}…</span></li>"
        )
    _lbl_str = ", ".join(_pr_state["labels"])
    _panel = f"""
<div style='font-family:Segoe UI,Consolas,Arial,sans-serif;
    border-top:3px solid #CFB53B;border-bottom:3px solid #CFB53B;
    border-left:3px solid #1A6FBF;border-right:3px solid #1A6FBF;
    border-radius:12px;padding:20px 26px;background:#0a0a0a;color:#f0f0f0;
    box-shadow:0 0 18px rgba(26,111,191,0.35),0 0 8px rgba(207,181,59,0.2);
    max-width:900px;margin:10px 0;'>
  <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:12px;'>
    <div>
      <div style='font-size:10px;letter-spacing:2px;color:#CFB53B;text-transform:uppercase;'>
        ◆ Rapid Paper Q&amp;A · ATE v3 ◆</div>
      <div style='font-size:18px;font-weight:700;color:#ffffff;margin-top:2px;'>Evidence-first Answer</div>
    </div>
    <div style='text-align:right;font-size:11px;color:#6a8aab;line-height:1.7;'>
      Q#{q_num} &nbsp;·&nbsp; {_ts}<br>
      <span style='color:#CFB53B;font-weight:600;'>{_mlbl} mode</span><br>
      <span style='color:#a0b8c8;font-size:10px;'>{_esc(_lbl_str)}</span>
    </div>
  </div>
  <div style='margin-bottom:14px;'>{_ibadge}</div>
  <div style='font-size:11px;color:#CFB53B;letter-spacing:1px;text-transform:uppercase;margin-bottom:4px;'>Question</div>
  <div style='font-size:14px;color:#c8d8e8;font-style:italic;
              border-left:3px solid #1A6FBF;padding-left:12px;margin-bottom:16px;line-height:1.6;'>
    {_esc(q)}</div>
  <div style='font-size:11px;color:#CFB53B;letter-spacing:1px;text-transform:uppercase;margin-bottom:6px;'>Answer</div>
  <div style='background:#111111;border-left:3px solid #CFB53B;border-right:1px solid #1A6FBF;
              border-radius:6px;padding:16px 20px;color:#e8e8e8;font-size:14px;line-height:1.8;'>
    {_esc(answer)}</div>
  <details style='margin-top:16px;'>
    <summary style='cursor:pointer;font-size:11px;letter-spacing:1px;color:#CFB53B;
                    text-transform:uppercase;user-select:none;outline:none;'>
      ▸ {len(hits)} Evidence Node(s) retrieved</summary>
    <ol style='margin-top:10px;padding-left:18px;color:#9ab8cc;line-height:1.6;'>
      {_src_rows}
    </ol>
  </details>
  <div style='margin-top:14px;font-size:10px;color:#2a4a6a;text-align:right;'>
    IBM Granite 3.1 · {_PR_MODEL} · {len(_pr_state["nodes"])} nodes indexed
  </div>
</div>"""
    _pr_display(_pr_HTML(_panel))

def _pr_render_history(history):
    if not history:
        print("  (no questions asked yet)")
        return
    def _esc(s):
        return (str(s).replace("&","&amp;").replace("<","&lt;")
                      .replace(">","&gt;").replace("\n","<br>"))
    _rows = "".join(
        f"<div style='border-left:3px solid #CFB53B;padding:10px 14px;"
        f"background:#0d0d0d;border-radius:4px;margin-bottom:12px;'>"
        f"<div style='font-size:11px;color:#CFB53B;margin-bottom:4px;'>"
        f"Q{idx2} · {ex['mode']} · {ex['ts']}</div>"
        f"<div style='margin-bottom:6px;'>{ex.get('integrity_badge','')}</div>"
        f"<div style='font-size:13px;color:#c8d8e8;font-style:italic;margin-bottom:6px;'>{_esc(ex['q'])}</div>"
        f"<div style='font-size:13px;color:#e0e0e0;background:#111;padding:8px 12px;"
        f"border-radius:4px;border-left:2px solid #1A6FBF;'>{_esc(ex['a'])}</div></div>"
        for idx2, ex in enumerate(history, 1)
    )
    _pr_display(_pr_HTML(
        f"<div style='font-family:Segoe UI,Arial,sans-serif;background:#0a0a0a;"
        f"border-top:3px solid #CFB53B;border-bottom:3px solid #CFB53B;"
        f"border-left:3px solid #1A6FBF;border-right:3px solid #1A6FBF;"
        f"border-radius:12px;padding:20px 26px;max-width:900px;margin:10px 0;"
        f"box-shadow:0 0 18px rgba(26,111,191,0.35);'>"
        f"<div style='font-size:10px;letter-spacing:2px;color:#CFB53B;text-transform:uppercase;'>"
        f"◆ Session History ◆</div>"
        f"<div style='font-size:18px;font-weight:700;color:#fff;margin:4px 0 16px;'>"
        f"{len(history)} Question(s) this session</div>"
        f"{_rows}</div>"
    ))

def _pr_render_help():
    _pr_display(_pr_HTML(
        "<div style='font-family:Segoe UI,Consolas,Arial,sans-serif;background:#0a0a0a;"
        "border-top:3px solid #CFB53B;border-bottom:3px solid #CFB53B;"
        "border-left:3px solid #1A6FBF;border-right:3px solid #1A6FBF;"
        "border-radius:12px;padding:20px 26px;max-width:900px;margin:10px 0;"
        "color:#f0f0f0;font-size:13px;line-height:1.9;"
        "box-shadow:0 0 18px rgba(26,111,191,0.35);'>"
        "<div style='font-size:10px;letter-spacing:2px;color:#CFB53B;text-transform:uppercase;'>"
        "◆ Rapid Paper Q&amp;A · Help ◆</div>"
        "<div style='font-size:18px;font-weight:700;color:#fff;margin:4px 0 16px;'>"
        "Answer Modes &amp; Commands</div>"
        "<table style='width:100%;border-collapse:collapse;'>"
        "<tr style='border-bottom:1px solid #1a2a3a;'>"
        "<th style='text-align:left;color:#CFB53B;padding:6px 10px;width:200px;'>Prefix</th>"
        "<th style='text-align:left;color:#CFB53B;padding:6px 10px;'>What it does</th></tr>"
        + "".join(
            f"<tr style='border-bottom:1px solid #111;'>"
            f"<td style='padding:6px 10px;color:#CFB53B;font-family:Consolas,monospace;'>{p}</td>"
            f"<td style='padding:6px 10px;color:#c8d8e8;'>{d}</td></tr>"
            for p, d in [
                ("!short",    "One paragraph — fastest, most direct"),
                ("!long",     "Full academic depth, multi-paragraph"),
                ("!summarise","4–6 paragraph structured summary"),
                ("!bullet",   "Structured bullet/numbered breakdown"),
                ("!compare",  "Contrast frameworks across sources"),
                ("!define",   "Define a concept from the paper"),
                ("!critique", "Critical analysis — strengths, limits"),
                ("(none)",    "Auto-detected from phrasing"),
                ("history",   "Render all Q&amp;A this session"),
                ("clear",     "Wipe conversation context"),
                ("end",       "Close the Q&amp;A panel"),
            ]
        ) +
        "</table>"
        "<div style='margin-top:16px;background:#0d1a0d;border:1px solid #2a4a2a;"
        "border-radius:6px;padding:10px 14px;font-size:12px;color:#4CAF50;line-height:1.8;'>"
        "<b>Zero-Plagiarism Integrity Checker</b><br>"
        "✅ &lt;5% overlap — Original synthesis &nbsp;·&nbsp; "
        "⚠️ 5–15% — Review citations &nbsp;·&nbsp; "
        "🚨 &gt;15% — Rephrase before using"
        "</div></div>"
    ))

# ══════════════════════════════════════════════════════════════════════════════
# CSS  (banner-matched: same gold + blue quad-border theme as cell 17)
# ══════════════════════════════════════════════════════════════════════════════
_pr_display(_pr_HTML("""<style>
.ate-main-panel {
    border-top:3px solid #CFB53B !important; border-bottom:3px solid #CFB53B !important;
    border-left:3px solid #1A6FBF !important; border-right:3px solid #1A6FBF !important;
    border-radius:12px !important;
    box-shadow:0 0 18px rgba(26,111,191,0.35),0 0 8px rgba(207,181,59,0.2) !important;
}
.ate-research-input textarea {
    background:linear-gradient(160deg,#141414 0%,#0a0a0a 55%,#080c10 100%) !important;
    box-shadow:inset 0 2px 8px rgba(0,0,0,0.7),inset 0 0 0 1px rgba(26,111,191,0.35) !important;
    border:1px solid #1A4A7F !important; border-radius:8px !important;
    color:#d4e8f8 !important; font-family:'Segoe UI',Arial,sans-serif !important;
    font-size:13px !important; line-height:1.7 !important;
    caret-color:#CFB53B !important; padding:12px 14px !important;
    resize:vertical !important; transition:border-color 0.2s,box-shadow 0.2s !important;
}
.ate-research-input textarea:focus {
    border-color:#1A6FBF !important;
    box-shadow:inset 0 2px 8px rgba(0,0,0,0.7),
               0 0 10px rgba(26,111,191,0.45),
               inset 0 0 0 1px rgba(26,111,191,0.6) !important;
    outline:none !important;
}
.ate-research-input textarea::placeholder {
    color:#4a7a9a !important; font-style:italic !important;
    font-family:'Segoe UI',Arial,sans-serif !important; font-size:12px !important;
}
.ate-research-input { background:#0a0a0a !important; }
.widget-html-content, .widget-html-content * { font-family:'Segoe UI',Arial,sans-serif !important; }
.widget-html-content code {
    color:#CFB53B !important; background:#0a0a0a !important;
    border:none !important; font-family:Consolas,monospace !important; font-size:11px !important;
}
.widget-vbox, .widget-hbox, .widget-box { background:#0a0a0a !important; }
.jp-OutputArea-output, .jp-OutputArea { background:#0a0a0a !important; }
.jp-OutputArea-output p, .jp-OutputArea-output div,
.jp-OutputArea-output span, .jp-OutputArea-output pre {
    color:#a0b8c8 !important; font-family:'Segoe UI',Arial,sans-serif !important;
}
.widget-button {
    font-family:'Segoe UI',Arial,sans-serif !important; font-size:12px !important;
    border-radius:6px !important; border:none !important;
    transition:filter 0.15s !important; letter-spacing:0.3px !important;
}
.widget-button:hover  { filter:brightness(1.18) !important; }
.widget-button:active { filter:brightness(0.88) !important; }
.widget-upload .jupyter-button, .widget-upload button {
    background:#0d2a4a !important; color:#d4e8f8 !important;
    border:1px solid #1A6FBF !important; font-family:'Segoe UI',Arial,sans-serif !important;
    font-size:12px !important; border-radius:6px !important;
}
.widget-output { background:#0a0a0a !important; }
pre, code { background:#0a0a0a !important; color:#a0b8c8 !important; }
</style>"""))

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — SOURCE INPUT PANEL
# ══════════════════════════════════════════════════════════════════════════════
_pr_src_area = _widgets.Textarea(
    placeholder=(
        "Paste a Google Drive share link or local file path, one per line.\n\n"
        "Examples:\n"
        "  https://drive.google.com/file/d/…/view?usp=sharing\n"
        "  C:\\Users\\you\\Documents\\paper.pdf"
    ),
    layout=_widgets.Layout(width="100%", min_height="120px"),
)
_pr_src_area.add_class("ate-research-input")

_pr_src_upload = _widgets.FileUpload(
    accept=".pdf",
    multiple=True,
    description="📎 Upload PDFs",
    style=_widgets.ButtonStyle(button_color="#0d2a4a", font_color="#d4e8f8", font_weight="bold"),
    layout=_widgets.Layout(width="148px"),
)
_pr_load_btn = _widgets.Button(
    description="⚙️  Load & Index",
    button_style="primary",
    style=_widgets.ButtonStyle(button_color="#1A6FBF", font_weight="bold"),
    layout=_widgets.Layout(width="148px", height="34px"),
    tooltip="Download, extract and index all sources",
)
_pr_src_status = _widgets.HTML(
    value="<span style='color:#6a8fa8;font-size:12px;font-family:Consolas,monospace;'>"
          "Paste links or upload PDFs, then click Load &amp; Index.</span>"
)
_pr_load_output = _widgets.Output()

_pr_src_header = _widgets.HTML(value=
    "<div style='font-family:Segoe UI,Arial,sans-serif;background:#0a0a0a;"
    "border-bottom:1px solid #1A4A7F;border-radius:10px 10px 0 0;"
    "padding:18px 22px 14px;'>"
    "<div style='font-size:10px;letter-spacing:2px;color:#CFB53B;text-transform:uppercase;'>"
    "◆ Rapid Paper Q&amp;A · Academic Truth Engine v3 ◆</div>"
    "<div style='font-size:20px;font-weight:700;color:#ffffff;margin:6px 0 14px;'>"
    "Loading &amp; Indexing Papers…</div>"
    "<div style='color:#a0b8c8;font-size:13px;line-height:1.8;'>"
    "<b style='color:#CFB53B;'>Answer modes</b> — prefix any question with: "
    "<code style='color:#CFB53B;'>!short</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!long</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!summarise</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!bullet</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!compare</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!define</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!critique</code><br>"
    "<b style='color:#4CAF50;'>Zero-plagiarism policy active</b> — every answer is integrity-checked."
    "</div></div>"
)

_pr_src_panel = _widgets.VBox(
    [
        _pr_src_header,
        _widgets.VBox(
            [
                _pr_src_area,
                _widgets.HBox(
                    [_pr_src_upload, _pr_load_btn, _pr_src_status],
                    layout=_widgets.Layout(align_items="center", gap="10px", flex_flow="row wrap"),
                ),
                _pr_load_output,
            ],
            layout=_widgets.Layout(gap="8px", padding="14px 22px 18px"),
        ),
    ],
    layout=_widgets.Layout(width="100%", max_width="960px"),
)
_pr_src_panel.add_class("ate-main-panel")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 — Q&A PANEL  (shown after successful indexing)
# ══════════════════════════════════════════════════════════════════════════════
_pr2_text = _widgets.Textarea(
    placeholder=(
        "Enter your research question here…\n"
        "Prefix with !short · !long · !summarise · !bullet · !compare · !define · !critique\n"
        "Or just ask naturally — mode is auto-detected from your phrasing.\n\n"
        "Type  help  for mode reference.   Type  end  to close."
    ),
    layout=_widgets.Layout(width="100%", min_height="160px"),
)
_pr2_text.add_class("ate-research-input")

_pr2_submit_btn = _widgets.Button(
    description="🔍  Research",
    button_style="primary",
    style=_widgets.ButtonStyle(button_color="#1A6FBF", font_weight="bold"),
    layout=_widgets.Layout(width="142px", height="34px"),
    tooltip="Submit research question",
)
_pr2_clear_btn = _widgets.Button(
    description="✕ Clear",
    style=_widgets.ButtonStyle(button_color="#CFB53B", font_color="#000000", font_weight="bold"),
    layout=_widgets.Layout(width="90px"),
    tooltip="Clear the question box",
)
_pr2_history_btn = _widgets.Button(
    description="📑 History",
    style=_widgets.ButtonStyle(button_color="#1A4A7F", font_color="#d4e8f8", font_weight="bold"),
    layout=_widgets.Layout(width="110px"),
    tooltip="Show session Q&A history",
)
_pr2_end_btn = _widgets.Button(
    description="○ End",
    style=_widgets.ButtonStyle(button_color="#CFB53B", font_color="#000000", font_weight="bold"),
    layout=_widgets.Layout(width="80px"),
    tooltip="Close the Research Engine",
)
_pr2_status = _widgets.HTML(
    value="<span style='color:#6a8fa8;font-size:12px;font-family:Consolas,monospace;'>"
          "Index ready — enter a question.</span>"
)
_pr2_output = _widgets.Output()
_pr2_side   = _widgets.Output()

# ── Side-panel builder (mirrors cell 17's Live Source Overview) ───────────────
def _pr2_build_side() -> str:
    _src_rows = ""
    _seen: set = set()
    for _lbl in _pr_state["labels"][:6]:
        if _lbl in _seen: continue
        _seen.add(_lbl)
        _src_rows += (
            "<div style='display:flex;align-items:flex-start;gap:8px;margin-bottom:10px;'>"
            "<span style='font-size:15px;'>🗎️</span>"
            "<span style='font-size:12px;color:#d4e8f8;line-height:1.4;'>"
            f"<strong>{_lbl[:34]}</strong><br>"
            "<span style='color:#CFB53B;font-size:10px;'>Indexed (In-Memory)</span>"
            "</span></div>"
        )
    if not _src_rows:
        _src_rows = "<div style='color:#6a8fa8;font-size:11px;font-family:Consolas,monospace;'>(loading…)</div>"
    n_nodes = len(_pr_state["nodes"])
    def _bar(f, t=8):
        f = max(0, min(f, t))
        return "█" * f + "░" * (t - f)
    _stat = (
        "<div style='font-family:Consolas,monospace;font-size:11px;color:#a0c4dc;line-height:1.9;'>"
        f"<div>Nodes Indexed &nbsp;<span style='color:#CFB53B;'>[{_bar(min(8, n_nodes // 10))}]</span></div>"
        f"<div style='font-size:9px;color:#6a8fa8;margin-top:-4px;margin-bottom:6px;'>{n_nodes} semantic chunks</div>"
        f"<div>Docs Loaded &nbsp;&nbsp;&nbsp;<span style='color:#CFB53B;'>[{_bar(min(8, len(_pr_state['labels']) * 2))}]</span></div>"
        "</div>"
    )
    return (
        "<div style='font-family:Segoe UI,Arial,sans-serif;display:flex;flex-direction:column;gap:12px;'>"
        "<div style='background:#0d1117;border:1px solid #1A6FBF;border-radius:10px;padding:14px 16px;'>"
        "<div style='font-size:13px;font-weight:700;color:#d4e8f8;margin-bottom:12px;'>"
        "📚 Live Source Overview</div>" + _src_rows + "</div>"
        "<div style='background:#0d1117;border:1px solid #1A6FBF;border-radius:10px;padding:14px 16px;'>"
        "<div style='font-size:13px;font-weight:700;color:#d4e8f8;margin-bottom:10px;'>"
        "⚙️ Status Monitor</div>" + _stat + "</div></div>"
    )

def _pr2_refresh_side():
    with _pr2_side:
        _pr_clear(wait=False)
        _pr_display(_pr_HTML(_pr2_build_side()))

def _pr2_set_status(msg, colour="#6a8fa8"):
    _pr2_status.value = (
        f"<span style='color:{colour};font-size:12px;"
        f"font-family:Consolas,monospace;'>{msg}</span>"
    )

# ── Q&A header + modes bar ────────────────────────────────────────────────────
_pr2_header = _widgets.HTML(value=
    "<div style='font-family:Segoe UI,Arial,sans-serif;background:#0a0a0a;"
    "border-bottom:1px solid #1A4A7F;border-radius:10px 10px 0 0;padding:18px 22px 14px;'>"
    "<div style='font-size:10px;letter-spacing:2px;color:#CFB53B;text-transform:uppercase;'>"
    "◆ Rapid Paper Q&amp;A · Academic Truth Engine v3 ◆</div>"
    "<div style='font-size:20px;font-weight:700;color:#ffffff;margin:4px 0 14px;'>"
    "📖 Research Engine Online</div>"
    "</div>"
)
_pr2_modes_bar = _widgets.HTML(value=
    "<div style='font-family:Segoe UI,Arial,sans-serif;font-size:12px;"
    "color:#a0b8c8;line-height:1.9;border-top:1px solid #1a2a3a;padding:8px 0 4px;'>"
    "<b style='color:#CFB53B;'>Modes</b> — prefix or auto-detected from phrasing:&nbsp;&nbsp;"
    "<code style='color:#CFB53B;'>!short</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!long</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!summarise</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!bullet</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!compare</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!define</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>!critique</code><br>"
    "<b style='color:#CFB53B;'>Commands</b>:&nbsp;"
    "<code style='color:#CFB53B;'>history</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>clear</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>help</code>&nbsp;·&nbsp;"
    "<code style='color:#CFB53B;'>end</code>"
    "</div>"
)

_pr2_left = _widgets.VBox(
    [
        _widgets.HBox(
            [_pr2_history_btn, _pr2_clear_btn, _pr2_end_btn],
            layout=_widgets.Layout(align_items="center", gap="6px", flex_flow="row wrap"),
        ),
        _pr2_text,
        _pr2_modes_bar,
        _widgets.HBox(
            [_pr2_submit_btn, _pr2_status],
            layout=_widgets.Layout(align_items="center", gap="10px"),
        ),
    ],
    layout=_widgets.Layout(gap="6px", flex="1 1 0", min_width="320px"),
)
_pr2_cols = _widgets.HBox(
    [_pr2_left, _pr2_side],
    layout=_widgets.Layout(gap="14px", align_items="flex-start", width="100%"),
)
_pr2_panel = _widgets.VBox(
    [
        _pr2_header,
        _widgets.VBox([_pr2_cols], layout=_widgets.Layout(padding="0 22px 14px")),
        _pr2_output,
    ],
    layout=_widgets.Layout(gap="0px", width="100%", max_width="960px", display="none"),
)
_pr2_panel.add_class("ate-main-panel")

# ── Root container — both panels stacked; only one visible at a time ──────────
_pr_container = _widgets.VBox(
    [_pr_src_panel, _pr2_panel],
    layout=_widgets.Layout(gap="0px"),
)
_pr_display(_pr_container)

# ── Skip Phase 1 if PDF_SOURCES was pre-filled ───────────────────────────────
if PDF_SOURCES:
    _pr_src_area.value = "\n".join(PDF_SOURCES)

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 CALLBACK — Load & Index
# ══════════════════════════════════════════════════════════════════════════════
def _pr_on_load(_b):
    _sources = [ln.strip() for ln in _pr_src_area.value.splitlines() if ln.strip()]

    # Collect uploaded files
    _ul_val   = _pr_src_upload.value
    _ul_files: list[tuple[str, bytes]] = []
    if isinstance(_ul_val, (list, tuple)):
        for _item in _ul_val:
            if isinstance(_item, dict):
                _ul_files.append((_item.get("name", "file"), bytes(_item.get("content", b""))))
    elif isinstance(_ul_val, dict):
        for _fn, _fd in _ul_val.items():
            _ul_files.append((_fn, bytes(_fd.get("content", b""))))

    if not _sources and not _ul_files:
        _pr_src_status.value = (
            "<span style='color:#e0a000;font-size:12px;'>"
            "⚠️ Paste at least one link or upload a PDF first.</span>"
        )
        return

    _pr_load_btn.disabled = True
    _pr_src_status.value = (
        "<span style='color:#1A6FBF;font-size:12px;'>"
        "⚙️ Loading documents… (this may take a minute)</span>"
    )

    with _pr_load_output:
        _pr_clear(wait=True)
        _n_total = len(_sources) + len(_ul_files)
        print(f"📚 Loading {_n_total} source(s)…")

        _pr_tmp   = tempfile.mkdtemp(prefix="pr18_")
        _docs_loc: list = []
        _lbls_loc: list[str] = []

        from llama_index.core.schema import Document as _LI_Doc

        for _src in _sources:
            try:
                if _pr_is_gdrive(_src):
                    _fid  = _pr_file_id(_src)
                    _dest = os.path.join(_pr_tmp, f"{_fid}.pdf")
                    _lbl  = f"Drive:{_fid[:14]}"
                    print(f"  📥 Downloading {_lbl}…")
                    _pr_download_gdrive(_src, _dest)
                    _path = _dest
                else:
                    if not os.path.exists(_src):
                        print(f"  ⚠️  Not found — skipping: {_src!r}"); continue
                    _path = _src
                    _lbl  = pathlib.Path(_src).name
                    print(f"  📄 {_lbl}")
                _txt = _pr_extract_text(_path)
                if not _txt.strip():
                    print(f"  ⚠️  No text extracted from {_lbl} — skipping."); continue
                _docs_loc.append(_LI_Doc(text=_txt, metadata={"source": _lbl, "file_name": _lbl}))
                _lbls_loc.append(_lbl)
                print(f"     ✅ {len(_txt):,} chars")
            except Exception as _le:
                print(f"  ⚠️  Failed ({_src}): {_le}")

        for _fn, _fc in _ul_files:
            try:
                _dest = os.path.join(_pr_tmp, _fn)
                with open(_dest, "wb") as _fh:
                    _fh.write(_fc)
                _txt = _pr_extract_text(_dest)
                if not _txt.strip():
                    print(f"  ⚠️  No text from {_fn} — skipping."); continue
                _docs_loc.append(_LI_Doc(text=_txt, metadata={"source": _fn, "file_name": _fn}))
                _lbls_loc.append(_fn)
                print(f"  📎 {_fn}: ✅ {len(_txt):,} chars")
            except Exception as _ue:
                print(f"  ⚠️  Upload failed ({_fn}): {_ue}")

        if not _docs_loc:
            print("❌ No documents loaded.")
            _pr_src_status.value = (
                "<span style='color:#e07070;font-size:12px;'>"
                "❌ No documents loaded. Check links/paths and retry.</span>"
            )
            _pr_load_btn.disabled = False
            return

        # Chunking
        print(f"\n⚙️  Chunking {len(_docs_loc)} doc(s)…")
        _spl = globals().get("splitter")
        if _spl is None:
            from llama_index.core.node_parser import SemanticSplitterNodeParser
            _em = globals().get("embed_model")
            if _em is None:
                from llama_index.embeddings.huggingface import HuggingFaceEmbedding
                from llama_index.core import Settings as _S
                _em = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
                _S.embed_model = _em
            _spl = SemanticSplitterNodeParser(
                buffer_size=3, breakpoint_percentile_threshold=95, embed_model=_em)
        _nodes_loc = _spl.get_nodes_from_documents(_docs_loc)
        print(f"  ✅ {len(_nodes_loc)} semantic nodes ready.")

        # Indexing
        print("🔍 Building in-memory vector index…")
        _VI = globals().get("VectorStoreIndex")
        if _VI is None:
            from llama_index.core import VectorStoreIndex as _VI
        _em2  = globals().get("embed_model") or _spl.embed_model
        _idx  = _VI(_nodes_loc, embed_model=_em2)
        _retr = _idx.as_retriever(similarity_top_k=PR_TOP_K)
        print(f"  ✅ Index ready — {len(_nodes_loc)} nodes · top-k={PR_TOP_K}\n")
        print("─" * 50)
        print(f"  Papers : {', '.join(_lbls_loc)}")
        print(f"  Nodes  : {len(_nodes_loc)}")
        print(f"  Policy : Zero-plagiarism · verbatim overlap detection active")
        print("─" * 50)

        _pr_state["nodes"]    = _nodes_loc
        _pr_state["retriever"] = _retr
        _pr_state["labels"]   = _lbls_loc

    # Switch to Q&A phase
    _pr_src_panel.layout.display = "none"
    _pr2_panel.layout.display    = ""
    _pr2_refresh_side()
    _pr2_set_status(
        f"✅ {len(_pr_state['nodes'])} nodes · {len(_pr_state['labels'])} doc(s) — ready.",
        "#50c878",
    )

_pr_load_btn.on_click(_pr_on_load)

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 CALLBACKS — Q&A
# ══════════════════════════════════════════════════════════════════════════════
def _pr2_on_clear(_b=None):
    _pr2_text.value = ""
    _pr2_set_status("Cleared — enter a new question.")
    with _pr2_output:
        _pr_clear()

def _pr2_show_history(_b=None):
    with _pr2_output:
        _pr_clear(wait=True)
        _pr_render_history(_pr_state["history"])

def _pr2_on_end(_b=None):
    for _btn in (_pr2_submit_btn, _pr2_end_btn, _pr2_history_btn, _pr2_clear_btn):
        _btn.disabled = True
    _pr2_set_status("🛑 Research Engine closed.", "#e07070")
    with _pr2_output:
        _pr_clear(wait=True)
        print("🛑 Session closed. Good luck with your assignment!")

def _pr2_on_submit(_b=None):
    _q_raw = _pr2_text.value.strip()
    if not _q_raw:
        _pr2_set_status("⚠️ Enter a question first.", "#e0a000")
        return

    _cmd = _q_raw.lower()
    if _cmd in ("end", "exit", "quit"):
        _pr2_on_end(); return
    if _cmd == "help":
        with _pr2_output:
            _pr_clear(wait=True)
            _pr_render_help()
        return
    if _cmd in ("history", "show history"):
        _pr2_show_history(); return
    if _cmd == "clear":
        _pr_state["ctx_mem"] = []
        _pr2_set_status("🗑️  Context cleared — PDFs still indexed.")
        return

    _mode, _q = _pr_detect_mode(_q_raw)
    _pr_state["q_num"] += 1
    _qn   = _pr_state["q_num"]
    _lbl  = _pr_mode_label(_mode)
    _pr2_set_status(f"🔍 Mode: {_lbl} | Retrieving evidence… (30–120 s)", "#1A6FBF")
    _pr2_submit_btn.disabled = True

    with _pr2_output:
        _pr_clear(wait=True)
        print(f"🔍 [{_lbl}] Retrieving evidence…")

    try:
        _hits = _pr_state["retriever"].retrieve(_q)
    except Exception as _re:
        with _pr2_output:
            _pr_clear(wait=True)
            print(f"⚠️  Retrieval error: {_re}")
        _pr2_set_status("⚠️ Retrieval error.", "#e07070")
        _pr2_submit_btn.disabled = False
        return

    if not _hits:
        with _pr2_output:
            _pr_clear(wait=True)
            print("⚠️  No relevant nodes found. Try rephrasing or a different mode.")
        _pr2_set_status("⚠️ No evidence found.", "#e0a000")
        _pr2_submit_btn.disabled = False
        return

    # Build context
    _ctx_parts = []
    for _i, _h in enumerate(_hits):
        _src = _h.metadata.get("source", "?")
        _ctx_parts.append(f"[SOURCE {_i+1} | {_src}]\n{_h.node.get_content()[:1200]}")
    _evidence = "\n\n---\n\n".join(_ctx_parts)

    # Conversation memory
    _conv_block = ""
    if _pr_state["ctx_mem"]:
        _lines = []
        for _ex in _pr_state["ctx_mem"][-PR_HISTORY_CTX:]:
            _lines.append(f"Previous Q: {_ex['q']}\nPrevious A: {_ex['a'][:400]}")
        _conv_block = "\n\nPRIOR CONVERSATION CONTEXT:\n" + "\n---\n".join(_lines)

    _instr  = _MODE_INSTRUCTIONS[_mode]
    _prompt = (
        f"You are a rigorous academic research assistant working with a student.\n"
        f"{_instr}\n"
        f"Ground every claim strictly in the SOURCE EXCERPTS below. "
        f"Do NOT speculate or add outside knowledge.\n"
        f"{_conv_block}\n\n"
        f"QUESTION:\n{_q}\n\n"
        f"SOURCE EXCERPTS:\n{_evidence}\n\n"
        f"ANSWER:"
    )

    with _pr2_output:
        _pr_clear(wait=True)
        print("🧠 Synthesising with Granite 3.1…")

    _t0 = time.time()
    try:
        _ans = _pr_granite(_prompt, _mode).strip()
    except Exception as _ge:
        with _pr2_output:
            _pr_clear(wait=True)
            if _pr_network_err(_ge):
                print("🌐 OFFLINE — cannot reach Replicate. Check your connection.")
            else:
                print(f"⚠️  Granite error: {_ge}")
        _pr2_set_status("⚠️ LLM error.", "#e07070")
        _pr2_submit_btn.disabled = False
        return

    _dur = time.time() - _t0
    _ratio, _, _ibadge = _pr_verbatim_check(_ans, _hits)

    with _pr2_output:
        _pr_clear(wait=True)
        _pr_render(_q, _ans, _hits, _qn, _mode)
        if _ratio >= PR_VERBATIM_ALERT:
            print(f"🚨 Verbatim risk ({_ratio*100:.1f}% overlap) — review before using.")
        elif _ratio >= PR_VERBATIM_WARN:
            print(f"⚠️  Overlap at {_ratio*100:.1f}% — check paraphrasing.")

    _entry = {
        "q": _q, "a": _ans, "mode": _lbl,
        "ts": _pr_dt.now().strftime("%H:%M:%S"),
        "integrity_badge": _ibadge,
    }
    _pr_state["history"].append(_entry)
    _pr_state["ctx_mem"].append(_entry)
    _pr2_set_status(f"✅ Done in {_dur:.1f}s | {_lbl}", "#50c878")
    _pr2_refresh_side()
    _pr2_submit_btn.disabled = False

_pr2_clear_btn.on_click(_pr2_on_clear)
_pr2_history_btn.on_click(_pr2_show_history)
_pr2_end_btn.on_click(_pr2_on_end)
_pr2_submit_btn.on_click(_pr2_on_submit)
